# Brain-to-Text '25 — **v6.2 · CONTINUE TRAINING from `b2t-v6-crossfit-1`**
Warm start: fold A ← `foldA_seed0_best_per.pt`, fold B ← `foldB_seed1_best_per.pt` (same split, same normalisation, same model/training code), then a fresh ~11 h time-based schedule on both T4s.

**Why the previous run stopped at 5h31m:** both workers were killed by the OS for host RAM 11 times (`exit code -9`); worker RSS grew ~linearly with steps (9.7 GB @ step 400 → 31 GB @ step 6200), so the 5 restarts per fold ran out. v6.2 adds, without touching the model or the training algorithm:
* **memory guard**: a worker that reaches `WORKER_RSS_LIMIT_GB` (or the machine drops below `WORKER_MIN_AVAIL_GB`) saves its resume state and exits with code 75; the notebook relaunches it immediately with `--resume` (not counted as a crash, unlimited). Nothing is lost and the OS never has to kill it.
* eval / resume timers survive restarts (before, every restart reset the 30-min eval timer, so fold A got 1 evaluation in 5 h).
* cuDNN plan-cache size cap via the environment variable `TORCH_CUDNN_V8_API_LRU_CACHE_LIMIT` (no numerical effect; limits host memory held for per-shape convolution plans).
* init checkpoint must load with **zero** missing/unexpected keys, otherwise the worker stops instead of silently training from random weights.

---
`MODE = 'train'` (≈11.5 h, both T4s) → Save Version → dataset from output `b2t_v6_ckpt/` → `MODE = 'llm'` (decode + `submission.csv`).

## v6.1 — why v6.0 died and what was changed
v6.0 log: both workers `exit code -9` at minute 27 (host-RAM OOM kill by the OS), GPU at 14.3/15.3 GB.

| Failure source in v6.0 | v6.1 |
|---|---|
| KenLM beam evaluator loaded inside **each** training worker (multi-GB host RAM each) | removed from training; checkpoint selection by official PER + greedy WER; real beam WER is measured once in `llm` mode |
| DataLoader subprocesses + pinned memory + memmap per worker | no subprocesses, no pinned memory; trials read with `os.pread` |
| one kill = whole run lost | resume state every 20 min; the notebook **relaunches a dead worker with `--resume`** (up to 5×) |
| GPU at 94 % → a long batch could OOM | patch embedding as strided Conv1d (no 7168-d tensor), expandable CUDA segments, **OOM batch is skipped** and the batch cap shrinks |
| `torchaudio.ctc_decoder` (not guaranteed on torch 2.10) | flashlight-text used directly (output verified identical), one shared KenLM per process, process count limited by free RAM |
| `multiprocessing.Pool` hangs forever if a worker is killed | `ProcessPoolExecutor` + automatic serial fallback |
| any exception in an analysis cell stops *Run All* | analysis stages are *soft*: they log the traceback and the notebook continues; a fallback `submission.csv` is written as early as possible |

Method (unchanged from v6.0): trial-level cross-fit of val (A/B/A/B/C per block; GPU0 train+A, GPU1 train+B, C unseen by both) · CR-CTC + time masking + electrode dropout + speed perturbation · exact-CTC hypothesis ensembling (fixes v2's 66 % posterior-averaging failure) · phonetic-neighbour candidate expansion · Qwen2.5-7B LoRA next-word-prediction fine-tune · fluency-gated log-linear rescoring tuned on out-of-fold A∪B and verified on C · official WER = Σ edit / Σ ref words after `remove_punctuation`.

## Decoding sweep (BLOCK 4.2b + BLOCK 4.5)

The 4.67% run decoded at ONE static setting. `SWEEP_CFG` now carries ranges and the best combination is picked on the tune folds.

**The finding that motivated this.** In that run the stage-2 optimum landed on the edge of the search grid in all three weights at once — `llm=2.0` is `max(B2)`, `ng=2.5` is `max(A2)`, `nw=-4.0` is `min(G2)` — and stage-1 picked `nw=-4.0 = min(G1)` as well. A tuner that stops at a boundary hasn't found an optimum, it has run out of grid. Widening costs about a second per grid, so the grid *range* is now a swept axis and every row is flagged when its weights still land on an edge.

| axis | range | cost |
|---|---|---|
| `gen_beam` | `[300]` | full regeneration each (~1h23m) — add 400 only if you have the hours |
| `gen_lm_weights` | 4 subsets | **free** — union generated once, subsets sliced |
| `gen_nbest` | `[25, 50, 75]` | **free** — n-best lists are nested |
| `expand_top/max_nb/max_new` | 4×2×2 | cheap — only new strings hit the CTC scorer |
| `stage1_grid` | `base`, `wide` | ~1 s per grid |
| `stage2_grid` | `base`, `wide`, `wider` | ~1–1.5 s per grid |
| `tol_rel` | `[0, 0.005, 0.02]` | **free** — re-picks an already-computed grid |
| `smooth_grid` | `[True, False]` | **free** — same |
| `llm_topk` | `[12, 24, 40]` | **free** — largest k scored once, smaller k is its prefix |
| gate percentile | `[0..40]` | already swept |

`tol_rel` was already implemented in `pick_from_grid` (a 1-SE-style rule that prefers the smallest `|llm|` weight among near-ties) but never wired to a config — it defaulted to plain argmin. It is swept now.

A `FeatCache` memoises `(acoustic, n-gram)` per (utterance, hypothesis), so rebuilding the pool dozens of times costs one CTC pass per *new* string rather than per variant.

**Selection is on A∪B only.** Fold C is carried in every table so you can see whether the choice transferred, and is never selected on. BLOCK 4.2b prints a bootstrap standard deviation of the tune WER next to the winning margin and says how many rival configs sit inside it. With this many axes on 570 tune trials, some of the tune noise *will* get fitted — read the C column, not the tune column, as the estimate of what generalises.


In [1]:
import time; NOTEBOOK_T0 = time.time()          # session clock starts here
import os
if os.environ.get('B2T_SKIP_LLM_PIP') is None:
    !pip install -q editdistance h5py psutil
    !pip install -q flashlight-text
    !pip install -q kenlm 2>/dev/null || pip install -q https://github.com/kpu/kenlm/archive/master.zip 2>/dev/null
    !pip install -q peft -U 2>/dev/null
    !pip install -q bitsandbytes -U 2>/dev/null
    !pip install -q transformers -U 2>/dev/null

# verify flashlight-text actually imports -- it's a compiled extension only touched
# inside beam-search worker subprocesses, so a silent install failure would otherwise
# surface ~30+ minutes later, deep inside Block 4.2, instead of right here.
try:
    from flashlight.lib.text.decoder import LexiconDecoder  # noqa: F401
    print('[deps] flashlight-text OK')
except ImportError as e:
    print(f'[deps] flashlight-text import FAILED ({e!r}); retrying install with output visible ...')
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', 'flashlight-text'], check=False)
    from flashlight.lib.text.decoder import LexiconDecoder  # raises clearly if still broken
    print('[deps] flashlight-text OK after retry')

print('deps ok')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.5/427.5 kB 5.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.8 MB/s eta 0:00:00
[deps] flashlight-text OK
deps ok


# BLOCK 1 · CONFIG (the only switch is `MODE`)

In [2]:
# ============================================================================
# BLOCK 1 - CONFIG, PATHS, HELPERS
# ============================================================================
import os, gc, re, sys, json, math, time, random, subprocess, importlib, shutil, pickle, contextlib
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
try:
    import kagglehub
except ImportError:
    kagglehub = None

NOTEBOOK_T0 = globals().get('NOTEBOOK_T0', time.time())

# ============================================================================
#  >>>>>>>>>>>>>>>>>>>>>>  T H E   O N L Y   S W I T C H  <<<<<<<<<<<<<<<<<<<<
# ============================================================================
MODE = 'llm'            # 'train'  |  'llm'

# ---- session time budget (train mode) ---------------------------------------
SESSION_HOURS = 11.5              # Kaggle limit you want to respect
NOTEBOOK_RESERVE_MIN = 10         # after workers stop: curves/tables/output flush
WORKER_FINAL_RESERVE_MIN = 8      # inside each worker: final EMA eval + checkpoint write
MONITOR_PRINT_MIN = 10            # how often the monitor echoes worker progress
MAX_WORKER_RESTARTS = 10          # a crashed worker is relaunched from its last resume state
WORKER_RSS_LIMIT_GB = 11.0        # memory guard: worker saves state + exits(75) above this RSS -> relaunched
WORKER_MIN_AVAIL_GB = 3.0         # ...or when the machine has less than this RAM available
RESTART_MIN_REMAINING_MIN = 10    # ...unless less than this much training time is left

# ---- where the trained checkpoints live (llm mode) --------------------------
TRAINED_MODEL_DIR = '/kaggle/input/datasets/anibiswas1906125/b2t-v6-crossfit-2'   # dataset made from b2t_v6_ckpt/ (auto-searched if wrong)
TRAINED_MODEL_DATASET = None
FALLBACK_CKPT_DATASET = None
CKPT_PREFERENCE = ['best_wer.pt', 'ema_final.pt', 'best_per.pt']   # tie-break order after real-WER screening

LEXICON_DATASET = 'heyyousum/quality-english-dataset-for-ngram-model-v2'
KENLM_DATASET = 'heyyousum/custom-4-gram-wiki-news-switchboard-updated-v3'
LEXICON_PATH = TOKENS_PATH = KENLM_PATH = None      # resolved in BLOCK 2

SEED = 1337
WORK = '/kaggle/working' if os.path.isdir('/kaggle') else os.path.abspath('./work')
CACHE_CANDIDATES = ['/tmp/b2t_cache', '/kaggle/temp/b2t_cache', '/var/tmp/b2t_cache', os.path.join(WORK, '_b2t_cache')]
CACHE_DIR = None                  # chosen in BLOCK 2 (disk-backed, >=14 GB free)
DATA_DIR_LOCAL = '/kaggle/input/competitions/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final'

# ---- cross-fit plan: one run per GPU -----------------------------------------
CROSSFIT_PATTERN = ['A', 'B', 'A', 'B', 'C']      # inside every (session, block) of val, by trial order
TRAIN_RUNS = [
    {'fold': 'A', 'seed': 0, 'gpu': 0, 'train_val_labels': ['A']},
    {'fold': 'B', 'seed': 1, 'gpu': 1, 'train_val_labels': ['B']},
]

# ---- v6.2: continue training from a previous run's checkpoints ----------------
INIT_CKPT_DIR = '/kaggle/input/datasets/anibiswas1906125/b2t-v6-crossfit-1'   # None -> train from scratch
INIT_PREFERENCE = ['best_per.pt', 'best_wer.pt', 'ema_final.pt']            # per fold: first file found is loaded
# FINAL-FIT variant (after v6 is validated): 'train_val_labels': ['A','B','C'] for both runs
# (no clean val left -> llm mode then reuses the weights tuned on the cross-fit run).

MODEL_CFG = dict(
    input_size=512, patch_size=14, patch_stride=4, d_model=512, gru_hidden=384, gru_layers=4,
    n_conformer=3, n_heads=8, d_ff=1536, conv_kernel=15, day_rank=32, day_dropout_p=0.15,
    dropout=0.35, attn_dropout=0.1, drop_path_rate=0.15, smooth_kernel_std=2.0, smooth_kernel_size=100,
)
TRAIN_CFG = dict(
    batch_size=32, max_bins_per_batch=32 * 1400,
    lr_max=7.0e-4, lr_min=1.0e-5, warmup_steps=2000, lr_day_scale=2.0,
    weight_decay=0.01, weight_decay_day=0.01, grad_clip=2.0, clip=5.0, ema_decay=0.999,
    interctc_weight=0.3, cr_weight=0.2,
    log_every=200, eval_every_min=30, eval_every_min_late=15, late_frac=0.6, resume_every_min=20,
    init_from=None,
)
AUG_CFG = dict(
    random_cut=3, speed_p=0.5, speed_lo=0.9, speed_hi=1.1,
    white_noise_std=0.8, constant_offset_std=0.2, static_gain_std=0.05,
    electrode_drop_p=0.10, electrode_drop_apply=0.5,
    time_mask_frac=0.20, time_mask_min=4, time_mask_max=24,
)
LLM_CFG = dict(
    screen_n=300, screen_beam=50, screen_lm_weight=3.0,
    gen_lm_weights=[0.5, 2.0, 3.5, 5.0], gen_beam=400, gen_nbest=75, n_proc=3,   # sparse-but-wide lm_weight span; nbest raised (near-free), beam kept at baseline; n_proc is auto-capped by free RAM
    expansion=True, expand_top=3, expand_max_nb=12, expand_max_new=600,
    use_llm=True, llm_name='meta-llama/Llama-3.1-8B', llm_finetune=True,
    lora_r=16, lora_alpha=32, lora_dropout=0.05, llm_ft_epochs=2, llm_ft_lr=2e-4,
    llm_ft_batch=6, llm_ft_grad_accum=6,        # QLoRA on a single T4: small micro-batch + accumulation
    llm_topk=24, llm_batch=16,                  # scoring batch trimmed down for the larger 7B model
    gate_percentiles=[0, 5, 10, 15, 20, 30, 40], smooth_grid=True, refit_on_all_oof=True,
    tune_labels=['A', 'B'], verify_labels=['C'],
)

# ============================================================================
#  DECODING SWEEP: search ranges instead of the single static decode setting.
#  Cost per axis, measured on this notebook's own 2h44m run:
#    gen_beam    -> FULL flashlight regeneration               ~1h23m per value
#    lm_weights  -> FREE: generate the union once, score any subset
#    nbest       -> FREE: n-best lists are nested, slice [:n] off the generated max
#    expand_*    -> cheap: only genuinely new strings hit the CTC scorer
#    stage-1/2 weight grid ranges -> ~1s per grid (see the note below)
#    tol_rel / smooth_grid        -> FREE: re-pick from an error grid already computed
#    llm_topk    -> FREE: score the largest k once, smaller k is a prefix of it
#    gate pct    -> already a sweep
#
#  WHY THE GRID RANGES ARE IN HERE. In the 4.67% run the stage-2 optimum landed on
#  the EDGE of the search grid in all three weights at once: llm=2.0 is max(B2),
#  ng=2.5 is max(A2), nw=-4.0 is min(G2) - and stage-1 picked nw=-4.0 = min(G1) too.
#  A tuner that stops at the boundary has not found an optimum, it has run out of
#  grid. Widening costs about a second per grid, so it is swept here and BLOCK 4.5
#  reports whether the winning weights still sit on an edge.
#
#  Selection is on the TUNE folds (A u B) only; fold C is never selected on, so it
#  stays an honest check - and BLOCK 4.2b/4.5 print a bootstrap noise floor next to
#  every margin, because sweeping this many axes WILL fit some of the tune noise.
# ============================================================================
SWEEP_CFG = dict(
    enable=True,
    mode='staged',               # 'staged' = coordinate descent (cheap) | 'grid' = full product
    passes=2,
    # ---- pool side (BLOCK 4.2b) ----
    beam=[300],                  # add 400 only if you can afford another ~1h30m of generation
    nbest=[25, 50, 75],
    lm_weight_sets=[
        [3.0],
        [2.0, 3.0, 4.0],
        [0.5, 2.0, 3.0, 3.5, 5.0],
        [0.5, 1.5, 2.5, 3.0, 4.0, 5.0],
    ],
    expand_top=[0, 3, 5, 8],
    expand_max_nb=[12, 20],
    expand_max_new=[600, 1200],
    stage1_grid=['base', 'wide'],
    # ---- stage-2 side (BLOCK 4.5) ----
    stage2_grid=['base', 'wide', 'wider'],
    tol_rel=[0.0, 0.005, 0.02],  # 1-SE-style tie-break toward a smaller |llm| weight
    smooth_grid=[True, False],
    llm_topk=[12, 24, 40],
    # ---- budget / reporting ----
    time_budget_min=75,          # pool sweep only (generation excluded)
    stage2_budget_min=25,
    min_minutes_left_for_extra_beam=240,
    cache_features=True,
    bootstrap=400,
)

STAGE1_GRIDS = {                 # (ng start, stop, step), (word-bonus start, stop, step)
    'base': ((0.0, 3.0, 0.1), (-4.0, 6.0, 0.25)),
    'wide': ((0.0, 4.0, 0.1), (-8.0, 8.0, 0.25)),
}
STAGE2_GRIDS = {                 # (ng...), (llm...), (word-bonus...)
    'base':  ((0.0, 2.5, 0.25), (0.0, 2.0, 0.1), (-4.0, 6.0, 0.5)),
    'wide':  ((0.0, 4.0, 0.25), (0.0, 4.0, 0.1), (-8.0, 8.0, 0.5)),
    'wider': ((0.0, 6.0, 0.25), (0.0, 6.0, 0.2), (-10.0, 10.0, 0.5)),
}


def mk_axis(spec):
    lo, hi, st = spec
    return np.round(np.arange(lo, hi + 1e-9, st), 3)


def stage1_axes(name):
    a, g = STAGE1_GRIDS[name]
    return mk_axis(a), mk_axis(g)


def stage2_axes(name):
    a, b, g = STAGE2_GRIDS[name]
    return mk_axis(a), mk_axis(b), mk_axis(g)


def on_edge(W, A, B=None, G=None):
    """True if a tuned weight sits on the boundary of its own search grid."""
    f = []
    if A is not None:
        f.append(W['ng'] <= A[0] + 1e-9 or W['ng'] >= A[-1] - 1e-9)
    if B is not None and len(B) > 1:
        f.append(W['llm'] <= B[0] + 1e-9 or W['llm'] >= B[-1] - 1e-9)
    if G is not None:
        f.append(W['nw'] <= G[0] + 1e-9 or W['nw'] >= G[-1] - 1e-9)
    return any(f)

CODE_DIR = os.path.join(WORK, 'b2t_code')
CKPT_OUT_DIR = os.path.join(WORK, 'b2t_v6_ckpt')
LOG_DIR = os.path.join(WORK, 'logs')
figures_dir = os.path.join(WORK, 'figures')
tables_dir = os.path.join(WORK, 'tables')
LLM_CACHE_DIR = os.path.join(WORK, 'llm_cache')

# local CPU smoke test hook (never set on Kaggle)
if os.environ.get('B2T_LOCAL_TEST'):
    exec(open(os.environ['B2T_LOCAL_TEST']).read())

for d in (WORK, CODE_DIR, CKPT_OUT_DIR, LOG_DIR, figures_dir, tables_dir, LLM_CACHE_DIR):
    os.makedirs(d, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS = torch.cuda.device_count()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def resolve_competition_path(local_path, competition_slug):
    if os.path.exists(local_path):
        return local_path
    if kagglehub is not None:
        return os.path.join(kagglehub.competition_download(competition_slug), 't15_copyTask_neuralData/hdf5_data_final')
    raise FileNotFoundError(local_path + ' not found and kagglehub unavailable.')


def resolve_kaggle_dataset(slug, local_dirname=None):
    if slug is None:
        raise FileNotFoundError('no dataset slug given')
    if os.path.isdir(slug):
        return slug
    local_dirname = local_dirname or slug.split('/')[-1]
    owner = slug.split('/')[0] if '/' in slug else None
    cands = ['/kaggle/input/' + local_dirname]
    if owner:
        cands.append('/kaggle/input/datasets/' + owner + '/' + local_dirname)
    if os.path.isdir('/kaggle/input'):
        for root, dirs, _ in os.walk('/kaggle/input'):
            for d in dirs:
                if local_dirname.lower() in d.lower() or d.lower() in local_dirname.lower():
                    cands.append(os.path.join(root, d))
    for p in cands:
        if os.path.exists(p):
            return p
    import kagglehub as _kh
    return _kh.dataset_download(slug)


def find_file(root_dir, pattern):
    m = sorted(glob(os.path.join(root_dir, '**', pattern), recursive=True))
    if not m:
        raise FileNotFoundError(f"No file matching '{pattern}' under {root_dir}")
    return m[0]


@contextlib.contextmanager
def figure_guard(name):
    """a plotting bug must never kill an 11-hour pipeline"""
    try:
        yield
    except Exception as e:
        print(f'  [figure_guard] {name} skipped: {e!r}')
    finally:
        plt.close('all')


def save_fig(fig, name):
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(figures_dir, f'{name}.{ext}'), dpi=300, bbox_inches='tight')
    print('  figure ->', os.path.join(figures_dir, name + '.png'))


def save_table(df, name):
    p = os.path.join(tables_dir, name)
    df.to_csv(p, index=False)
    print('  table  ->', p)


from IPython.core.magic import register_cell_magic
import traceback
STAGE_FAILED = []


@register_cell_magic
def run_if(line, cell):
    """%%run_if <modes...> [--soft] [--needs-ok]
    --soft     : an exception is printed (full traceback) and recorded in STAGE_FAILED; Run All continues
    --needs-ok : skipped when an earlier soft stage failed
    --optional : like --soft, but a failure does not block later stages (analysis/figures only)"""
    args = line.split()
    modes = [a for a in args if not a.startswith('--')]
    optional = '--optional' in args
    soft, needs_ok = ('--soft' in args) or optional, '--needs-ok' in args
    if MODE not in modes:
        print(f'[skip] this cell runs only for MODE in {modes} (MODE={MODE!r})')
        return
    if needs_ok and STAGE_FAILED:
        print(f'[skip] an earlier stage failed ({STAGE_FAILED}); the fallback submission.csv stays in place')
        return
    shell = get_ipython()
    code = shell.transform_cell(cell)
    try:
        exec(compile(code, '<cell>', 'exec'), shell.user_ns)
    except Exception as e:
        if not soft:
            raise
        lines_ = [l for l in cell.strip().splitlines() if l.startswith('# BLOCK')]
        name = lines_[0][2:60] if lines_ else 'cell'
        if not optional:
            STAGE_FAILED.append(name)
        print('\n' + '!' * 90 + f'\n{'OPTIONAL' if optional else 'SOFT'} STAGE FAILED ({e!r}) - notebook continues. Traceback:\n' + traceback.format_exc() + '!' * 90)


plt.rcParams.update({'figure.dpi': 110, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
print(f'MODE={MODE} | device={DEVICE} x{N_GPUS} | torch {torch.__version__} | WORK={WORK}')
if SWEEP_CFG['enable']:
    print(f"decoding sweep ON ({SWEEP_CFG['mode']}): beam {SWEEP_CFG['beam']} | nbest {SWEEP_CFG['nbest']} | "
          f"{len(SWEEP_CFG['lm_weight_sets'])} weight sets | expand_top {SWEEP_CFG['expand_top']} | "
          f"stage-1 grid {SWEEP_CFG['stage1_grid']} | stage-2 grid {SWEEP_CFG['stage2_grid']} | "
          f"tol_rel {SWEEP_CFG['tol_rel']} | llm_topk {SWEEP_CFG['llm_topk']}")
print(f'session clock: {(time.time() - NOTEBOOK_T0) / 60:.1f} min used of {SESSION_HOURS} h')

MODE=llm | device=cuda x2 | torch 2.10.0+cu128 | WORK=/kaggle/working
decoding sweep ON (staged): beam [300] | nbest [25, 50, 75] | 4 weight sets | expand_top [0, 3, 5, 8] | stage-1 grid ['base', 'wide'] | stage-2 grid ['base', 'wide', 'wider'] | tol_rel [0.0, 0.005, 0.02] | llm_topk [12, 24, 40]
session clock: 1.7 min used of 11.5 h


In [3]:
# ============================================================================
# BLOCK 0 - LOAD KAGGLE SECRETS INTO ENVIRONMENT (Kaggle does NOT auto-export
#           attached secrets as env vars -- they must be pulled explicitly)
# ============================================================================
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('[secrets] HF_TOKEN loaded into environment from Kaggle Secrets.')
except Exception as e:
    print(f'[secrets] Could not load HF_TOKEN from Kaggle Secrets ({e!r}). '
          f'Make sure it is attached via Add-ons -> Secrets and enabled for this notebook.')


[secrets] HF_TOKEN loaded into environment from Kaggle Secrets.


In [4]:
# ============================================================================
# BLOCK 1a - EARLY HF ACCESS CHECK (fail fast instead of after ~1.5h of compute)
# ============================================================================
if LLM_CFG.get('use_llm', True):
    import os
    from huggingface_hub import HfApi
    from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError

    _llm_name = LLM_CFG['llm_name']
    _hf_token = os.environ.get('HF_TOKEN')
    print(f"[hf-check] verifying access to '{_llm_name}' "
          f"(HF_TOKEN {'found' if _hf_token else 'NOT SET'}) ...")
    try:
        HfApi().model_info(_llm_name, token=_hf_token)
        print(f"[hf-check] OK -- '{_llm_name}' is reachable with the current token.")
    except GatedRepoError as e:
        raise RuntimeError(
            f"\n[hf-check] '{_llm_name}' is a GATED repo and access was refused.\n"
            f"  1) Visit https://huggingface.co/{_llm_name} and accept the license "
            f"(button: 'Agree and access repository').\n"
            f"  2) Make sure a Kaggle Secret named HF_TOKEN is attached to THIS notebook "
            f"(Add-ons -> Secrets -> enable for this notebook), not just saved globally.\n"
            f"  3) The token must belong to the account that accepted the license, with at "
            f"least 'Read' scope.\n"
            f"Fix this BEFORE running the rest of the notebook -- this check exists so you "
            f"don't lose ~1.5h of compute discovering it at the LLM stage."
        ) from e
    except RepositoryNotFoundError as e:
        raise RuntimeError(f"[hf-check] '{_llm_name}' was not found on the Hub -- check the model name for typos.") from e
    except Exception as e:
        print(f"[hf-check] WARNING: could not verify access ({e!r}); continuing, but the LLM stage may still fail later.")


[hf-check] verifying access to 'meta-llama/Llama-3.1-8B' (HF_TOKEN found) ...
[hf-check] OK -- 'meta-llama/Llama-3.1-8B' is reachable with the current token.


# BLOCK 1b · module files (shared by the notebook and the training worker processes)

In [5]:
%%writefile {CODE_DIR}/b2t_core.py
# =============================================================================
# b2t_core.py  --  Brain-to-Text '25  |  v6 core (shared by train workers + llm)
#   data cache (memmap), trial-level cross-fit split, B2TNetV4 acoustic model,
#   GPU augmentation, CTC + inter-CTC + CR-CTC losses, OFFICIAL PER/WER metrics
# =============================================================================
import os, re, json, math, time, random, pickle
from glob import glob
from pathlib import Path

import numpy as np
import editdistance
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from scipy.ndimage import gaussian_filter1d

PHONEME_VOCAB = [
    'BLANK', 'AA', 'AE', 'AH', 'AO', 'AW', 'AY', 'B', 'CH', 'D', 'DH',
    'EH', 'ER', 'EY', 'F', 'G', 'HH', 'IH', 'IY', 'JH', 'K', 'L', 'M', 'N',
    'NG', 'OW', 'OY', 'P', 'R', 'S', 'SH', 'T', 'TH', 'UH', 'UW', 'V', 'W',
    'Y', 'Z', 'ZH', ' | ',
]
N_CLASSES = len(PHONEME_VOCAB)          # 41
BLANK_ID = 0
SIL_ID = N_CLASSES - 1                  # 40  (' | ')
PHONE2ID = {p.strip(): i for i, p in enumerate(PHONEME_VOCAB)}   # '|' -> 40


# -----------------------------------------------------------------------------
# OFFICIAL text normalisation + WER  (nejm-brain-to-text/model_training/
# evaluate_model_helpers.remove_punctuation  +  evaluate_model.py aggregate WER)
#   WER = (S + D + I) / N = sum_i editdistance(ref_i.split(), hyp_i.split())
#                           / sum_i len(ref_i.split())
# -----------------------------------------------------------------------------
def remove_punctuation(sentence):
    sentence = re.sub(r'[^a-zA-Z\- \']', '', sentence)
    sentence = sentence.replace('- ', ' ').lower()
    sentence = sentence.replace('--', '').lower()
    sentence = sentence.replace(" '", "'").lower()
    sentence = sentence.strip()
    sentence = ' '.join([word for word in sentence.split() if word != ''])
    return sentence


def official_wer(refs, hyps):
    """Corpus-level WER exactly as evaluate_model.py. Returns (wer, edits, n_ref_words)."""
    tot_ed, tot_n = 0, 0
    for r, h in zip(refs, hyps):
        rw = remove_punctuation(r or '').split()
        hw = remove_punctuation(h or '').split()
        tot_ed += editdistance.eval(rw, hw)
        tot_n += len(rw)
    return (tot_ed / max(tot_n, 1)), tot_ed, tot_n


def utt_word_errors(ref, hyp):
    rw = remove_punctuation(ref or '').split()
    return editdistance.eval(rw, remove_punctuation(hyp or '').split()), len(rw)


def official_per(pred_seqs, true_seqs):
    """Aggregate PER like rnn_trainer.validation: sum edit distance / sum true length."""
    ed = sum(editdistance.eval(list(p), list(t)) for p, t in zip(pred_seqs, true_seqs))
    n = sum(len(t) for t in true_seqs)
    return ed / max(n, 1), ed, n


# -----------------------------------------------------------------------------
# sessions / cache
# -----------------------------------------------------------------------------
def get_session2idx(data_dir):
    paths = glob(data_dir + '/**/data_*.hdf5', recursive=True)
    sessions = sorted(set(Path(p).parent.name for p in paths))
    return {s: i for i, s in enumerate(sessions)}


def trial_key(session, block, trial):
    return f'{session}|{int(block)}|{int(trial)}'


def build_cache(data_dir, cache_dir, splits=('train', 'val', 'test'), log=print):
    """One float16 flat binary per split + a meta pickle. Shared by every process
    through the OS page cache (np.memmap), so two training workers do not each
    hold a private ~8 GB copy of the neural data in RAM."""
    import h5py
    os.makedirs(cache_dir, exist_ok=True)
    for split in splits:
        done_flag = os.path.join(cache_dir, f'{split}.done')
        if os.path.exists(done_flag):
            log(f'[cache] {split}: already cached')
            continue
        files = sorted(glob(data_dir + f'/**/data_{split}.hdf5', recursive=True))
        bin_path = os.path.join(cache_dir, f'{split}_neural.f16')
        meta = []
        offset = 0
        t0 = time.time()
        with open(bin_path, 'wb') as fout:
            for fp in files:
                session = Path(fp).parent.name
                with h5py.File(fp, 'r') as f:
                    entries = []
                    for k in f.keys():
                        tr = f[k]
                        if 'input_features' not in tr:
                            continue
                        entries.append((int(tr.attrs.get('block_num', 0)),
                                        int(tr.attrs.get('trial_num', 0)), k))
                    entries.sort(key=lambda e: (e[0], e[1]))     # same order as v2 BLOCK 6
                    for block, trial, k in entries:
                        tr = f[k]
                        n = int(tr.attrs['n_time_steps'])
                        x = np.asarray(tr['input_features'][:n], dtype=np.float16)
                        fout.write(np.ascontiguousarray(x).tobytes())
                        s = tr.attrs.get('sentence_label', '')
                        s = s.decode('utf-8') if isinstance(s, bytes) else (s or '')
                        if 'seq_class_ids' in tr:
                            plen = int(tr.attrs['seq_len']) if 'seq_len' in tr.attrs else len(tr['seq_class_ids'])
                            ph = np.asarray(tr['seq_class_ids'][:plen], dtype=np.int16)
                        else:
                            ph = np.zeros(0, dtype=np.int16)
                        meta.append({'session': session, 'block': block, 'trial': trial,
                                     'n': n, 'offset': offset, 'sentence': s, 'phonemes': ph,
                                     'key': trial_key(session, block, trial)})
                        offset += n
        with open(os.path.join(cache_dir, f'{split}_meta.pkl'), 'wb') as f:
            pickle.dump({'meta': meta, 'n_frames': offset, 'n_feat': 512}, f)
        open(done_flag, 'w').close()
        log(f'[cache] {split}: {len(meta)} trials, {offset} frames, '
            f'{offset * 512 * 2 / 1e9:.2f} GB, {time.time() - t0:.0f}s')


class CacheReader:
    """Reads trials with os.pread (no np.memmap): nothing is mapped into the process
    address space, so RSS stays small and the OS can always reclaim the page cache."""
    def __init__(self, cache_dir, split):
        with open(os.path.join(cache_dir, f'{split}_meta.pkl'), 'rb') as f:
            d = pickle.load(f)
        self.meta = d['meta']
        self.n_feat = d['n_feat']
        self.n_frames = d['n_frames']
        self.path = os.path.join(cache_dir, f'{split}_neural.f16')
        self._fd, self._pid = None, None

    def _fdesc(self):
        if self._fd is None or self._pid != os.getpid():
            self._fd, self._pid = os.open(self.path, os.O_RDONLY), os.getpid()
        return self._fd

    def __getstate__(self):
        st = dict(self.__dict__); st['_fd'] = None; st['_pid'] = None
        return st

    def __len__(self):
        return len(self.meta)

    def read_frames(self, offset, n):
        nbytes = n * self.n_feat * 2
        buf = os.pread(self._fdesc(), nbytes, offset * self.n_feat * 2)
        return np.frombuffer(buf, dtype=np.float16).reshape(n, self.n_feat)

    def neural(self, i):
        m = self.meta[i]
        return self.read_frames(m['offset'], m['n'])


def compute_norm_stats(reader, chunk=200_000):
    n = reader.n_frames
    s1 = np.zeros(reader.n_feat, np.float64)
    s2 = np.zeros(reader.n_feat, np.float64)
    for a in range(0, n, chunk):
        x = reader.read_frames(a, min(chunk, n - a)).astype(np.float32)
        s1 += x.sum(0, dtype=np.float64)
        s2 += (x * x).sum(0, dtype=np.float64)
        del x
    mean = s1 / max(n, 1)
    std = np.sqrt(np.maximum(s2 / max(n, 1) - mean ** 2, 0))
    std[std < 1e-6] = 1.0
    return {'mean': torch.tensor(mean, dtype=torch.float32), 'std': torch.tensor(std, dtype=torch.float32)}


def pick_cache_dir(candidates, need_gb=float(os.environ.get('B2T_CACHE_NEED_GB', '14')), log=print):
    """first writable, non-tmpfs directory with enough free disk."""
    tmpfs = set()
    try:
        for line in open('/proc/mounts'):
            parts = line.split()
            if len(parts) > 2 and parts[2] in ('tmpfs', 'ramfs'):
                tmpfs.add(parts[1])
    except Exception:
        pass
    for c in candidates:
        try:
            os.makedirs(c, exist_ok=True)
            probe = os.path.join(c, '.probe'); open(probe, 'w').close(); os.remove(probe)
            mount = c
            while not os.path.ismount(mount) and mount != '/':
                mount = os.path.dirname(mount)
            st = os.statvfs(c)
            free_gb = st.f_bavail * st.f_frsize / 1e9
            done = os.path.exists(os.path.join(c, 'test.done'))
            if mount in tmpfs:
                log(f'[cache] {c}: RAM-backed (tmpfs) -> skipped'); continue
            if free_gb < need_gb and not done:
                log(f'[cache] {c}: only {free_gb:.1f} GB free -> skipped'); continue
            log(f'[cache] using {c} (mount {mount}, {free_gb:.1f} GB free)')
            return c
        except Exception as e:
            log(f'[cache] {c}: not usable ({e!r})')
    raise RuntimeError(f'no usable cache directory among {candidates}')


def make_crossfit_split(val_meta, pattern=('A', 'B', 'A', 'B', 'C'), seed=1337):
    """TRIAL-level split INSIDE every (session, block) of val.
    Why trial-level and not block-level: Kaggle test trials are interleaved with
    val trials inside the SAME blocks (see v2 BLOCK 5.1c: t15.2025.01.10 val block 8
    and test block 8, etc.). A model trained on part of a block and scored on the
    rest of that block is the honest proxy for 'trained on val, scored on test'."""
    rng = random.Random(seed)
    groups = {}
    for m in val_meta:
        groups.setdefault((m['session'], m['block']), []).append(m)
    labels = {}
    for g in sorted(groups):
        items = sorted(groups[g], key=lambda m: m['trial'])
        off = rng.randrange(len(pattern))
        for j, m in enumerate(items):
            labels[m['key']] = pattern[(j + off) % len(pattern)]
    return labels


# -----------------------------------------------------------------------------
# datasets / batching
# -----------------------------------------------------------------------------
class TrialSet(torch.utils.data.Dataset):
    """items = list of (reader_name, index)."""
    def __init__(self, readers, items, session2idx):
        self.readers = readers
        self.items = items
        self.session2idx = session2idx
        self.lengths = [readers[r].meta[i]['n'] for r, i in items]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, j):
        r, i = self.items[j]
        rd = self.readers[r]
        m = rd.meta[i]
        return {'neural': np.array(rd.neural(i), dtype=np.float16, copy=True), 'target': m['phonemes'].astype(np.int64),
                'day': self.session2idx[m['session']], 'j': j}


def collate_trials(batch):
    B = len(batch)
    T = max(b['neural'].shape[0] for b in batch)
    L = max(max(len(b['target']) for b in batch), 1)
    x = torch.zeros(B, T, batch[0]['neural'].shape[1], dtype=torch.float16)
    tg = torch.zeros(B, L, dtype=torch.long)
    lens, tlens, days, js = [], [], [], []
    for k, b in enumerate(batch):
        n = b['neural'].shape[0]
        x[k, :n] = torch.from_numpy(b['neural'])
        t = b['target']
        tg[k, :len(t)] = torch.from_numpy(t)
        lens.append(n); tlens.append(len(t)); days.append(b['day']); js.append(b['j'])
    return {'neural': x, 'target': tg, 'lengths': torch.tensor(lens), 'target_lengths': torch.tensor(tlens),
            'day_idx': torch.tensor(days), 'j': torch.tensor(js)}


class BucketBatchSampler(torch.utils.data.Sampler):
    """Length-bucketed batches capped by #trials AND #bins (bounded memory)."""
    def __init__(self, lengths, max_batch, max_bins, shuffle=True, seed=0):
        self.lengths = np.asarray(lengths)
        self.max_batch, self.max_bins, self.shuffle = max_batch, max_bins, shuffle
        self.rng = np.random.RandomState(seed)
        self._batches = self._make()

    def _make(self):
        L = self.lengths
        if self.shuffle:
            order = np.argsort(L * self.rng.uniform(0.9, 1.1, size=len(L)))
        else:
            order = np.argsort(L)
        batches, cur, cur_max = [], [], 0
        for i in order:
            m = max(cur_max, L[i])
            if cur and (len(cur) + 1 > self.max_batch or m * (len(cur) + 1) > self.max_bins):
                batches.append(cur); cur, m = [], L[i]
            cur.append(int(i)); cur_max = m
        if cur:
            batches.append(cur)
        if self.shuffle:
            self.rng.shuffle(batches)
        return batches

    def __iter__(self):
        batches = self._batches
        self._batches = self._make()
        return iter(batches)

    def __len__(self):
        return len(self._batches)


# -----------------------------------------------------------------------------
# GPU augmentation (train only)
# -----------------------------------------------------------------------------
def speed_perturb(x, lengths, lo, hi):
    """x [B,T,C] -> resampled along time by one factor per batch (keeps both CR views aligned)."""
    f = float(np.random.uniform(lo, hi))
    if abs(f - 1.0) < 1e-3:
        return x, lengths
    B, T, C = x.shape
    newT = max(int(round(T * f)), 2)
    y = F.interpolate(x.transpose(1, 2).float(), size=newT, mode='linear', align_corners=False)
    new_len = torch.clamp((lengths.float() * f).round().long(), min=1, max=newT)
    return y.transpose(1, 2).to(x.dtype), new_len


def view_augment(x, lengths, cfg):
    """independent per-view noise: white noise, constant offset, static gain, electrode dropout."""
    B, T, C = x.shape
    dev = x.device
    if cfg.get('static_gain_std', 0) > 0:
        x = x * (1.0 + torch.randn(B, 1, C, device=dev, dtype=x.dtype) * cfg['static_gain_std'])
    if cfg.get('white_noise_std', 0) > 0:
        x = x + torch.randn_like(x) * cfg['white_noise_std']
    if cfg.get('constant_offset_std', 0) > 0:
        x = x + torch.randn(B, 1, C, device=dev, dtype=x.dtype) * cfg['constant_offset_std']
    p = cfg.get('electrode_drop_p', 0.0)
    if p > 0 and C % 2 == 0:
        n_el = C // 2
        apply = (torch.rand(B, 1, device=dev) < cfg.get('electrode_drop_apply', 0.5)).to(x.dtype)
        keep_el = (torch.rand(B, n_el, device=dev) >= p).to(x.dtype)
        keep_el = 1.0 - apply * (1.0 - keep_el)                       # only for 'apply' trials
        keep = torch.cat([keep_el, keep_el], dim=1)                   # tx + sbp of the same electrode
        x = x * keep[:, None, :]
    return x


def make_time_mask(lengths, T, frac, min_span, max_span, device):
    """bool [B,T], True = masked. ~frac of each trial's valid bins, spans in [min_span,max_span].
    Built on CPU and moved once (per-span GPU slicing costs hundreds of kernel launches per step)."""
    B = len(lengths)
    mask = np.zeros((B, T), dtype=bool)
    if frac > 0:
        mean_span = 0.5 * (min_span + max_span)
        for b in range(B):
            n = int(lengths[b])
            k = int(round(frac * n / mean_span))
            if k <= 0 or n <= min_span:
                continue
            spans = np.random.randint(min_span, max_span + 1, size=k)
            starts = np.random.randint(0, max(n - min_span, 1), size=k)
            for s0, w in zip(starts, spans):
                mask[b, s0:min(s0 + w, n)] = True
    return torch.from_numpy(mask).to(device, non_blocking=True)


# -----------------------------------------------------------------------------
# model: B2TNetV4
#   v2 lineage (low-rank day adapter + generic slot, Patch -> BiGRU x4 -> Conformer x3,
#   inter-CTC heads) with:
#     * FIX: ConvModule GroupNorm(1,C) over [C,T] leaked padded-frame statistics
#       into valid frames -> replaced by per-frame LayerNorm (no time pooling).
#     * FIX: day adapter is an exact identity at init (V zero-init, gain param
#       decays to 1 instead of a free scale) so weight-decay pulls sessions toward
#       the shared solution rather than toward 0.
#     * NEW: in-model time masking (applied after smoothing, before patching).
#     * ResBiGRU uses enforce_sorted=False (CR-CTC batches are two concatenated views).
# -----------------------------------------------------------------------------
def drop_path(x, p, training):
    if p == 0.0 or not training:
        return x
    keep = 1 - p
    mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
    return x * mask / keep


class RoPE(nn.Module):
    def __init__(self, dim, max_len=8192, base=10000.0):
        super().__init__()
        inv = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        ang = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer('cos', ang.cos().repeat_interleave(2, -1), persistent=False)
        self.register_buffer('sin', ang.sin().repeat_interleave(2, -1), persistent=False)

    def forward(self, x):
        T = x.shape[-2]
        cos = self.cos[:T].to(x.dtype)
        sin = self.sin[:T].to(x.dtype)
        x_r = torch.stack([-x[..., 1::2], x[..., 0::2]], dim=-1).flatten(-2)
        return x * cos + x_r * sin


class MHSA(nn.Module):
    def __init__(self, d, h, p):
        super().__init__()
        self.h, self.dk, self.p = h, d // h, p
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d)
        self.rope = RoPE(self.dk)

    def forward(self, x, pad_mask):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.h, self.dk).transpose(1, 2) for t in (q, k, v)]
        q, k = self.rope(q), self.rope(k)
        o = F.scaled_dot_product_attention(q, k, v, attn_mask=pad_mask[:, None, None, :],
                                           dropout_p=self.p if self.training else 0.0)
        return self.out(o.transpose(1, 2).reshape(B, T, D))


class ConvModule(nn.Module):
    def __init__(self, d, kernel, p):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.pw1 = nn.Conv1d(d, 2 * d, 1)
        self.dw = nn.Conv1d(d, d, kernel, padding=kernel // 2, groups=d)
        self.dwnorm = nn.LayerNorm(d)          # per-frame: no padded-frame leakage
        self.act = nn.SiLU()
        self.pw2 = nn.Conv1d(d, d, 1)
        self.drop = nn.Dropout(p)

    def forward(self, x, pad_mask):
        m = pad_mask[:, :, None].to(x.dtype)
        y = (self.norm(x) * m).transpose(1, 2)
        y = self.dw(F.glu(self.pw1(y), dim=1))
        y = self.act(self.dwnorm(y.transpose(1, 2)) * m).transpose(1, 2)
        return self.drop(self.pw2(y).transpose(1, 2))


class ConformerBlock(nn.Module):
    def __init__(self, d, h, d_ff, kernel, p, p_attn):
        super().__init__()
        def ff():
            return nn.Sequential(nn.Linear(d, d_ff), nn.SiLU(), nn.Dropout(p), nn.Linear(d_ff, d), nn.Dropout(p))
        self.ff1_norm, self.ff1 = nn.LayerNorm(d), ff()
        self.attn_norm, self.attn, self.attn_drop = nn.LayerNorm(d), MHSA(d, h, p_attn), nn.Dropout(p)
        self.conv = ConvModule(d, kernel, p)
        self.ff2_norm, self.ff2 = nn.LayerNorm(d), ff()
        self.final_norm = nn.LayerNorm(d)

    def forward(self, x, pad_mask, dp=0.0):
        x = x + 0.5 * drop_path(self.ff1(self.ff1_norm(x)), dp, self.training)
        x = x + drop_path(self.attn_drop(self.attn(self.attn_norm(x), pad_mask)), dp, self.training)
        x = x + drop_path(self.conv(x, pad_mask), dp, self.training)
        x = x + 0.5 * drop_path(self.ff2(self.ff2_norm(x)), dp, self.training)
        return self.final_norm(x)


class ResBiGRU(nn.Module):
    def __init__(self, d, hidden, p):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.gru = nn.GRU(d, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(2 * hidden, d)
        self.drop = nn.Dropout(p)

    def forward(self, x, lengths_cpu):
        y = self.norm(x)
        packed = pack_padded_sequence(y, lengths_cpu, batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.shape[1])
        return x + self.drop(self.proj(out.to(x.dtype)))


class B2TNetV4(nn.Module):
    ARCH = 'B2TNetV4'

    def __init__(self, n_days, cfg):
        super().__init__()
        C = cfg['input_size']
        self.cfg = cfg
        self.patch, self.stride = cfg['patch_size'], cfg['patch_stride']
        self.n_days, self.generic_idx = n_days, n_days
        self.day_dropout_p = cfg.get('day_dropout_p', 0.0)
        D = n_days + 1

        k = np.zeros(cfg['smooth_kernel_size'], np.float32); k[len(k) // 2] = 1
        gk = gaussian_filter1d(k, cfg['smooth_kernel_std'])
        gk = np.squeeze(gk[np.argwhere(gk > 0.01)]); gk = gk / gk.sum()
        self.register_buffer('gauss_kernel', torch.tensor(gk, dtype=torch.float32).view(1, 1, -1))

        r = cfg.get('day_rank', 32)
        self.day_U = nn.Parameter(torch.randn(D, C, r) * 0.02)
        self.day_V = nn.Parameter(torch.zeros(D, r, C))          # exact identity at init
        self.day_bias = nn.Parameter(torch.zeros(D, 1, C))
        self.day_gain = nn.Parameter(torch.zeros(D, 1, C))       # scale = 1 + gain
        self.day_act = nn.Softsign()

        d = cfg['d_model']
        # patch embedding as a strided Conv1d == Linear on unfolded 14-bin patches, but it never
        # materialises the [B, T', 512*14] tensor (that tensor + its LayerNorm were ~40% of GPU memory)
        self.in_norm = nn.LayerNorm(C)
        self.in_conv = nn.Conv1d(C, d, kernel_size=self.patch, stride=self.stride)
        self.in_act = nn.GELU()
        self.in_drop = nn.Dropout(cfg['dropout'])
        self.grus = nn.ModuleList([ResBiGRU(d, cfg['gru_hidden'], cfg['dropout']) for _ in range(cfg['gru_layers'])])
        self.blocks = nn.ModuleList([ConformerBlock(d, cfg['n_heads'], cfg['d_ff'], cfg['conv_kernel'],
                                                    cfg['dropout'], cfg['attn_dropout'])
                                     for _ in range(cfg['n_conformer'])])
        self.dp_rates = [float(v) for v in torch.linspace(0, cfg['drop_path_rate'], cfg['n_conformer'])]
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, N_CLASSES)
        self.aux_head_gru = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, N_CLASSES))
        self.aux_head_mid = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, N_CLASSES))
        self.aux_mid_at = max(cfg['n_conformer'] - 1, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def day_parameters(self):
        return [self.day_U, self.day_V, self.day_bias, self.day_gain]

    def out_lengths(self, lengths):
        L = torch.clamp(lengths, min=self.patch)
        return torch.clamp((L - self.patch) // self.stride + 1, min=1)

    def forward(self, x, lengths, day_idx, time_mask=None, return_aux=False):
        B, T, C = x.shape
        if self.training and self.day_dropout_p > 0:
            drop = torch.rand(B, device=day_idx.device) < self.day_dropout_p
            if drop.any():
                day_idx = day_idx.clone(); day_idx[drop] = self.generic_idx
        U, V = self.day_U[day_idx].to(x.dtype), self.day_V[day_idx].to(x.dtype)
        x = x + torch.bmm(torch.bmm(x, U), V)
        x = x * (1.0 + self.day_gain[day_idx].to(x.dtype)) + self.day_bias[day_idx].to(x.dtype)
        x = self.day_act(x)

        x = x.transpose(1, 2)
        x = F.conv1d(x, self.gauss_kernel.repeat(C, 1, 1).to(x.dtype), padding='same', groups=C)
        if time_mask is not None:
            x = x.masked_fill(time_mask[:, None, :T], 0.0)
        if x.shape[-1] < self.patch:
            x = F.pad(x, (0, self.patch - x.shape[-1]))
        x = self.in_norm(x.transpose(1, 2)).transpose(1, 2)
        x = self.in_drop(self.in_act(self.in_conv(x))).transpose(1, 2)        # [B,T',d]

        out_len = torch.clamp(self.out_lengths(lengths.to(x.device)), max=x.shape[1])
        Tp = x.shape[1]
        pad_mask = torch.arange(Tp, device=x.device)[None, :] < out_len[:, None]
        x = x * pad_mask[:, :, None].to(x.dtype)
        lens_cpu = out_len.detach().cpu().clamp(min=1)
        for g in self.grus:
            x = g(x, lens_cpu)
        x = x * pad_mask[:, :, None].to(x.dtype)
        aux1 = self.aux_head_gru(x) if return_aux else None
        aux2 = None
        for i, blk in enumerate(self.blocks):
            x = blk(x, pad_mask, dp=self.dp_rates[i]) * pad_mask[:, :, None].to(x.dtype)
            if return_aux and (i + 1) == self.aux_mid_at:
                aux2 = self.aux_head_mid(x)
        lp = torch.log_softmax(self.head(self.norm(x)).float(), dim=-1).transpose(0, 1)   # [T',B,C]
        if return_aux:
            auxs = [torch.log_softmax(a.float(), -1).transpose(0, 1) for a in (aux1, aux2) if a is not None]
            return lp, out_len, auxs
        return lp, out_len


# -----------------------------------------------------------------------------
# losses
# -----------------------------------------------------------------------------
def ctc_mean(lp, targets, out_len, tgt_len):
    return F.ctc_loss(lp, targets, out_len, tgt_len, blank=BLANK_ID, reduction='mean', zero_infinity=True)


def cr_ctc_consistency(lp, out_len, tgt_len, B):
    """CR-CTC symmetric KL between the two views' frame posteriors (stop-grad target).
    Summed over valid frames per utterance, divided by target length (same scale as
    ctc reduction='mean'), averaged over utterances."""
    lp1, lp2 = lp[:, :B], lp[:, B:]
    T = lp.shape[0]
    valid = (torch.arange(T, device=lp.device)[:, None] < out_len[None, :B]).float()     # [T,B]
    kl12 = F.kl_div(lp1, lp2.detach(), log_target=True, reduction='none').sum(-1)     # KL(p2||p1)
    kl21 = F.kl_div(lp2, lp1.detach(), log_target=True, reduction='none').sum(-1)
    kl = 0.5 * (kl12 + kl21) * valid
    return (kl.sum(0) / tgt_len[:B].clamp(min=1).float()).mean()


# -----------------------------------------------------------------------------
# EMA
# -----------------------------------------------------------------------------
class EMA:
    def __init__(self, model, decay):
        import copy
        self.decay = decay
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.n = 0

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        pe = [p for p in self.model.parameters()]
        pm = [p.detach() for p in model.parameters()]
        try:
            torch._foreach_lerp_(pe, pm, 1.0 - d)
        except Exception:
            for a, b in zip(pe, pm):
                a.mul_(d).add_(b, alpha=1 - d)


# -----------------------------------------------------------------------------
# lexicon + greedy decoding
# -----------------------------------------------------------------------------
def load_lexicon(path):
    """-> word2prons {word: [tuple(phones), ...]} (order preserved), pron2words {tuple: [words]}"""
    word2prons, pron2words = {}, {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if '\t' in line:
                w, rest = line.split('\t', 1)
                ph = [p for p in rest.replace('|', ' ').split() if p]
            else:
                parts = line.split()
                w, ph = parts[0], [p for p in parts[1:] if p != '|']
            if not ph:
                continue
            ph = tuple(ph)
            if any(p not in PHONE2ID for p in ph):
                continue
            lst = word2prons.setdefault(w, [])
            if ph not in lst:
                lst.append(ph)
            pw = pron2words.setdefault(ph, [])
            if w not in pw:
                pw.append(w)
    return word2prons, pron2words


def greedy_collapse(lp_tb, out_len):
    """lp [T,B,C] -> list of collapsed id lists (blank removed, repeats merged)."""
    am = lp_tb.argmax(-1).transpose(0, 1).cpu().numpy()
    res = []
    for b in range(am.shape[0]):
        seq = am[b, :int(out_len[b])]
        out, prev = [], -1
        for t in seq:
            t = int(t)
            if t != BLANK_ID and t != prev:
                out.append(t)
            prev = t
        res.append(out)
    return res


def phones_to_words(ids, pron2words, word_rank=None):
    words, cur = [], []

    def flush():
        if cur:
            ws = pron2words.get(tuple(PHONEME_VOCAB[p].strip() for p in cur))
            if not ws:
                words.append('<unk>')
            else:
                words.append(min(ws, key=lambda w: word_rank.get(w, 1e9)) if word_rank else ws[0])
    for p in ids:
        if p == SIL_ID:
            flush(); cur = []
        else:
            cur.append(p)
    flush()
    return ' '.join(words)


@torch.no_grad()
def forward_batch(model, batch, device, norm, clip, day_override=None, amp=True):
    x = batch['neural'].to(device, non_blocking=True).float()
    x = torch.clamp((x - norm['mean'].to(device)) / norm['std'].to(device), -clip, clip)
    lengths = batch['lengths'].to(device)
    day = batch['day_idx'].to(device)
    if day_override is not None:
        day = day_override[day.cpu()].to(device)
    with torch.autocast('cuda', dtype=torch.float16, enabled=(amp and x.is_cuda)):
        lp, out_len = model(x, lengths, day)
    return lp.float(), out_len


def human_time(s):
    s = int(max(s, 0))
    return f'{s // 3600:d}h{(s % 3600) // 60:02d}m'

Writing /kaggle/working/b2t_code/b2t_core.py


In [6]:
%%writefile {CODE_DIR}/train_worker.py
# =============================================================================
# train_worker.py  --  one process = one GPU = one cross-fit fold   (v6.1, crash-proof)
#   python train_worker.py --config run_foldA_seed0.json [--resume]
#
# Robustness design (v6.0 was OOM-killed by the OS after 27 min):
#   * no KenLM / flashlight in the worker (it was the biggest host-RAM user)
#   * no DataLoader subprocesses, no pinned memory, no memmap (pread per trial)
#   * CUDA OOM on a batch -> batch skipped, bins-per-batch cap lowered, training continues
#   * evaluation errors never stop training
#   * resume state every `resume_every_min` minutes (the notebook relaunches with --resume)
#   * every checkpoint is written to a temp file and atomically renamed
# =============================================================================
import argparse, json, os, sys, time, math, random, gc, traceback
os.environ.setdefault('TORCH_CUDNN_V8_API_LRU_CACHE_LIMIT', '64')     # v6.2: bound cuDNN shape-keyed plan cache
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch
import torch.nn.functional as F
from b2t_core import *   # noqa


def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)


def mem_status():
    s = ''
    try:
        import psutil
        vm = psutil.virtual_memory()
        s += f'RAM avail {vm.available / 1e9:.1f}G rss {psutil.Process().memory_info().rss / 1e9:.1f}G'
    except Exception:
        pass
    if torch.cuda.is_available():
        s += f' | GPU alloc {torch.cuda.memory_allocated() / 1e9:.1f}G peak {torch.cuda.max_memory_allocated() / 1e9:.1f}G'
    return s


RECYCLE_EXIT_CODE = 75     # v6.2 memory guard: 'state saved, please relaunch me with --resume'


def rss_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1e9, psutil.virtual_memory().available / 1e9
    except Exception:
        return 0.0, 1e9


def atomic_save(obj, path):
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


def is_oom(e):
    msg = str(e)
    return isinstance(e, torch.cuda.OutOfMemoryError) or 'out of memory' in msg.lower() \
        or 'ALLOC_FAILED' in msg


@torch.no_grad()
def evaluate(model, hold_ds, readers, norm, cfg, device, pron2words):
    model.eval()
    preds, trues, refs, hyps = [], [], [], []
    for idx in BucketBatchSampler(hold_ds.lengths, 32, 32 * 2000, False, 0):
        batch = collate_trials([hold_ds[i] for i in idx])
        lp, out_len = forward_batch(model, batch, device, norm, cfg['clip'])
        ids = greedy_collapse(lp, out_len)
        for k, j in enumerate(batch['j'].tolist()):
            r, i = hold_ds.items[j]
            m = readers[r].meta[i]
            preds.append(ids[k]); trues.append([int(p) for p in m['phonemes']])
            refs.append(m['sentence']); hyps.append(phones_to_words(ids[k], pron2words))
    return {'per': float(official_per(preds, trues)[0]), 'greedy_wer': float(official_wer(refs, hyps)[0]),
            'n': len(refs)}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--config', required=True)
    ap.add_argument('--resume', action='store_true')
    args = ap.parse_args()
    cfg = json.load(open(args.config))
    out_dir = cfg['out_dir']; os.makedirs(out_dir, exist_ok=True)
    resume_dir = cfg['resume_dir']; os.makedirs(resume_dir, exist_ok=True)
    tag = f"fold{cfg['fold']}_seed{cfg['seed']}"
    resume_path = os.path.join(resume_dir, f'{tag}_resume.pt')
    seed = int(cfg['seed'])
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    torch.backends.cudnn.benchmark = False
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
    except Exception:
        pass
    deadline = float(cfg['deadline_ts'])
    train_end = deadline - float(cfg['final_reserve_s'])

    readers = {'train': CacheReader(cfg['cache_dir'], 'train'), 'val': CacheReader(cfg['cache_dir'], 'val')}
    session2idx = cfg['session2idx']; n_days = len(session2idx)
    norm = torch.load(os.path.join(cfg['cache_dir'], 'norm_stats.pt'), weights_only=False)
    labels = json.load(open(cfg['split_path']))['labels']
    fold_labels = set(cfg['train_val_labels'])

    train_items = [('train', i) for i, m in enumerate(readers['train'].meta) if len(m['phonemes']) > 0]
    hold_items = []
    for i, m in enumerate(readers['val'].meta):
        if len(m['phonemes']) == 0:
            continue
        (train_items if labels.get(m['key']) in fold_labels else hold_items).append(('val', i))
    train_ds = TrialSet(readers, train_items, session2idx)
    hold_ds = TrialSet(readers, hold_items, session2idx)
    trained_days = sorted(set(session2idx[readers[r].meta[i]['session']] for r, i in train_items))
    json.dump({'untrained_days': [d for d in range(n_days) if d not in set(trained_days)], 'trained_days': trained_days},
              open(os.path.join(out_dir, 'trained_days.json'), 'w'))

    mcfg = cfg['model']
    use_amp = device == 'cuda'

    def build_all():
        mdl = B2TNetV4(n_days, mcfg).to(device)
        day_ids = set(id(p) for p in mdl.day_parameters())
        decay = [p for p in mdl.parameters() if id(p) not in day_ids and p.ndim >= 2]
        no_decay = [p for p in mdl.parameters() if id(p) not in day_ids and p.ndim < 2]
        o = torch.optim.AdamW([
            {'params': decay, 'weight_decay': cfg['weight_decay'], 'lr_scale': 1.0},
            {'params': no_decay, 'weight_decay': 0.0, 'lr_scale': 1.0},
            {'params': mdl.day_parameters(), 'weight_decay': cfg['weight_decay_day'], 'lr_scale': cfg['lr_day_scale']},
        ], lr=cfg['lr_max'], betas=(0.9, 0.98), eps=1e-6)
        return mdl, o, torch.amp.GradScaler('cuda', enabled=use_amp)

    model, opt, scaler = build_all()

    st = {'step': 0, 'epoch': 0, 't_start': time.time(), 't_warm_end': None, 'best_sel': float('inf'),
          'best_per': float('inf'), 'max_bins': int(cfg['max_bins_per_batch']), 'restarts': 0, 'oom_skips': 0,
          'nan_skips': 0}
    resumed = False
    ema = None
    if args.resume and os.path.exists(resume_path):
        try:
            ck = torch.load(resume_path, map_location=device, weights_only=False)
            model.load_state_dict(ck['model'])
            opt.load_state_dict(ck['opt'])
            scaler.load_state_dict(ck['scaler'])
            ema = EMA(model, cfg['ema_decay'])
            ema.model.load_state_dict(ck['ema']); ema.n = ck['ema_n']
            st.update(ck['state']); st['restarts'] += 1
            resumed = True
            del ck
            log(f'[{tag}] RESUMED from {resume_path} at step {st["step"]} epoch {st["epoch"]} (restart #{st["restarts"]})')
        except Exception as e:
            log(f'[{tag}] resume state unreadable ({e!r}) -> starting fresh')
            model, opt, scaler = build_all()
            ema = None
    elif args.resume:
        log(f'[{tag}] --resume given but no resume state yet -> starting fresh')
    if not resumed:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        if device == 'cuda':
            torch.cuda.manual_seed_all(seed)
        if cfg.get('init_from'):
            ck = torch.load(cfg['init_from'], map_location=device, weights_only=False)
            res = model.load_state_dict(ck['model_state_dict'], strict=False)
            if res.missing_keys or res.unexpected_keys:
                raise RuntimeError(f'init_from {cfg["init_from"]} does not match the model: '
                                   f'missing={res.missing_keys[:5]} unexpected={res.unexpected_keys[:5]}')
            log(f'[{tag}] initialised from {cfg["init_from"]} (0 missing / 0 unexpected keys; source step '
                f'{ck.get("step")}, epoch {ck.get("epoch")}, metrics {ck.get("metrics", {}).get("per")})')
            del ck
        ema = EMA(model, cfg['ema_decay'])
    else:
        rs = seed + 1000 * st['restarts']
        random.seed(rs); np.random.seed(rs); torch.manual_seed(rs)

    log(f'[{tag}] device={device} {torch.cuda.get_device_name(0) if device == "cuda" else ""} | '
        f'params {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M | train trials {len(train_items)} '
        f'(val labels {sorted(fold_labels)}) | holdout {len(hold_items)} | train-end in {human_time(train_end - time.time())} | '
        f'{mem_status()}')

    pron2words = load_lexicon(cfg['lexicon_path'])[1] if cfg.get('lexicon_path') else {}
    aug = cfg['aug']
    metrics_path = os.path.join(out_dir, f'{tag}_metrics.jsonl')
    warmup = int(cfg['warmup_steps'])
    run_loss = {'ctc': 0.0, 'aux': 0.0, 'cr': 0.0, 'n': 0}
    # v6.2: timers are part of the resume state, so restarts do not postpone evaluations
    next_eval = st.get('next_eval_ts') or (time.time() + 60 * cfg['eval_every_min'])
    next_resume = st.get('next_resume_ts') or (time.time() + 60 * cfg['resume_every_min'])
    st['next_eval_ts'], st['next_resume_ts'] = next_eval, next_resume
    rss0, _ = rss_gb()
    rss_limit = max(float(cfg.get('rss_limit_gb', 1e9)), rss0 + float(os.environ.get('B2T_RSS_MARGIN_GB', '2.0')))
    min_avail = float(cfg.get('min_avail_gb', 0.0))
    steps_this_life = 0
    log(f'[{tag}] memory guard: recycle above RSS {rss_limit:.1f} GB or below {min_avail:.1f} GB available '
        f'(RSS now {rss0:.1f} GB)')
    epoch_times = []

    def lr_now():
        if st['step'] < warmup:
            return cfg['lr_max'] * (st['step'] + 1) / warmup
        if st['t_warm_end'] is None:
            st['t_warm_end'] = time.time()
        prog = (time.time() - st['t_warm_end']) / max(train_end - st['t_warm_end'], 1.0)
        prog = min(max(prog, 0.0), 1.0)
        return cfg['lr_min'] + 0.5 * (cfg['lr_max'] - cfg['lr_min']) * (1 + math.cos(math.pi * prog))

    def save_ckpt(name, metrics):
        atomic_save({'model_state_dict': ema.model.state_dict(), 'config': mcfg, 'n_days': n_days,
                     'arch': 'B2TNetV4', 'fold': cfg['fold'], 'seed': seed,
                     'train_val_labels': sorted(fold_labels), 'epoch': st['epoch'], 'step': st['step'],
                     'metrics': metrics}, os.path.join(out_dir, f'{tag}_{name}.pt'))

    def save_resume():
        try:
            atomic_save({'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(),
                         'ema': ema.model.state_dict(), 'ema_n': ema.n, 'state': dict(st)}, resume_path)
        except Exception as e:
            log(f'[{tag}] resume save failed (training continues): {e!r}')

    def do_eval(final=False):
        try:
            t0 = time.time()
            m = evaluate(ema.model, hold_ds, readers, norm, cfg, device, pron2words)
            m.update({'epoch': st['epoch'], 'step': st['step'], 'hours': (time.time() - st['t_start']) / 3600,
                      'lr': float(lr_now()), 'train_ctc': run_loss['ctc'] / max(run_loss['n'], 1),
                      'train_aux': run_loss['aux'] / max(run_loss['n'], 1),
                      'train_cr': run_loss['cr'] / max(run_loss['n'], 1), 'eval_s': time.time() - t0,
                      'max_bins': st['max_bins'], 'oom_skips': st['oom_skips'], 'nan_skips': st['nan_skips']})
            flag = ''
            if m['greedy_wer'] < st['best_sel'] or (m['greedy_wer'] == st['best_sel'] and m['per'] < st['best_per']):
                st['best_sel'] = m['greedy_wer']; save_ckpt('best_wer', m); flag += ' *best_wer*'
            if m['per'] < st['best_per']:
                st['best_per'] = m['per']; save_ckpt('best_per', m); flag += ' *best_per*'
            if final:
                save_ckpt('ema_final', m)
            with open(metrics_path, 'a') as f:
                f.write(json.dumps(m) + '\n')
            log(f"[{tag}] EVAL ep{st['epoch']} step{st['step']} {m['hours']:.2f}h | PER {m['per'] * 100:.2f}% | "
                f"greedyWER {m['greedy_wer'] * 100:.2f}% | ctc {m['train_ctc']:.3f} aux {m['train_aux']:.3f} "
                f"cr {m['train_cr']:.3f} | lr {m['lr']:.2e} | {m['eval_s']:.0f}s{flag} | {mem_status()}")
        except Exception as e:
            log(f'[{tag}] evaluation failed (training continues): {e!r}\n{traceback.format_exc()}')
            if final:
                try:
                    save_ckpt('ema_final', {})
                except Exception:
                    pass
        finally:
            for k in run_loss:
                run_loss[k] = 0.0
            model.train()
            if device == 'cuda':
                torch.cuda.empty_cache()

    fake_crash = int(os.environ.get('B2T_FAKE_CRASH_STEP', '-1'))     # local smoke test only
    fake_oom = int(os.environ.get('B2T_FAKE_OOM_STEP', '-1'))         # local smoke test only
    model.train()
    stop = False
    while not stop:
        st['epoch'] += 1
        te = time.time()
        sampler = BucketBatchSampler(train_ds.lengths, cfg['batch_size'], st['max_bins'], True,
                                     seed * 7919 + st['epoch'] + 100 * st['restarts'])
        for idx in sampler:
            if time.time() >= train_end:
                stop = True
                break
            if st['step'] == fake_crash and not resumed:
                log(f'[{tag}] (smoke test) simulated hard crash'); sys.stdout.flush(); os._exit(137)
            batch = collate_trials([train_ds[i] for i in idx])
            lr = lr_now()
            for g in opt.param_groups:
                g['lr'] = lr * g['lr_scale']
            xv = lp = auxs = loss = ctc = aux = cr = tmask = None
            try:
                if st['step'] == fake_oom:
                    fake_oom = -1
                    raise torch.cuda.OutOfMemoryError('CUDA out of memory (smoke test)')
                x = batch['neural'].to(device).float()
                x = torch.clamp((x - norm['mean'].to(device)) / norm['std'].to(device), -cfg['clip'], cfg['clip'])
                lengths = batch['lengths'].to(device)
                tgt, tlen = batch['target'].to(device), batch['target_lengths'].to(device)
                day = batch['day_idx'].to(device)
                B = x.shape[0]
                if aug['random_cut'] > 0:
                    cut = int(np.random.randint(0, aug['random_cut']))
                    if cut and x.shape[1] - cut > 20:
                        x = x[:, cut:]; lengths = (lengths - cut).clamp(min=1)
                if np.random.rand() < aug['speed_p']:
                    x, lengths = speed_perturb(x, lengths, aug['speed_lo'], aug['speed_hi'])
                n_views = 2 if cfg['cr_weight'] > 0 else 1
                xv = torch.cat([view_augment(x, lengths, aug) for _ in range(n_views)], 0)
                del x
                lv, dv = lengths.repeat(n_views), day.repeat(n_views)
                tv, tlv = tgt.repeat(n_views, 1), tlen.repeat(n_views)
                tmask = make_time_mask(lv.tolist(), xv.shape[1], aug['time_mask_frac'], aug['time_mask_min'],
                                       aug['time_mask_max'], device)
                with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                    lp, out_len, auxs = model(xv, lv, dv, time_mask=tmask, return_aux=True)
                ctc = ctc_mean(lp, tv, out_len, tlv)
                aux = torch.stack([ctc_mean(a, tv, out_len, tlv) for a in auxs]).mean() if auxs else ctc * 0
                loss = (1 - cfg['interctc_weight']) * ctc + cfg['interctc_weight'] * aux
                cr = cr_ctc_consistency(lp, out_len, tlv, B) if n_views == 2 else ctc * 0
                loss = loss + cfg['cr_weight'] * cr
                opt.zero_grad(set_to_none=True)
                if not torch.isfinite(loss):
                    st['nan_skips'] += 1
                    if st['nan_skips'] % 20 == 1:
                        log(f'[{tag}] non-finite loss, batch skipped (total {st["nan_skips"]})')
                    continue
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
                scaler.step(opt)
                scaler.update()
                ema.update(model)
                st['step'] += 1
                run_loss['ctc'] += float(ctc); run_loss['aux'] += float(aux); run_loss['cr'] += float(cr)
                run_loss['n'] += 1
            except Exception as e:
                if not is_oom(e):
                    raise
                st['oom_skips'] += 1
                xv = lp = auxs = loss = ctc = aux = cr = tmask = None
                opt.zero_grad(set_to_none=True)
                gc.collect()
                if device == 'cuda':
                    torch.cuda.empty_cache()
                # a fresh scaler (same scale) avoids "unscale_() already called" if OOM hit mid-step
                scaler = torch.amp.GradScaler('cuda', init_scale=float(scaler.get_scale()) if use_amp else 2.0 ** 16,
                                              enabled=use_amp)
                st['max_bins'] = max(int(st['max_bins'] * 0.85), 4000)
                log(f'[{tag}] CUDA OOM on batch (T={int(batch["lengths"].max())}, B={len(idx)}) -> skipped, '
                    f'max_bins_per_batch now {st["max_bins"]} (from next epoch) | {mem_status()}')
                continue
            if st['step'] % cfg['log_every'] == 0:
                log(f"[{tag}] ep{st['epoch']} step{st['step']} loss {float(loss):.3f} ctc {float(ctc):.3f} "
                    f"cr {float(cr):.4f} lr {lr:.2e} | train-end in {human_time(train_end - time.time())} | {mem_status()}")
            now = time.time()
            steps_this_life += 1
            if now >= next_eval:
                do_eval()
                late = (now - st['t_start']) / max(train_end - st['t_start'], 1) > cfg['late_frac']
                next_eval = time.time() + 60 * (cfg['eval_every_min_late'] if late else cfg['eval_every_min'])
                st['next_eval_ts'] = next_eval
            if now >= next_resume:
                next_resume = time.time() + 60 * cfg['resume_every_min']
                st['next_resume_ts'] = next_resume
                save_resume()
            if steps_this_life % 10 == 0 and steps_this_life >= 50:
                r, av = rss_gb()
                if (r > rss_limit or av < min_avail) and (deadline - now > 180 or av < 1.5):   # keep the final eval
                    st['next_eval_ts'], st['next_resume_ts'] = next_eval, next_resume
                    save_resume()
                    log(f'[{tag}] MEMORY GUARD: RSS {r:.1f} GB / available {av:.1f} GB at step {st["step"]} -> '
                        f'state saved, exiting for a clean relaunch (code {RECYCLE_EXIT_CODE})')
                    sys.stdout.flush()
                    os._exit(RECYCLE_EXIT_CODE)
        if not stop:
            epoch_times.append(time.time() - te)
            if len(epoch_times) in (2, 5) or st['epoch'] % 25 == 0:
                ept = float(np.median(epoch_times[-5:]))
                rem = max(train_end - time.time(), 0)
                log(f"[{tag}] BUDGET: epoch time ~{ept / 60:.2f} min -> ~{int(rem / max(ept, 1e-6))} more epochs, "
                    f"projected TOTAL ~{st['epoch'] + int(rem / max(ept, 1e-6))} epochs "
                    f"(time-based cosine lands on lr_min at the deadline)")
    log(f"[{tag}] training loop finished at epoch {st['epoch']}, step {st['step']}; final EMA eval...")
    do_eval(final=True)
    log(f"[{tag}] DONE. best greedyWER {st['best_sel'] * 100:.2f}% | best PER {st['best_per'] * 100:.2f}% | "
        f"OOM skips {st['oom_skips']} | NaN skips {st['nan_skips']} | restarts {st['restarts']}")


if __name__ == '__main__':
    try:
        main()
    except SystemExit:
        raise
    except BaseException:
        print(time.strftime('%H:%M:%S'), 'FATAL worker exception:\n' + traceback.format_exc(), flush=True)
        sys.exit(3)

Writing /kaggle/working/b2t_code/train_worker.py


In [7]:
%%writefile {CODE_DIR}/b2t_decode.py
# =============================================================================
# b2t_decode.py  --  v6 "llm" stage
#   1) candidate POOL  : flashlight beam n-best from every out-of-fold acoustic
#                        model x several lm_weights (deduplicated text)
#   2) EXPANSION       : single-word phonetic-neighbour substitutions (phoneme
#                        edit distance <= 1, homophones included) of the top hyps
#                        -> puts words that never survived the beam into the pool
#   3) EXACT FEATURES  : acoustic log P(phones | x) by CTC forward (alignment-free,
#                        so multi-seed/fold ensembling is an average of exact
#                        sequence log-likelihoods -- NOT posterior averaging, which
#                        destroyed WER in v2 (66%) because CTC spikes of independent
#                        models are not time-aligned), KenLM ln P(text), task-adapted
#                        LLM ln P(text), #words
#   4) TUNING          : log-linear weights + fluency gate chosen on out-of-fold val
#                        by exact corpus WER (vectorised), flat-region smoothing
# =============================================================================
import os, sys, json, math, time, contextlib, threading, copy, random, gc
import numpy as np
import torch
import torch.nn.functional as F
import editdistance

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from b2t_core import (PHONE2ID, SIL_ID, BLANK_ID, N_CLASSES, load_lexicon, remove_punctuation,
                      official_wer, utt_word_errors)

LN10 = math.log(10.0)


@contextlib.contextmanager
def silence_fd():
    sys.stdout.flush(); sys.stderr.flush()
    so, se = os.dup(1), os.dup(2)
    nul = os.open(os.devnull, os.O_WRONLY)
    try:
        os.dup2(nul, 1); os.dup2(nul, 2)
        yield
    finally:
        os.dup2(so, 1); os.dup2(se, 2); os.close(nul); os.close(so); os.close(se)


# -----------------------------------------------------------------------------
# 1) flashlight beam search in a process pool (CPU)
# -----------------------------------------------------------------------------
class FLDecoder:
    """flashlight-text lexicon beam search used directly (same construction as
    torchaudio.models.decoder.ctc_decoder, which is not guaranteed on new torch images).
    ONE KenLM + ONE trie are shared by all lm_weight decoders -> a single LM copy per process."""
    def __init__(self, lexicon, tokens, lm, lm_weights, beam, nbest, word_score=0.0, beam_threshold=50.0):
        from flashlight.lib.text.decoder import (CriterionType, LexiconDecoder, LexiconDecoderOptions, Trie,
                                                 SmearingMode, ZeroLM)
        from flashlight.lib.text.dictionary import create_word_dict, Dictionary, load_words
        KenLM = None
        try:
            from flashlight.lib.text.decoder.kenlm import KenLM
        except Exception:
            try:
                from flashlight.lib.text.decoder import KenLM
            except Exception:
                KenLM = None
        self.tokens = Dictionary(tokens)
        lex = load_words(lexicon)
        self.word_dict = create_word_dict(lex)
        if lm and KenLM is not None:
            self.lm = KenLM(lm, self.word_dict)
        else:
            if lm:
                print('WARNING: flashlight built without KenLM -> beam search without n-gram LM', flush=True)
            self.lm = ZeroLM()
        sil = self.tokens.get_index('|')
        blank = self.tokens.get_index('BLANK')
        trie = Trie(self.tokens.index_size(), sil)
        start = self.lm.start(False)
        for word, spellings in lex.items():
            wi = self.word_dict.get_index(word)
            _, sc = self.lm.score(start, wi)
            for sp in spellings:
                trie.insert([self.tokens.get_index(t) for t in sp], wi, sc)
        trie.smear(SmearingMode.MAX)
        self.trie = trie
        unk = self.word_dict.get_index('<unk>')
        self.nbest = nbest
        self.decoders = {}
        for lw in lm_weights:
            opts = LexiconDecoderOptions(beam_size=beam, beam_size_token=self.tokens.index_size(),
                                         beam_threshold=beam_threshold, lm_weight=lw, word_score=word_score,
                                         unk_score=float('-inf'), sil_score=0.0, log_add=False,
                                         criterion_type=CriterionType.CTC)
            self.decoders[lw] = LexiconDecoder(opts, trie, self.lm, sil, blank, unk, [], False)

    def decode(self, em):
        """em: np [T, C] -> {lm_weight: [unique word strings, best first]}"""
        em = np.ascontiguousarray(em, dtype=np.float32)
        T, N = em.shape
        out = {}
        for lw, dec in self.decoders.items():
            texts, seen = [], set()
            if T > 0:
                for r in dec.decode(em.ctypes.data, T, N)[:self.nbest]:
                    t = ' '.join(self.word_dict.get_entry(x) for x in r.words if x >= 0)
                    if t not in seen:
                        seen.add(t); texts.append(t)
            out[lw] = texts
        return out


_DEC = None


def _init_beam_worker(spec):
    global _DEC
    try:
        torch.set_num_threads(1)
    except Exception:
        pass
    with silence_fd():
        _DEC = FLDecoder(spec['lexicon'], spec['tokens'], spec['lm'], spec['lm_weights'], spec['beam'],
                         spec['nbest'], spec.get('word_score', 0.0))


def _beam_task(task):
    key, em = task
    try:
        return key, _DEC.decode(em)
    except Exception as e:
        print(f'beam task {key} failed: {e!r}', flush=True)
        return key, {lw: [] for lw in _DEC.decoders}


def safe_n_proc(requested, lm_path, log=print):
    """never start more LM-holding processes than host RAM allows."""
    n = max(1, int(requested))
    try:
        import psutil
        avail = psutil.virtual_memory().available
        lm_bytes = os.path.getsize(lm_path) if lm_path and os.path.exists(lm_path) else 0
        per_proc = 1.3 * lm_bytes + 1.5e9
        fit = int((avail - 5e9) // per_proc)
        if os.environ.get('B2T_FORCE_NPROC'):                       # local smoke test only
            fit = int(os.environ['B2T_FORCE_NPROC'])
        n = max(1, min(n, fit, os.cpu_count() or 1))
        if os.environ.get('B2T_FORCE_NPROC'):
            n = int(os.environ['B2T_FORCE_NPROC'])
        log(f'  beam processes: {n} (requested {requested}; RAM available {avail / 1e9:.1f} GB, '
            f'KenLM file {lm_bytes / 1e9:.2f} GB, budget {per_proc / 1e9:.1f} GB/process)')
    except Exception as e:
        log(f'  beam processes: {n} (RAM check unavailable: {e!r})')
    return n


def _beam_chunk(chunk):
    if os.environ.get('B2T_KILL_WORKER') and random.random() < 0.3:   # local smoke test only
        os._exit(9)
    return [_beam_task(t) for t in chunk]


def _ram_str():
    try:
        import psutil
        return f' | RAM {psutil.virtual_memory().percent:.0f}%'
    except Exception:
        return ''


def run_beam_pool(tasks, spec, n_proc=2, log=print, desc='beam', chunk=16):
    """tasks: list of (key, np.array[T,C]) -> {key: {lm_weight: [texts...]}}
    ProcessPoolExecutor detects dead workers (BrokenProcessPool); anything not finished is
    decoded serially in this process, so a killed worker can never hang or abort the run."""
    results, t0 = {}, time.time()
    n = len(tasks)
    if n == 0:
        return results
    report = max(n // 10, 1)
    last = [0]

    def progress():
        if len(results) - last[0] >= report or len(results) == n:
            last[0] = len(results)
            log(f'  [{desc}] {len(results)}/{n}  {time.time() - t0:.0f}s{_ram_str()}')

    n_proc = safe_n_proc(n_proc, spec.get('lm'), log) if n_proc > 1 else 1
    if n_proc > 1:
        try:
            import multiprocessing as mp
            from concurrent.futures import ProcessPoolExecutor, as_completed
            chunks = [tasks[a:a + chunk] for a in range(0, n, chunk)]
            with ProcessPoolExecutor(max_workers=n_proc, mp_context=mp.get_context('spawn'),
                                     initializer=_init_beam_worker, initargs=(spec,)) as ex:
                futs = [ex.submit(_beam_chunk, c) for c in chunks]
                for f in as_completed(futs):
                    for key, out in f.result():
                        results[key] = out
                    progress()
        except Exception as e:
            log(f'  [{desc}] process pool stopped ({e!r}); {n - len(results)} tasks left -> serial')
    todo = [t for t in tasks if t[0] not in results]
    if todo:
        _init_beam_worker(spec)
        for t in todo:
            key, out = _beam_task(t)
            results[key] = out
            progress()
    return results


# -----------------------------------------------------------------------------
# n-gram scorer (KenLM python binding)
# -----------------------------------------------------------------------------
class NgramScorer:
    """KenLM python binding; the model can be released (m=None) and is reloaded on demand."""
    def __init__(self, path):
        self.path = path
        self.m = None
        self._load()
        self.cache, self.uni = {}, {}

    def _load(self):
        import kenlm
        self.m = kenlm.Model(self.path)

    def ln(self, text):
        v = self.cache.get(text)
        if v is None:
            if self.m is None:
                self._load()
            v = self.m.score(text, bos=True, eos=True) * LN10
            self.cache[text] = v
        return v

    def unigram(self, w):
        v = self.uni.get(w)
        if v is None:
            if self.m is None:
                self._load()
            v = self.m.score(w, bos=False, eos=False) * LN10
            self.uni[w] = v
        return v


# -----------------------------------------------------------------------------
# 2) pronunciation lexicon, CTC targets, phonetic neighbours
# -----------------------------------------------------------------------------
class PronLexicon:
    def __init__(self, lexicon_path, sil_trailing=True, max_variants=2):
        self.word2prons, self.pron2words = load_lexicon(lexicon_path)
        self.word2ids = {w: [[PHONE2ID[p] for p in pr] for pr in prons] for w, prons in self.word2prons.items()}
        self.sil_trailing = sil_trailing
        self.max_variants = max_variants
        self._index = None
        self._nb_cache = {}

    def has(self, w):
        return w in self.word2ids

    def targets(self, text):
        """list of phoneme-id target variants (first pron + single alt-pron swaps), None if OOV."""
        words = text.split()
        if not words:
            return [[]]
        prons = []
        for w in words:
            p = self.word2ids.get(w)
            if p is None:
                return None
            prons.append(p)

        def assemble(choice):
            seq = []
            for k, (w_prons, c) in enumerate(zip(prons, choice)):
                seq += w_prons[c]
                if k < len(prons) - 1 or self.sil_trailing:
                    seq.append(SIL_ID)
            return seq
        base = [0] * len(prons)
        out = [assemble(base)]
        for k, w_prons in enumerate(prons):
            for c in range(1, len(w_prons)):
                if len(out) >= self.max_variants:
                    return out
                ch = list(base); ch[k] = c
                out.append(assemble(ch))
        return out

    def _build_index(self):
        idx = {}
        for pr in self.pron2words:
            keys = {pr} | {pr[:i] + pr[i + 1:] for i in range(len(pr))}
            for k in keys:
                idx.setdefault(k, []).append(pr)
        self._index = idx

    def neighbours(self, w, max_n, ngram=None):
        """words whose pronunciation is within phoneme edit distance <= 1 of any pron of w."""
        key = (w, max_n)
        if key in self._nb_cache:
            return self._nb_cache[key]
        if self._index is None:
            self._build_index()
        cands = set()
        for pr in self.word2prons.get(w, []):
            keys = {pr} | {pr[:i] + pr[i + 1:] for i in range(len(pr))}
            for k in keys:
                for q in self._index.get(k, ()):
                    if q == pr or editdistance.eval(pr, q) <= 1:
                        cands.update(self.pron2words[q])
        cands.discard(w)
        cands = list(cands)
        if ngram is not None and len(cands) > max_n:
            cands.sort(key=lambda x: -ngram.unigram(x))
        cands = cands[:max_n]
        self._nb_cache[key] = cands
        return cands


def detect_sil_convention(train_meta, lex_path, n_max=2000):
    """Does the ground-truth phoneme sequence end with ' | ' after the last word?"""
    w2p, _ = load_lexicon(lex_path)
    n = end_sil = sil_eq_words = sil_eq_words_m1 = exact_t = exact_nt = 0
    for m in train_meta[:n_max]:
        words = remove_punctuation(m['sentence']).split()
        ph = [int(p) for p in m['phonemes']]
        if not words or not ph or any(w not in w2p for w in words):
            continue
        n += 1
        end_sil += int(ph[-1] == SIL_ID)
        ns = sum(1 for p in ph if p == SIL_ID)
        sil_eq_words += int(ns == len(words)); sil_eq_words_m1 += int(ns == len(words) - 1)
        seq = []
        for k, w in enumerate(words):
            seq += [PHONE2ID[p] for p in w2p[w][0]]
            seq.append(SIL_ID)
        exact_t += int(seq == ph); exact_nt += int(seq[:-1] == ph)
    trailing = end_sil >= 0.5 * max(n, 1)
    return {'n_checked': n, 'frac_end_with_sil': end_sil / max(n, 1),
            'frac_nsil_eq_nwords': sil_eq_words / max(n, 1), 'frac_nsil_eq_nwords_minus1': sil_eq_words_m1 / max(n, 1),
            'exact_match_trailing': exact_t / max(n, 1), 'exact_match_no_trailing': exact_nt / max(n, 1),
            'sil_trailing': bool(trailing)}


# -----------------------------------------------------------------------------
# 3) exact acoustic score: log P(phones | x) via CTC forward on GPU
# -----------------------------------------------------------------------------
@torch.no_grad()
def ctc_logprob(em, targets, device, chunk=1024):
    """em: np [T,C] log-probs; targets: list of id lists -> np.float64 [K] (-1e9 if impossible)."""
    K = len(targets)
    out = np.full(K, -1e9, dtype=np.float64)
    if K == 0:
        return out
    E = torch.from_numpy(np.ascontiguousarray(em, dtype=np.float32)).to(device)
    T = E.shape[0]
    for a in range(0, K, chunk):
        tg = targets[a:a + chunk]
        k = len(tg)
        Lmax = max(1, max(len(t) for t in tg))
        tgt = torch.zeros(k, Lmax, dtype=torch.long)
        for i, t in enumerate(tg):
            if t:
                tgt[i, :len(t)] = torch.tensor(t)
        tl = torch.tensor([len(t) for t in tg], dtype=torch.long)
        lp = E[:, None, :].expand(T, k, E.shape[1]).contiguous()
        loss = F.ctc_loss(lp, tgt.to(device), torch.full((k,), T, dtype=torch.long, device=device),
                          tl.to(device), blank=BLANK_ID, reduction='none', zero_infinity=False)
        v = (-loss).double().cpu().numpy()
        v[~np.isfinite(v)] = -1e9
        out[a:a + k] = v
    return out


def acoustic_scores(texts, ems, lex, device):
    """texts -> [K, n_models] best-variant CTC log-likelihood under each emission."""
    flat, owner = [], []
    for ci, t in enumerate(texts):
        vs = lex.targets(t)
        if vs is None:
            continue
        for v in vs:
            flat.append(v); owner.append(ci)
    owner = np.asarray(owner, dtype=np.int64)
    res = np.full((len(texts), len(ems)), -1e9, dtype=np.float64)
    if not flat:
        return res
    for mi, em in enumerate(ems):
        s = ctc_logprob(em, flat, device)
        np.maximum.at(res[:, mi], owner, s)
    return res


# -----------------------------------------------------------------------------
# candidate pool container
# -----------------------------------------------------------------------------
class Pool:
    """per-utterance candidate lists with features."""
    def __init__(self):
        self.texts = []      # list[list[str]]
        self.src = []        # list[list[str]] provenance tags
        self.am = []         # list[np [K]] mean acoustic ll over the utterance's models
        self.am_min = []     # list[np [K]] min over models (disagreement-aware)
        self.ng = []         # list[np [K]]
        self.nw = []         # list[np [K]]
        self.llm = []        # list[np [K]] (nan = not scored)
        self.ntok = []

    def add_utt(self, texts, src):
        self.texts.append(list(texts)); self.src.append(list(src))
        K = len(texts)
        self.am.append(np.zeros(K)); self.am_min.append(np.zeros(K)); self.ng.append(np.zeros(K))
        self.nw.append(np.array([len(t.split()) for t in texts], dtype=np.float64))
        self.llm.append(np.full(K, np.nan)); self.ntok.append(np.zeros(K))

    def extend_utt(self, u, new_texts, tag):
        have = set(self.texts[u])
        add = [t for t in dict.fromkeys(new_texts) if t not in have]
        if not add:
            return []
        self.texts[u] += add; self.src[u] += [tag] * len(add)
        k = len(add)
        self.am[u] = np.concatenate([self.am[u], np.zeros(k)])
        self.am_min[u] = np.concatenate([self.am_min[u], np.zeros(k)])
        self.ng[u] = np.concatenate([self.ng[u], np.zeros(k)])
        self.nw[u] = np.concatenate([self.nw[u], [len(t.split()) for t in add]])
        self.llm[u] = np.concatenate([self.llm[u], np.full(k, np.nan)])
        self.ntok[u] = np.concatenate([self.ntok[u], np.zeros(k)])
        return add

    def __len__(self):
        return len(self.texts)


def fill_features(pool, utt_ems, lex, ngram, device, only_new_from=None, log=print):
    """compute am/ng for all candidates (or those at index >= only_new_from[u])."""
    t0 = time.time()
    for u in range(len(pool)):
        start = 0 if only_new_from is None else only_new_from[u]
        texts = pool.texts[u][start:]
        if not texts:
            continue
        A = acoustic_scores(texts, utt_ems[u], lex, device)
        pool.am[u][start:] = A.mean(1)
        pool.am_min[u][start:] = A.min(1)
        if ngram is not None:
            pool.ng[u][start:] = [ngram.ln(t) for t in texts]
        if (u + 1) % 500 == 0:
            log(f'  features {u + 1}/{len(pool)}  {time.time() - t0:.0f}s')


def expand_pool(pool, lex, ngram, weights, top_e=3, max_nb=12, max_new=600):
    """single-word phonetic-neighbour substitutions of each utterance's top_e candidates."""
    starts = []
    a, g = weights['ng'], weights['nw']
    for u in range(len(pool)):
        starts.append(len(pool.texts[u]))
        S = pool.am[u] + a * pool.ng[u] + g * pool.nw[u]
        order = np.argsort(-S)[:top_e]
        new = []
        for ci in order:
            words = pool.texts[u][ci].split()
            for j, w in enumerate(words):
                for nb in lex.neighbours(w, max_nb, ngram):
                    new.append(' '.join(words[:j] + [nb] + words[j + 1:]))
                if len(new) >= max_new:
                    break
        pool.extend_utt(u, new[:max_new], 'expand')
    return starts


# -----------------------------------------------------------------------------
# 4) LLM: next-word-prediction LoRA fine-tune + batched multi-GPU scoring
# -----------------------------------------------------------------------------
def _llm_ids(tok, text):
    eos = tok.eos_token_id
    return [eos] + tok(text, add_special_tokens=False)['input_ids'] + [eos]


def finetune_llm_nwp(model_name, sentences, cfg, log=print):
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    dev0 = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    use_amp = dev0 != 'cpu'
    hf_token = os.environ.get('HF_TOKEN')  # meta-llama/Llama-3.1-8B is gated -- needs an accepted-license HF token
    tok = AutoTokenizer.from_pretrained(model_name, token=hf_token)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    if not cfg.get('llm_finetune', True):
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16 if use_amp else torch.float32, token=hf_token).to(dev0).eval()
        return model, tok, {'finetuned': False}

    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

    # ---- QLoRA: 4-bit base + fp16 LoRA adapters (fits Qwen2.5-7B on a single T4) ----
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16 if use_amp else torch.float32,   # T4 (Turing) has no native bf16 tensor cores -- fp16 is the correct compute dtype here
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=bnb_cfg if use_amp else None,
        torch_dtype=torch.float32 if not use_amp else None,
        device_map={'': 0} if use_amp else None,
        token=hf_token,
    )
    if not use_amp:
        model = model.to(dev0)
    model.config.use_cache = False
    if use_amp:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
        model.enable_input_require_grads()

    lcfg = LoraConfig(r=cfg['lora_r'], lora_alpha=cfg['lora_alpha'], lora_dropout=cfg['lora_dropout'],
                      target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
                      task_type='CAUSAL_LM')
    model = get_peft_model(model, lcfg)
    rng = random.Random(0)
    sents = sorted(set(s for s in sentences if s))
    rng.shuffle(sents)
    n_dev = max(int(0.05 * len(sents)), 50)
    dev, trn = sents[:n_dev], sents[n_dev:]
    data = [_llm_ids(tok, s) for s in trn]
    dev_ids = [_llm_ids(tok, s) for s in dev]
    bs, epochs = cfg['llm_ft_batch'], cfg['llm_ft_epochs']
    accum = max(int(cfg.get('llm_ft_grad_accum', 1)), 1)     # QLoRA: gradient accumulation over small micro-batches
    steps_total = epochs * math.ceil(math.ceil(len(data) / bs) / accum)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg['llm_ft_lr'], weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / 30) * 0.5 * (1 + math.cos(math.pi * min(s / max(steps_total, 1), 1.0))))
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    amp_dtype = torch.float16 if use_amp else torch.float32   # matches bnb_4bit_compute_dtype; pairs correctly with GradScaler below

    def batchify(seqs):
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long)
        lab = torch.full((len(seqs), L), -100, dtype=torch.long)
        att = torch.zeros((len(seqs), L), dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i, :len(s)] = torch.tensor(s); lab[i, 1:len(s)] = torch.tensor(s[1:]); att[i, :len(s)] = 1
        return ids.to(dev0), lab.to(dev0), att.to(dev0)

    @torch.no_grad()
    def dev_nll():
        model.eval()
        tot, n = 0.0, 0
        dev_bs = max(bs, 8)   # keep eval batch close to the train micro-batch, not a fixed 64
        for a in range(0, len(dev_ids), dev_bs):
            ids, lab, att = batchify(dev_ids[a:a + dev_bs])
            with torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                logits = model(input_ids=ids, attention_mask=att).logits
            l = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.shape[-1]), lab[:, 1:].reshape(-1),
                                ignore_index=-100, reduction='sum')
            tot += float(l); n += int((lab[:, 1:] != -100).sum())
        model.train()
        return tot / max(n, 1)

    hist = {'dev_nll_before': dev_nll()}
    log(f'  [llm-ft] {model_name}: {len(trn)} train / {len(dev)} dev sentences, dev NLL/token before = '
        f'{hist["dev_nll_before"]:.3f} (QLoRA 4-bit, batch={bs} x accum={accum})')
    best_nll, best_state = hist['dev_nll_before'], None
    step = 0
    model.train()
    for ep in range(epochs):
        order = list(range(len(data)))
        rng.shuffle(order)
        t0 = time.time()
        opt.zero_grad(set_to_none=True)
        n_batches = math.ceil(len(order) / bs)
        for bi, a in enumerate(range(0, len(order), bs)):
            ids, lab, att = batchify([data[i] for i in order[a:a + bs]])
            with torch.autocast('cuda', dtype=amp_dtype, enabled=use_amp):
                logits = model(input_ids=ids, attention_mask=att).logits
            loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.shape[-1]), lab[:, 1:].reshape(-1),
                                   ignore_index=-100) / accum
            scaler.scale(loss).backward()
            is_last_in_epoch = (bi == n_batches - 1)
            if (bi + 1) % accum == 0 or is_last_in_epoch:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
                step += 1
        nll = dev_nll()
        hist[f'dev_nll_ep{ep + 1}'] = nll
        log(f'  [llm-ft] epoch {ep + 1}/{epochs}: dev NLL/token {nll:.3f} ({time.time() - t0:.0f}s)')
        if nll < best_nll:
            best_nll = nll
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if 'lora_' in k}
    if best_state is not None:
        model.load_state_dict(best_state, strict=False)
    del opt, sched, scaler                      # free optimizer state BEFORE the merge memory spike
    gc.collect()
    if use_amp:
        torch.cuda.empty_cache()
    model.eval()                                # NOTE: no merge_and_unload() -- merging a 4-bit base needs a
                                                 # full fp16 dequant on top of what's already resident, which
                                                 # caused a CUDA OOM on the 7B run. LoRA adapters stay active
                                                 # and apply automatically on every forward pass through this
                                                 # PeftModel, so scoring is correct without merging -- and the
                                                 # model keeps its compact 4-bit footprint.
    model.config.use_cache = True
    hist.update({'finetuned': True, 'best_dev_nll': best_nll})
    return model, tok, hist

class LLMScorer:
    def __init__(self, model, tok, n_gpus):
        self.tok = tok
        self.replicas = [(model, 'cuda:0' if torch.cuda.is_available() else 'cpu')]
        for g in range(1, n_gpus):
            self.replicas.append((copy.deepcopy(model).to(f'cuda:{g}').eval(), f'cuda:{g}'))
        self.cache = {}

    @torch.no_grad()
    def _run(self, model, dev, seqs, out, idxs, bs):
        pad = self.tok.pad_token_id
        for a in range(0, len(seqs), bs):
            chunk = seqs[a:a + bs]
            L = max(len(s) for s in chunk)
            ids = torch.full((len(chunk), L), pad, dtype=torch.long)
            att = torch.zeros((len(chunk), L), dtype=torch.long)
            for i, s in enumerate(chunk):
                ids[i, :len(s)] = torch.tensor(s); att[i, :len(s)] = 1
            ids, att = ids.to(dev), att.to(dev)
            logits = model(input_ids=ids, attention_mask=att).logits[:, :-1]
            lp = torch.log_softmax(logits.float(), -1).gather(-1, ids[:, 1:, None]).squeeze(-1)
            lp = (lp * att[:, 1:].float()).sum(1)
            for i, v in enumerate(lp.tolist()):
                out[idxs[a + i]] = v

    def score(self, texts, bs=32):
        """-> (sum ln P incl. final EOS, n scored tokens) for each text."""
        uniq = [t for t in dict.fromkeys(texts) if t not in self.cache]
        if uniq:
            seqs = [_llm_ids(self.tok, t) for t in uniq]
            order = sorted(range(len(uniq)), key=lambda i: len(seqs[i]))
            res = {}
            parts = [order[g::len(self.replicas)] for g in range(len(self.replicas))]
            threads = []
            for (model, dev), part in zip(self.replicas, parts):
                th = threading.Thread(target=self._run, args=(model, dev, [seqs[i] for i in part], res, part, bs))
                th.start(); threads.append(th)
            for th in threads:
                th.join()
            for i, t in enumerate(uniq):
                self.cache[t] = (res[i], len(seqs[i]) - 1)
        vals = [self.cache[t] for t in texts]
        return np.array([v[0] for v in vals]), np.array([v[1] for v in vals], dtype=np.float64)


# -----------------------------------------------------------------------------
# 5) vectorised tuning by exact corpus WER
# -----------------------------------------------------------------------------
def pad_matrix(lists, fill, K=None, sel=None):
    sel = range(len(lists)) if sel is None else sel
    sel = list(sel)
    K = K or max(len(lists[u]) for u in sel)
    M = np.full((len(sel), K), fill, dtype=np.float64)
    for r, u in enumerate(sel):
        v = np.asarray(lists[u], dtype=np.float64)
        M[r, :len(v)] = v
    return M


def errors_lists(pool, refs):
    errs, nref = [], []
    for u in range(len(pool)):
        rw = remove_punctuation(refs[u] or '').split()
        errs.append(np.array([editdistance.eval(rw, remove_punctuation(t).split()) for t in pool.texts[u]], dtype=np.float64))
        nref.append(len(rw))
    return errs, np.asarray(nref, dtype=np.float64)


class Tuner:
    """Feature matrices for a subset of utterances; score = am + a*ng + b*llm + g*nw."""
    def __init__(self, pool, errs, nref, sel, topk_mask=None):
        self.sel = list(sel)
        K = max(len(pool.texts[u]) for u in self.sel)
        self.AM = pad_matrix(pool.am, -1e18, K, self.sel)
        self.NG = pad_matrix(pool.ng, 0.0, K, self.sel)
        self.NW = pad_matrix(pool.nw, 0.0, K, self.sel)
        L = pad_matrix(pool.llm, np.nan, K, self.sel)
        self.LLM = np.nan_to_num(L, nan=0.0)
        self.E = pad_matrix(errs, 1e6, K, self.sel)
        self.N = nref[self.sel].sum()
        self.valid = self.AM > -1e17
        if topk_mask is not None:
            self.valid &= pad_matrix(topk_mask, 0.0, K, self.sel) > 0
        self.AM = np.where(self.valid, self.AM, -1e18)

    def errors(self, a, b, g, rows=None):
        S = self.AM + a * self.NG + b * self.LLM + g * self.NW
        E = self.E
        if rows is not None:
            S, E = S[rows], E[rows]
        pick = S.argmax(1)
        return E[np.arange(len(pick)), pick].sum(), pick

    def grid(self, A, Bs, G, rows=None):
        AM, NG, LL, NW, E = self.AM, self.NG, self.LLM, self.NW, self.E
        if rows is not None:
            AM, NG, LL, NW, E = AM[rows], NG[rows], LL[rows], NW[rows], E[rows]
        err = np.full((len(A), len(Bs), len(G)), np.inf)
        if AM.shape[0] == 0:
            return np.zeros((len(A), len(Bs), len(G)))
        ar = np.arange(AM.shape[0])
        for i, a in enumerate(A):
            base_a = AM + a * NG
            for j, b in enumerate(Bs):
                base_ab = base_a + b * LL if b != 0 else base_a
                for k, g in enumerate(G):
                    pick = (base_ab + g * NW).argmax(1)
                    err[i, j, k] = E[ar, pick].sum()
        return err


def subpool(pool, masks, errs=None):
    """keep only candidates with mask>0 (per utterance); returns (Pool, errs_subset)."""
    sp, se = Pool(), []
    for u in range(len(pool)):
        keep = np.where(np.asarray(masks[u]) > 0)[0]
        if len(keep) == 0:
            keep = np.array([0])
        sp.texts.append([pool.texts[u][i] for i in keep]); sp.src.append([pool.src[u][i] for i in keep])
        for name in ('am', 'am_min', 'ng', 'nw', 'llm', 'ntok'):
            getattr(sp, name).append(getattr(pool, name)[u][keep].copy())
        if errs is not None:
            se.append(errs[u][keep].copy())
    return sp, (se if errs is not None else None)


def utt_scores(pool, u, W):
    llm = np.nan_to_num(pool.llm[u], nan=0.0)
    return pool.am[u] + W['ng'] * pool.ng[u] + W.get('llm', 0.0) * llm + W['nw'] * pool.nw[u]


def topk_masks(pool, W, k):
    masks = []
    for u in range(len(pool)):
        S = utt_scores(pool, u, W)
        m = np.zeros(len(S))
        m[np.argsort(-S)[:k]] = 1
        for i, s in enumerate(pool.src[u]):
            if s.endswith('#0'):
                m[i] = 1                      # every generator's 1-best always survives
        masks.append(m)
    return masks


def fluency(pool, W):
    """per-utterance LLM log-prob per token of the utterance's best candidate under W (no LLM term)."""
    W0 = dict(W, llm=0.0)
    f = np.zeros(len(pool))
    for u in range(len(pool)):
        i = int(np.argmax(utt_scores(pool, u, W0)))
        f[u] = pool.llm[u][i] / max(pool.ntok[u][i], 1) if np.isfinite(pool.llm[u][i]) else 0.0
    return f


def predict_texts(pool, W=None, gate=None, flu=None, sel=None):
    """W: single weight dict; or gate={'tau','hi','lo'} with per-utterance fluency flu."""
    sel = range(len(pool)) if sel is None else sel
    out, regimes = {}, {}
    for u in sel:
        if gate is not None:
            use_lo = gate.get('lo') is not None and flu[u] < gate['tau']
            Wu = gate['lo'] if use_lo else gate['hi']
            regimes[u] = 'lo' if use_lo else 'hi'
        else:
            Wu = W
        out[u] = pool.texts[u][int(np.argmax(utt_scores(pool, u, Wu)))]
    return out, regimes


def pick_from_grid(err, A, Bs, G, smooth=True, tol_rel=0.0):
    """Pick (ng, llm, nw) weights from the 3-D tune-fold error grid.

    Plain argmin picks whichever grid cell has the single lowest word-error count on the
    TUNE fold -- with a grid this dense that's often a point that's only marginally ahead
    of many neighbours by tune-fold noise, not by a real generalizable margin. That cell
    can carry a much larger LLM weight than cells that tie with it, and the larger weight
    is exactly the part that fails to transfer to the verify fold / the Kaggle test set.

    Fix (1-SE-style tolerance rule): among all cells within `tol_rel` (relative, on word-
    error COUNT) of the smoothed grid's minimum, pick the one with the SMALLEST |llm|
    weight. This only pulls the LLM weight down when the larger weight wasn't actually
    earning a meaningfully better tune-fold score in the first place.
    """
    from scipy.ndimage import uniform_filter
    A, Bs, G = np.asarray(A, dtype=float), np.asarray(Bs, dtype=float), np.asarray(G, dtype=float)  # stage-1 passes plain
                                                                                                       # Python lists (e.g. [0.0])
                                                                                                       # for the unused llm axis --
                                                                                                       # fancy-indexing those below
                                                                                                       # needs real ndarrays.
    raw = np.unravel_index(np.argmin(err), err.shape)
    base = err
    if smooth and err.size > 1:
        base = uniform_filter(err, size=3, mode='nearest') + 1e-6 * err
    best = float(base[np.unravel_index(np.argmin(base), base.shape)])
    if tol_rel > 0:
        tol = tol_rel * best                            # opt-in only: tol_rel=0 (default) reproduces plain argmin exactly
        ii, jj, kk = np.where(base <= best + tol)
        order = np.argsort(np.abs(Bs[jj]))              # among near-ties, prefer the smallest LLM weight
        idx = (ii[order[0]], jj[order[0]], kk[order[0]])
    else:
        idx = np.unravel_index(np.argmin(base), base.shape)   # exact argmin -- no tie-break bias applied
    return ({'ng': float(A[idx[0]]), 'llm': float(Bs[idx[1]]), 'nw': float(G[idx[2]]), 'err': float(err[idx])},
            {'ng': float(A[raw[0]]), 'llm': float(Bs[raw[1]]), 'nw': float(G[raw[2]]), 'err': float(err[raw])})

Writing /kaggle/working/b2t_code/b2t_decode.py


# BLOCK 2 · DATA: memmap cache, normalisation, cross-fit split, dataset statistics
Runs in both modes. The cache lives in `/tmp` (not in the output) and is shared by both worker processes through the page cache.

In [8]:
# ============================================================================
# BLOCK 2 - DATA
# ============================================================================
sys.path.insert(0, CODE_DIR)
import b2t_core, b2t_decode
importlib.reload(b2t_core); importlib.reload(b2t_decode)
from b2t_core import *
from b2t_decode import *

DATA_DIR = resolve_competition_path(DATA_DIR_LOCAL, 'brain-to-text-25')
session2idx = get_session2idx(DATA_DIR)
idx2session = {v: k for k, v in session2idx.items()}
n_days = len(session2idx)
print(f'data: {DATA_DIR} | {n_days} sessions')

CACHE_DIR = pick_cache_dir(CACHE_CANDIDATES)
RESUME_DIR = os.path.join(os.path.dirname(CACHE_DIR), 'b2t_resume')
t0 = time.time()
build_cache(DATA_DIR, CACHE_DIR)
readers = {s: CacheReader(CACHE_DIR, s) for s in ('train', 'val', 'test')}
print(f'cache ready in {time.time() - t0:.0f}s:', {s: len(r) for s, r in readers.items()})

norm_path = os.path.join(CACHE_DIR, 'norm_stats.pt')
INIT_DIR_FOUND = None
if MODE == 'train' and INIT_CKPT_DIR:
    if not os.path.isdir(INIT_CKPT_DIR):
        raise FileNotFoundError(f'INIT_CKPT_DIR not found: {INIT_CKPT_DIR} (add the dataset as an input, or set INIT_CKPT_DIR=None)')
    _hits = sorted(glob(os.path.join(INIT_CKPT_DIR, '**', 'fold*_seed*_*.pt'), recursive=True))
    if not _hits:
        raise FileNotFoundError(f'no fold*_seed*_*.pt under {INIT_CKPT_DIR}')
    INIT_DIR_FOUND = os.path.dirname(_hits[0])
    print('continue-training source:', INIT_DIR_FOUND)
    _init_norm = os.path.join(INIT_DIR_FOUND, 'norm_stats.pt')
    if os.path.exists(_init_norm):
        shutil.copy(_init_norm, norm_path)
        print('normalisation: norm_stats.pt copied from the init run (identical preprocessing)')
if MODE == 'train' and not os.path.exists(norm_path):
    torch.save(compute_norm_stats(readers['train']), norm_path)
NORM = torch.load(norm_path, weights_only=False) if os.path.exists(norm_path) else None


def auto_find_ckpt_dir():
    cands = []
    if TRAINED_MODEL_DATASET:
        try:
            cands.append(resolve_kaggle_dataset(TRAINED_MODEL_DATASET))
        except Exception as e:
            print('dataset slug failed:', e)
    for c in (TRAINED_MODEL_DIR, FALLBACK_CKPT_DATASET):
        if c and os.path.isdir(str(c)):
            cands.append(c)
    for root in ('/kaggle/input', CKPT_OUT_DIR):
        if os.path.isdir(root):
            cands.append(root)
    for c in cands:
        hits = sorted(glob(os.path.join(c, '**', 'fold*_seed*_*.pt'), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    return None

if LEXICON_PATH is None:
    _lex_ds = resolve_kaggle_dataset(LEXICON_DATASET)
    LEXICON_PATH, TOKENS_PATH = find_file(_lex_ds, 'lexicon.txt'), find_file(_lex_ds, 'tokens.txt')
if KENLM_PATH is None:
    KENLM_PATH = find_file(resolve_kaggle_dataset(KENLM_DATASET), '*.bin')
print('lexicon:', LEXICON_PATH, '\nkenlm  :', KENLM_PATH)
with open(TOKENS_PATH) as f:
    _toks = [l.rstrip('\n').strip() for l in f]
assert [t.strip() for t in _toks[:N_CLASSES]] == [p.strip() for p in PHONEME_VOCAB], 'tokens.txt != PHONEME_VOCAB'

# ---- cross-fit split (llm mode re-uses the split saved next to the checkpoints) ----
if MODE == 'llm':
    try:
        _ck = auto_find_ckpt_dir()
        SPLIT = json.load(open(find_file(_ck, 'split.json')))
        print('split: loaded from checkpoint dataset')
    except Exception as e:
        print('split.json not found next to checkpoints -> regenerating deterministically:', repr(e))
        SPLIT = {'labels': make_crossfit_split(readers['val'].meta, CROSSFIT_PATTERN, SEED), 'pattern': CROSSFIT_PATTERN}
else:
    SPLIT = {'labels': make_crossfit_split(readers['val'].meta, CROSSFIT_PATTERN, SEED), 'pattern': CROSSFIT_PATTERN,
             'seed': SEED}
    if INIT_DIR_FOUND and os.path.exists(os.path.join(INIT_DIR_FOUND, 'split.json')):
        _init_split = json.load(open(os.path.join(INIT_DIR_FOUND, 'split.json')))
        _same = _init_split.get('labels') == SPLIT['labels']
        print(f"split: using split.json of the init run (identical to regenerated split: {_same})")
        SPLIT = _init_split
SPLIT_PATH = os.path.join(WORK, 'split.json')
json.dump(SPLIT, open(SPLIT_PATH, 'w'))
VAL_LABELS = [SPLIT['labels'].get(m['key'], 'C') for m in readers['val'].meta]
print('val split counts:', pd.Series(VAL_LABELS).value_counts().to_dict())

# ---- dataset statistics (tables/) ----
try:
    rows = []
    for s, r in readers.items():
        for m in r.meta:
            rows.append({'split': s, 'session': m['session'], 'block': m['block'], 'T': m['n'],
                         'n_words': len(remove_punctuation(m['sentence']).split()), 'n_phon': len(m['phonemes'])})
    stats_df = pd.DataFrame(rows)
    per_day = stats_df.pivot_table(index='session', columns='split', values='T', aggfunc='count', fill_value=0).reset_index()
    save_table(per_day, 'dataset_trials_per_day.csv')
    summ = stats_df.groupby('split').agg(trials=('T', 'size'), sessions=('session', 'nunique'), mean_T=('T', 'mean'),
                                         median_T=('T', 'median'), max_T=('T', 'max'), mean_words=('n_words', 'mean')).reset_index()
    print(summ.to_string(index=False)); save_table(summ, 'dataset_summary.csv')

    train_sent = set(remove_punctuation(m['sentence']) for m in readers['train'].meta)
    val_sent = [remove_punctuation(m['sentence']) for m in readers['val'].meta]
    print(f'val sentences also present verbatim in train: {np.mean([s in train_sent for s in val_sent]) * 100:.1f}%')
except Exception as e:
    print('dataset statistics skipped:', repr(e))

try:
    SIL_INFO = detect_sil_convention(readers['train'].meta, LEXICON_PATH)
except Exception as e:
    print('SIL convention check failed, assuming trailing silence:', repr(e))
    SIL_INFO = {'sil_trailing': True}
print('phoneme target convention:', SIL_INFO)
json.dump(SIL_INFO, open(os.path.join(WORK, 'sil_convention.json'), 'w'))
print(f'session clock: {(time.time() - NOTEBOOK_T0) / 60:.1f} min')

data: /kaggle/input/competitions/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final | 45 sessions
[cache] using /tmp/b2t_cache (mount /, 1102.1 GB free)
[cache] train: 8072 trials, 7061713 frames, 7.23 GB, 247s
[cache] val: 1426 trials, 1314955 frames, 1.35 GB, 48s
[cache] test: 1450 trials, 1329030 frames, 1.36 GB, 46s
cache ready in 341s: {'train': 8072, 'val': 1426, 'test': 1450}
lexicon: /kaggle/input/datasets/heyyousum/quality-english-dataset-for-ngram-model-v2/lexicon.txt 
kenlm  : /kaggle/input/datasets/heyyousum/custom-4-gram-wiki-news-switchboard-updated-v3/custom_4gram.bin
split: loaded from checkpoint dataset
val split counts: {'A': 572, 'B': 570, 'C': 284}
  table  -> /kaggle/working/tables/dataset_trials_per_day.csv
split  trials  sessions     mean_T  median_T  max_T  mean_words
 test    1450        41 916.572414     894.0   2140    0.000000
train    8072        45 874.840560     836.0   2475    6.274158
  val    1426        41 922.128331     890.5   2382    6.623422

# BLOCK 3 · TRAIN (`MODE='train'`)
Two independent worker processes (`CUDA_VISIBLE_DEVICES=0` / `1`). The LR cosine is a function of *elapsed time*: it lands on `lr_min` at `deadline − WORKER_FINAL_RESERVE_MIN`, whatever the T4 speed, so the run never stops early and never overruns.  
Every 30 min (15 min in the last 40 %): EMA weights evaluated on the fold holdout with official PER and greedy WER; `best_wer.pt`, `best_per.pt` overwritten on improvement; `ema_final.pt` at the end. Every 20 min a resume state is written; if a worker dies for any reason the monitor relaunches it with `--resume`.

In [9]:
%%run_if train
# ============================================================================
# BLOCK 3.1 - LAUNCH ONE WORKER PER GPU, MONITOR, AUTO-RESTART ON CRASH
# ============================================================================
DEADLINE_TS = NOTEBOOK_T0 + SESSION_HOURS * 3600 - NOTEBOOK_RESERVE_MIN * 60
TRAIN_END_TS = DEADLINE_TS - WORKER_FINAL_RESERVE_MIN * 60
local_test = bool(os.environ.get('B2T_LOCAL_TEST'))
runs = TRAIN_RUNS if (N_GPUS >= len(TRAIN_RUNS) or local_test) else TRAIN_RUNS[:max(N_GPUS, 1)]
if not local_test and N_GPUS < len(TRAIN_RUNS):
    print(f'WARNING: {N_GPUS} GPU(s) visible -> running only fold(s) {[r["fold"] for r in runs]}.')
os.makedirs(RESUME_DIR, exist_ok=True)
shutil.copy(norm_path, os.path.join(CKPT_OUT_DIR, 'norm_stats.pt'))
shutil.copy(SPLIT_PATH, os.path.join(CKPT_OUT_DIR, 'split.json'))
shutil.copy(os.path.join(WORK, 'sil_convention.json'), os.path.join(CKPT_OUT_DIR, 'sil_convention.json'))


def ram_line():
    try:
        import psutil
        vm = psutil.virtual_memory()
        return f'host RAM used {vm.percent:.0f}% ({vm.available / 1e9:.1f} GB available)'
    except Exception:
        return ''


def launch(entry, resume):
    run = entry['run']
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(run['gpu']), PYTHONUNBUFFERED='1', OMP_NUM_THREADS='2',
               PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True', TORCH_CUDNN_V8_API_LRU_CACHE_LIMIT='64')
    if resume:
        env.pop('B2T_FAKE_CRASH_STEP', None)
    cmd = [sys.executable, '-u', os.path.join(CODE_DIR, 'train_worker.py'), '--config', entry['cfg_path']]
    if resume:
        cmd.append('--resume')
    logf = open(entry['log_path'], 'a')
    entry['proc'] = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT, env=env, cwd=WORK)
    entry['logf'] = logf
    print(f"{'RE' if resume else ''}launched fold {run['fold']} seed {run['seed']} on GPU{run['gpu']} "
          f"(pid {entry['proc'].pid}{', --resume' if resume else ''}) -> {entry['log_path']}")


def _tail(path, n=400):
    try:
        with open(path, errors='replace') as f:
            return f.readlines()[-n:]
    except Exception:
        return []


entries = []
for run in runs:
    cfg = dict(TRAIN_CFG)
    cfg.update({'fold': run['fold'], 'seed': run['seed'], 'train_val_labels': run['train_val_labels'],
                'out_dir': CKPT_OUT_DIR, 'resume_dir': RESUME_DIR, 'cache_dir': CACHE_DIR, 'split_path': SPLIT_PATH,
                'session2idx': session2idx, 'deadline_ts': DEADLINE_TS,
                'final_reserve_s': WORKER_FINAL_RESERVE_MIN * 60, 'model': MODEL_CFG, 'aug': AUG_CFG,
                'lexicon_path': LEXICON_PATH, 'rss_limit_gb': WORKER_RSS_LIMIT_GB,
                'min_avail_gb': WORKER_MIN_AVAIL_GB})
    tag = f"fold{run['fold']}_seed{run['seed']}"
    # v6.2: warm start from the same fold of the previous run (never cross folds: that would leak holdout data)
    cfg['init_from'] = TRAIN_CFG.get('init_from')
    if INIT_DIR_FOUND:
        _init = None
        for _name in INIT_PREFERENCE:
            _c = sorted(glob(os.path.join(INIT_CKPT_DIR, '**', f'{tag}_{_name}'), recursive=True))
            if _c:
                _init = _c[0]; break
        if _init is None:
            raise FileNotFoundError(f'no {tag}_{{{"|".join(INIT_PREFERENCE)}}} under {INIT_CKPT_DIR}')
        _ck = torch.load(_init, map_location='cpu', weights_only=False)
        if sorted(_ck.get('train_val_labels', [])) != sorted(run['train_val_labels']):
            raise ValueError(f"{_init} was trained on val labels {_ck.get('train_val_labels')}, run expects {run['train_val_labels']}")
        _m = _ck.get('metrics', {}) or {}
        print(f"{tag}: init <- {_init} (step {_ck.get('step')}, epoch {_ck.get('epoch')}, "
              f"PER {_m.get('per', float('nan')) * 100:.2f}%, greedyWER {_m.get('greedy_wer', float('nan')) * 100:.2f}%)")
        cfg['init_from'] = _init
        del _ck
    cfg_path = os.path.join(CKPT_OUT_DIR, f'run_{tag}.json')
    json.dump(cfg, open(cfg_path, 'w'), indent=1)
    stale = os.path.join(RESUME_DIR, f'{tag}_resume.pt')
    if os.path.exists(stale):
        os.remove(stale)                     # never resume from an older session's state
    log_path = os.path.join(LOG_DIR, f'train_{tag}.log')
    open(log_path, 'w').close()
    e = {'run': run, 'tag': tag, 'cfg_path': cfg_path, 'log_path': log_path, 'restarts': 0, 'recycles': 0, 'done': False}
    launch(e, resume=False)
    entries.append(e)
print(f'train end in {human_time(TRAIN_END_TS - time.time())} | {(time.time() - NOTEBOOK_T0) / 60:.1f} min used for setup | {ram_line()}')

last_print = 0.0
while True:
    alive = False
    for e in entries:
        if e['done']:
            continue
        rc = e['proc'].poll()
        if rc is None:
            alive = True
            continue
        e['logf'].close()
        if rc == 0:
            e['done'] = True
            print(f"fold {e['run']['fold']} finished normally")
            continue
        if rc == 75 and DEADLINE_TS - time.time() > 60:         # v6.2 memory guard: planned, state already saved
            e['recycles'] += 1
            print(f"[memory guard] fold {e['run']['fold']} saved its state and exited (recycle #{e['recycles']}, "
                  f"{ram_line()}) -> relaunching with --resume")
            time.sleep(3)
            launch(e, resume=True)
            alive = True
            continue
        print(f"\n!!! fold {e['run']['fold']} exited with code {rc} ({'killed by the OS, usually host-RAM OOM' if rc in (-9, 137) else 'exception, see log'}) | {ram_line()}")
        for l in _tail(e['log_path'], 25):
            print('   ' + l.rstrip()[:300])
        if e['restarts'] < MAX_WORKER_RESTARTS and TRAIN_END_TS - time.time() > RESTART_MIN_REMAINING_MIN * 60:
            e['restarts'] += 1
            print(f"-> relaunching from the last resume state (restart {e['restarts']}/{MAX_WORKER_RESTARTS})")
            time.sleep(3)
            launch(e, resume=True)
            alive = True
        else:
            e['done'] = True
            print(f"-> not relaunching fold {e['run']['fold']} (restarts used or too close to the deadline); its saved checkpoints are kept")
    if not alive:
        break
    if time.time() - last_print >= MONITOR_PRINT_MIN * 60:
        last_print = time.time()
        print(f"\n===== monitor @ session {human_time(time.time() - NOTEBOOK_T0)} | train end in "
              f"{human_time(TRAIN_END_TS - time.time())} | {ram_line()} =====")
        for e in entries:
            lines = _tail(e['log_path'])
            key = [l for l in lines if 'EVAL' in l or 'BUDGET' in l or 'OOM' in l or 'RESUMED' in l or 'MEMORY GUARD' in l][-3:]
            status = 'done' if e['done'] else 'running'
            print(f"-- fold {e['run']['fold']} [{status}, crash restarts {e['restarts']}, memory recycles {e['recycles']}]")
            for l in key + lines[-1:]:
                print('   ' + l.rstrip()[:300])
        try:
            print(subprocess.run(['nvidia-smi', '--query-gpu=index,utilization.gpu,memory.used,memory.total',
                                  '--format=csv,noheader'], capture_output=True, text=True, timeout=30).stdout.strip())
        except Exception:
            pass
    if time.time() > DEADLINE_TS + 20 * 60:
        for e in entries:
            if not e['done'] and e['proc'].poll() is None:
                e['proc'].kill()
        print('killed workers still running 20 min past the deadline')
    time.sleep(15)

for e in entries:
    print(f"\nfold {e['run']['fold']} (restarts {e['restarts']}) last lines:")
    for l in _tail(e['log_path'], 5):
        print('   ' + l.rstrip()[:300])
CKPTS_WRITTEN = sorted(glob(os.path.join(CKPT_OUT_DIR, 'fold*_seed*_*.pt')))
print(f'\ncheckpoints written: {len(CKPTS_WRITTEN)}')
if not CKPTS_WRITTEN:
    raise RuntimeError('no checkpoint was produced by any worker - see the log excerpts above')

[skip] this cell runs only for MODE in ['train'] (MODE='llm')


In [10]:
%%run_if train --soft
# ============================================================================
# BLOCK 3.2 - TRAINING REPORT (curves, summary table, bundle manifest)
# ============================================================================
curves = {}
for f in sorted(glob(os.path.join(CKPT_OUT_DIR, '*_metrics.jsonl'))):
    tag = os.path.basename(f).replace('_metrics.jsonl', '')
    _rows = [json.loads(l) for l in open(f) if l.strip()]
    if not _rows:
        continue
    curves[tag] = pd.DataFrame(_rows)
    curves[tag]['run'] = tag
if curves:
    allc = pd.concat(curves.values(), ignore_index=True)
    save_table(allc, 'training_eval_history.csv')
    with figure_guard('training_curves'):
        panels = [('per', 'PER (greedy, %)'), ('greedy_wer', 'WER greedy lexicon (%)'), ('train_ctc', 'train CTC loss')]
        fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
        for ax, (col, title) in zip(axes, panels):
            for tag, df in curves.items():
                if col in df:
                    y = df[col] * (100 if col != 'train_ctc' else 1)
                    ax.plot(df['hours'], y, marker='o', ms=3, lw=1.5, label=tag)
            ax.set_title(title); ax.set_xlabel('hours'); ax.grid(alpha=0.3)
        axes[0].legend(frameon=False, fontsize=8)
        fig.suptitle('B2TNetV4 cross-fit training (EMA weights, fold holdout)', y=1.03)
        save_fig(fig, 'training_curves')
    summ_rows = []
    for tag, df in curves.items():
        sel = df['greedy_wer']
        i = int(sel.idxmin())
        summ_rows.append({'run': tag, 'evals': len(df), 'last_epoch': int(df['epoch'].iloc[-1]),
                          'last_step': int(df['step'].iloc[-1]), 'hours': round(float(df['hours'].iloc[-1]), 2),
                          'best_greedy_wer_%': round(float(sel.min()) * 100, 2), 'at_hours': round(float(df['hours'][i]), 2),
                          'best_per_%': round(float(df['per'].min()) * 100, 2),
                          'final_per_%': round(float(df['per'].iloc[-1]) * 100, 2),
                          'final_greedy_wer_%': round(float(df['greedy_wer'].iloc[-1]) * 100, 2)})
    summ_df = pd.DataFrame(summ_rows); print(summ_df.to_string(index=False)); save_table(summ_df, 'training_summary.csv')

manifest = sorted(os.path.relpath(p, CKPT_OUT_DIR) for p in glob(os.path.join(CKPT_OUT_DIR, '*')))
json.dump({'files': manifest, 'model_cfg': MODEL_CFG, 'train_cfg': TRAIN_CFG, 'aug_cfg': AUG_CFG,
           'runs': TRAIN_RUNS, 'session_hours': SESSION_HOURS}, open(os.path.join(CKPT_OUT_DIR, 'manifest.json'), 'w'), indent=1)
print('\ncheckpoint bundle:', CKPT_OUT_DIR)
for m in manifest:
    print('  ', m)
print('\nNEXT: Save Version -> New Dataset from this output (folder b2t_v6_ckpt) -> set TRAINED_MODEL_DIR -> MODE="llm".')
if CACHE_DIR.startswith(WORK):
    shutil.rmtree(CACHE_DIR, ignore_errors=True); shutil.rmtree(RESUME_DIR, ignore_errors=True)
    print('removed the data cache from /kaggle/working so it is not saved as output')
print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')

[skip] this cell runs only for MODE in ['train'] (MODE='llm')


# BLOCK 4 · LLM / DECODING (`MODE='llm'`)
Tuning protocol (all numbers are **out-of-fold**): every val trial is decoded only by acoustic models that never trained on it (A → model B, B → model A, C → both = exactly the test situation). Weights are chosen on **A∪B**, verified once on **C**, then (optionally) refit on A∪B∪C for the submission. Checkpoint screening uses each fold's own holdout.

In [11]:
%%run_if llm
# ============================================================================
# BLOCK 4.1 - CHECKPOINTS: locate, day override, REAL-WER screening, emissions
# ============================================================================
import traceback


CKPT_DIR = auto_find_ckpt_dir()
if CKPT_DIR is None:
    raise FileNotFoundError('No fold*_seed*_*.pt found (TRAINED_MODEL_DIR, /kaggle/input/**). '
                            'Run MODE="train", Save Version, and add its output as an input dataset.')
print('checkpoints:', CKPT_DIR)
NORM = torch.load(find_file(CKPT_DIR, 'norm_stats.pt'), weights_only=False)
try:
    SIL_INFO = json.load(open(find_file(CKPT_DIR, 'sil_convention.json')))
except Exception:
    pass
CLIP = TRAIN_CFG['clip']

# ---- day-index override for never-trained days (same semantics as v2) ----
GENERIC_DAY_IDX = n_days
DAY_OVERRIDE = torch.arange(n_days, dtype=torch.long)
untrained_days_set = None
for f in glob(os.path.join(CKPT_DIR, '**', 'trained_days.json'), recursive=True):
    s = set(int(d) for d in json.load(open(f)).get('untrained_days', []))
    untrained_days_set = s if untrained_days_set is None else (untrained_days_set & s)
untrained_days_set = untrained_days_set or set()
for d in untrained_days_set:
    DAY_OVERRIDE[d] = GENERIC_DAY_IDX
print('days routed through the generic slot:', [idx2session[d] for d in sorted(untrained_days_set)] or 'none')

groups = {}
for f in sorted(glob(os.path.join(CKPT_DIR, '**', 'fold*_seed*_*.pt'), recursive=True)):
    m = re.match(r'(fold[^_]+_seed\d+)_(.+\.pt)$', os.path.basename(f))
    if m and m.group(2) in CKPT_PREFERENCE:
        groups.setdefault(m.group(1), {})[m.group(2)] = f
print('checkpoint groups:', {k: sorted(v) for k, v in groups.items()})


def load_am(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    mdl = B2TNetV4(ck['n_days'], ck['config'])
    mdl.load_state_dict(ck['model_state_dict'])
    return mdl.to(DEVICE).eval(), ck


def extract_emissions(mdl, split, idxs):
    ds = TrialSet(readers, [(split, i) for i in idxs], session2idx)
    out = [None] * len(idxs)
    for bidx in BucketBatchSampler(ds.lengths, 24, 24 * 2500, False, 0):
        b = collate_trials([ds[i] for i in bidx])
        lp, ol = forward_batch(mdl, b, DEVICE, NORM, CLIP, day_override=DAY_OVERRIDE)
        lp = lp.cpu()
        for k, j in enumerate(b['j'].tolist()):
            out[j] = lp[:int(ol[k]), k].numpy().astype(np.float16)
    return out


n_val, n_test = len(readers['val']), len(readers['test'])
val_refs = [m['sentence'] for m in readers['val'].meta]
screen_spec = dict(lexicon=LEXICON_PATH, tokens=TOKENS_PATH, lm=KENLM_PATH, beam=LLM_CFG['screen_beam'], nbest=1,
                   lm_weights=[LLM_CFG['screen_lm_weight']])
screen_tasks, screen_meta = [], {}
t0 = time.time()
for tag, cands in groups.items():
    ck_any = torch.load(next(iter(cands.values())), map_location='cpu', weights_only=False)
    seen = set(ck_any.get('train_val_labels', []))
    hold = [j for j in range(n_val) if VAL_LABELS[j] not in seen]
    sub = sorted(random.Random(SEED).sample(hold, min(LLM_CFG['screen_n'], len(hold)))) if hold else []
    screen_meta[tag] = {'seen': seen, 'sub': sub}
    if len(cands) == 1 or not sub:
        continue
    for name, path in cands.items():
        try:
            mdl, _ = load_am(path)
            ems = extract_emissions(mdl, 'val', sub)
            screen_tasks += [((tag, name, k), ems[k]) for k in range(len(sub))]
            del mdl
        except Exception as e:
            print(f'  {tag}/{name}: unreadable checkpoint skipped ({e!r})')
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()
try:
    screen_out = run_beam_pool(screen_tasks, screen_spec, LLM_CFG['n_proc'], desc='screen') if screen_tasks else {}
except Exception as e:
    print('screening decode failed -> CKPT_PREFERENCE order used:', repr(e)); screen_out = {}

MODELS, screen_rows = [], []
lw0 = LLM_CFG['screen_lm_weight']
for tag, cands in groups.items():
    sub = screen_meta[tag]['sub']
    scores = {}
    for name in cands:
        if screen_out and (tag, name, 0) in screen_out:
            hyps = [(screen_out[(tag, name, k)][lw0] or [''])[0] for k in range(len(sub))]
            scores[name] = official_wer([val_refs[j] for j in sub], hyps)[0]
            screen_rows.append({'group': tag, 'checkpoint': name, 'screen_wer_%': round(scores[name] * 100, 2),
                                'n_trials': len(sub)})
    order = (sorted(scores, key=lambda n: (scores[n], CKPT_PREFERENCE.index(n))) if scores else [])
    order += [n for n in sorted(cands, key=CKPT_PREFERENCE.index) if n not in order]
    mdl = None
    for best_name in order:
        try:
            mdl, ck = load_am(cands[best_name]); break
        except Exception as e:
            print(f'  {tag}/{best_name}: load failed ({e!r})')
    if mdl is None:
        continue
    MODELS.append({'tag': tag, 'ckpt': cands[best_name], 'labels_seen': set(ck.get('train_val_labels', [])),
                   'model': mdl, 'screen_wer': scores.get(best_name), 'arch_metrics': ck.get('metrics', {})})
    print(f"{tag}: using {best_name} (screen WER {scores.get(best_name, float('nan')) * 100:.2f}%), "
          f"trained on val labels {sorted(MODELS[-1]['labels_seen'])}")
if screen_rows:
    save_table(pd.DataFrame(screen_rows), 'checkpoint_screening.csv')
print(f'screening took {time.time() - t0:.0f}s')

# ---- full emissions for the chosen checkpoints ----
if not MODELS:
    raise RuntimeError('no loadable acoustic checkpoint')
VAL_EMS, TEST_EMS = {}, {}
for M in MODELS:
    VAL_EMS[M['tag']] = extract_emissions(M['model'], 'val', list(range(n_val)))
    TEST_EMS[M['tag']] = extract_emissions(M['model'], 'test', list(range(n_test)))
OOF = [[mi for mi, M in enumerate(MODELS) if VAL_LABELS[j] not in M['labels_seen']] for j in range(n_val)]
n_no_oof = sum(1 for o in OOF if not o)
if n_no_oof:
    print(f'WARNING: {n_no_oof} val trials were trained on by every model (final-fit run?) -> excluded from tuning.')

# greedy PER / greedy-lexicon WER of the first out-of-fold model (reference rows)
_, PRON2WORDS = load_lexicon(LEXICON_PATH)
GREEDY_IDS, GREEDY_TXT = [None] * n_val, [''] * n_val
for j in range(n_val):
    if OOF[j]:
        em = VAL_EMS[MODELS[OOF[j][0]]['tag']][j]
        GREEDY_IDS[j] = greedy_collapse(torch.from_numpy(em.astype(np.float32))[:, None, :], [em.shape[0]])[0]
        GREEDY_TXT[j] = phones_to_words(GREEDY_IDS[j], PRON2WORDS)
_oof_idx = [j for j in range(n_val) if OOF[j]]
print(f"OOF greedy PER {official_per([GREEDY_IDS[j] for j in _oof_idx], [list(readers['val'].meta[j]['phonemes']) for j in _oof_idx])[0] * 100:.2f}% | "
      f"OOF greedy-lexicon WER {official_wer([val_refs[j] for j in _oof_idx], [GREEDY_TXT[j] for j in _oof_idx])[0] * 100:.2f}%")
# ---- fallback submission #1 (greedy lexicon, ensemble-free) - replaced later if everything succeeds ----
_greedy_test = []
for j in range(n_test):
    em = TEST_EMS[MODELS[0]['tag']][j]
    _greedy_test.append(phones_to_words(greedy_collapse(torch.from_numpy(em.astype(np.float32))[:, None, :], [em.shape[0]])[0], PRON2WORDS))
pd.DataFrame({'id': range(n_test), 'text': _greedy_test}).to_csv(os.path.join(WORK, 'submission.csv'), index=False)
print('fallback submission.csv #1 written (greedy lexicon)')
print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')

checkpoints: /kaggle/input/datasets/anibiswas1906125/b2t-v6-crossfit-2/b2t_v6_ckpt
days routed through the generic slot: none
checkpoint groups: {'foldA_seed0': ['best_per.pt', 'best_wer.pt'], 'foldB_seed1': ['best_per.pt', 'best_wer.pt']}
  beam processes: 1 (requested 3; RAM available 30.4 GB, KenLM file 19.12 GB, budget 26.4 GB/process)
  [screen] 120/1200  297s | RAM 10%
  [screen] 240/1200  301s | RAM 10%
  [screen] 360/1200  305s | RAM 10%
  [screen] 480/1200  310s | RAM 10%
  [screen] 600/1200  314s | RAM 10%
  [screen] 720/1200  318s | RAM 10%
  [screen] 840/1200  323s | RAM 10%
  [screen] 960/1200  327s | RAM 10%
  [screen] 1080/1200  332s | RAM 10%
  [screen] 1200/1200  336s | RAM 10%
foldA_seed0: using best_per.pt (screen WER 7.61%), trained on val labels ['A']
foldB_seed1: using best_wer.pt (screen WER 7.46%), trained on val labels ['B']
  table  -> /kaggle/working/tables/checkpoint_screening.csv
screening took 369s
OOF greedy PER 6.86% | OOF greedy-lexicon WER 43.02%
fallb

In [12]:
%%run_if llm --soft
# ============================================================================
# BLOCK 4.2 - CANDIDATE POOL (flashlight n-best x OOF models x lm_weights) + fallback submission
#   Generation happens ONCE per beam, at the UNION of every lm_weight the sweep may
#   ask for and at max(nbest). Any (subset of weights, n <= max nbest) variant is then
#   free to evaluate in BLOCK 4.2b, because n-best lists are nested.
# ============================================================================
_SW = SWEEP_CFG if SWEEP_CFG.get('enable') else {}
_lw_union = set(LLM_CFG['gen_lm_weights']) | {LLM_CFG['screen_lm_weight']}
for _s in _SW.get('lm_weight_sets', []):
    _lw_union |= set(float(w) for w in _s)
GEN_LW = sorted(_lw_union)
NBEST_MAX = int(max([LLM_CFG['gen_nbest']] + [int(n) for n in _SW.get('nbest', [])]))
BEAM_GRID = sorted(set([int(LLM_CFG['gen_beam'])] + [int(b) for b in _SW.get('beam', [])]))

tasks = [(('val', j, mi), VAL_EMS[MODELS[mi]['tag']][j]) for j in range(n_val) for mi in OOF[j]]
tasks += [(('test', j, mi), TEST_EMS[MODELS[mi]['tag']][j]) for j in range(n_test) for mi in range(len(MODELS))]

GEN_BY_BEAM = {}
for _bi, _beam in enumerate(BEAM_GRID):
    _left = SESSION_HOURS * 60 - (time.time() - NOTEBOOK_T0) / 60
    if _bi and _left < _SW.get('min_minutes_left_for_extra_beam', 0):
        print(f'skipping beam={_beam}: only {_left:.0f} min of session left '
              f'(needs {_SW.get("min_minutes_left_for_extra_beam")})')
        continue
    gen_spec = dict(lexicon=LEXICON_PATH, tokens=TOKENS_PATH, lm=KENLM_PATH, beam=_beam,
                    nbest=NBEST_MAX, lm_weights=GEN_LW)
    print(f'beam search: {len(tasks)} decodes x {len(GEN_LW)} lm_weights {GEN_LW}, beam={_beam}, '
          f'nbest={NBEST_MAX}, {LLM_CFG["n_proc"]} processes')
    t0 = time.time()
    GEN_BY_BEAM[_beam] = run_beam_pool(tasks, gen_spec, LLM_CFG['n_proc'], desc=f'generate b{_beam}')
    print(f'  beam={_beam} generation took {human_time(time.time() - t0)}')
if not GEN_BY_BEAM:
    raise RuntimeError('no beam width could be generated')


def build_pool(split, n, model_lists, gen=None, lw_set=None, nbest=None):
    gen = GEN if gen is None else gen
    lw_set = GEN_LW if lw_set is None else lw_set
    nbest = NBEST_MAX if nbest is None else nbest
    pool = Pool()
    for j in range(n):
        texts, src, seen = [], [], set()
        for mi in model_lists[j]:
            for lw in lw_set:
                for r, t in enumerate(gen[(split, j, mi)][lw][:nbest]):
                    if t not in seen:
                        seen.add(t); texts.append(t); src.append(f"{MODELS[mi]['tag']}@{lw}#{r}")
        if not texts:
            texts, src = [''], ['empty#0']
        pool.add_utt(texts, src)
    return pool


# provisional selection = widest pool available; BLOCK 4.2b replaces it with the swept winner
BEAM_SEL, LW_SEL, NBEST_SEL = max(GEN_BY_BEAM), list(GEN_LW), NBEST_MAX
GEN = GEN_BY_BEAM[BEAM_SEL]
POOL_VAL = build_pool('val', n_val, OOF)
POOL_TEST = build_pool('test', n_test, [list(range(len(MODELS)))] * n_test)
_lw0 = LLM_CFG['screen_lm_weight']
BASE_VAL = [(GEN[('val', j, OOF[j][0])][_lw0] or [''])[0] if OOF[j] else '' for j in range(n_val)]
BASE_TEST = [(GEN[('test', j, 0)][_lw0] or [''])[0] for j in range(n_test)]

TUNE_IDX = [j for j in range(n_val) if OOF[j] and VAL_LABELS[j] in LLM_CFG['tune_labels']]
VERIFY_IDX = [j for j in range(n_val) if OOF[j] and VAL_LABELS[j] in LLM_CFG['verify_labels']]
ALL_IDX = [j for j in range(n_val) if OOF[j]]
if not TUNE_IDX:
    print('WARNING: no out-of-fold A/B trials -> tuning on all OOF trials'); TUNE_IDX = ALL_IDX


def wer_on(idx, hyps):
    hyps = hyps if isinstance(hyps, dict) else dict(enumerate(hyps))
    return official_wer([val_refs[j] for j in idx], [hyps[j] for j in idx])[0]


print(f'generated beams: {sorted(GEN_BY_BEAM)} | provisional pool: beam={BEAM_SEL} '
      f'lm_weights={LW_SEL} nbest={NBEST_SEL}')
print(f'pool size: val mean {np.mean([len(t) for t in POOL_VAL.texts]):.1f} | test mean {np.mean([len(t) for t in POOL_TEST.texts]):.1f}')
print(f'flashlight 1-best (lm_weight={_lw0}, first OOF model): tune {wer_on(TUNE_IDX, BASE_VAL) * 100:.2f}% | '
      f'verify(C) {wer_on(VERIFY_IDX, BASE_VAL) * 100:.2f}% | all OOF {wer_on(ALL_IDX, BASE_VAL) * 100:.2f}%')

sub_df = pd.DataFrame({'id': range(n_test), 'text': BASE_TEST})
sub_df.to_csv(os.path.join(WORK, 'submission.csv'), index=False)
print('fallback submission.csv #2 written (flashlight 1-best) - replaced by BLOCK 4.6')
print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')


beam search: 4610 decodes x 8 lm_weights [0.5, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0], beam=300, nbest=75, 3 processes
  beam processes: 1 (requested 3; RAM available 28.6 GB, KenLM file 19.12 GB, budget 26.4 GB/process)
  [generate b300] 461/4610  713s | RAM 15%
  [generate b300] 922/4610  1600s | RAM 15%
  [generate b300] 1383/4610  2440s | RAM 15%
  [generate b300] 1844/4610  3276s | RAM 15%
  [generate b300] 2305/4610  3950s | RAM 16%
  [generate b300] 2766/4610  4756s | RAM 16%
  [generate b300] 3227/4610  5643s | RAM 16%
  [generate b300] 3688/4610  6480s | RAM 16%
  [generate b300] 4149/4610  7289s | RAM 16%
  [generate b300] 4610/4610  8236s | RAM 16%
  beam=300 generation took 2h17m
beam search: 4610 decodes x 8 lm_weights [0.5, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0], beam=400, nbest=75, 3 processes
  beam processes: 1 (requested 3; RAM available 28.3 GB, KenLM file 19.12 GB, budget 26.4 GB/process)
  [generate b400] 461/4610  949s | RAM 16%
  [generate b400] 922/4610  2146s | RAM 16%

In [13]:
%%run_if llm --soft --needs-ok
# ============================================================================
# BLOCK 4.2b - DECODING SWEEP (pool side): beam x lm_weights x nbest x expansion
#   Selection is on the TUNE folds (A u B) only. Fold C is carried along in the table
#   purely so you can see whether the choice transferred - it is never selected on.
#   Everything here reuses the ONE generation from BLOCK 4.2, so the only real cost
#   is the CTC feature pass over genuinely new hypothesis strings.
# ============================================================================
LEX = PronLexicon(LEXICON_PATH, sil_trailing=SIL_INFO['sil_trailing'])
try:
    NGRAM = NgramScorer(KENLM_PATH)
except Exception as e:
    NGRAM = None
    print('WARNING: kenlm python module unavailable -> n-gram feature disabled:', repr(e))
UTT_EMS_VAL = [[VAL_EMS[MODELS[mi]['tag']][j] for mi in OOF[j]] for j in range(n_val)]
UTT_EMS_TEST = [[TEST_EMS[M['tag']][j] for M in MODELS] for j in range(n_test)]

S1_GRID = (SWEEP_CFG['stage1_grid'][0] if SWEEP_CFG.get('enable') else 'base')
A1, G1 = stage1_axes(S1_GRID)


class FeatCache:
    """memoise (am_mean, am_min, ng) per (utterance, hypothesis). A sweep rebuilds the
    pool many times over largely the same strings; without this every variant would pay
    the full CTC pass again. Drop-in replacement for fill_features()."""

    def __init__(self, n, enabled=True):
        self.d = [dict() for _ in range(n)] if enabled else None
        self.hits = self.miss = 0

    def fill(self, pool, utt_ems, lex, ngram, device, only_new_from=None, log=print):
        if self.d is None:
            return fill_features(pool, utt_ems, lex, ngram, device, only_new_from=only_new_from, log=log)
        t0 = time.time()
        for u in range(len(pool)):
            start = 0 if only_new_from is None else only_new_from[u]
            texts = pool.texts[u][start:]
            if not texts:
                continue
            d = self.d[u]
            miss = [t for t in dict.fromkeys(texts) if t not in d]
            if miss:
                A = acoustic_scores(miss, utt_ems[u], lex, device)
                ng = [ngram.ln(t) for t in miss] if ngram is not None else [0.0] * len(miss)
                for i, t in enumerate(miss):
                    d[t] = (float(A[i].mean()), float(A[i].min()), float(ng[i]))
            self.miss += len(miss); self.hits += len(texts) - len(miss)
            v = [d[t] for t in texts]
            pool.am[u][start:] = [x[0] for x in v]
            pool.am_min[u][start:] = [x[1] for x in v]
            pool.ng[u][start:] = [x[2] for x in v]
            if (u + 1) % 500 == 0 and miss:
                log(f'  features {u + 1}/{len(pool)}  {time.time() - t0:.0f}s')

    def release(self):
        self.d = None
        gc.collect()


FCV = FeatCache(n_val, enabled=bool(SWEEP_CFG.get('cache_features', True)))
FCT = FeatCache(n_test, enabled=False)          # test pool is built once; no cache needed


def stage1_on(pool, errs, nref, idx, A=None, G=None):
    A = A1 if A is None else A
    G = G1 if G is None else G
    tu = Tuner(pool, errs, nref, idx)
    err = tu.grid(A, [0.0], G)
    W, Wraw = pick_from_grid(err, A, [0.0], G, smooth=LLM_CFG['smooth_grid'])
    return W, Wraw, err


def boot_std(idx, pred, n_boot):
    """utterance-bootstrap sd of the corpus WER: the noise floor a sweep margin must beat."""
    if not n_boot or len(idx) < 10:
        return float('nan')
    en = [utt_word_errors(val_refs[j], pred[j]) for j in idx]
    e = np.array([x[0] for x in en], dtype=np.float64)
    n = np.array([x[1] for x in en], dtype=np.float64)
    I = np.random.default_rng(SEED).integers(0, len(idx), (int(n_boot), len(idx)))
    return float(np.std(e[I].sum(1) / np.maximum(n[I].sum(1), 1.0)))


SW_ROWS, SW_T0 = [], time.time()
_SEEN_VARIANTS = {}


def eval_variant(beam, lw_set, nbest, exp, tag='', s1grid=None):
    s1grid = s1grid or S1_GRID
    key = (int(beam), tuple(float(w) for w in lw_set), int(nbest),
           int(exp['top']), int(exp['max_nb']), int(exp['max_new']), s1grid)
    if key in _SEEN_VARIANTS:
        return _SEEN_VARIANTS[key]
    _A, _G = stage1_axes(s1grid)
    t0 = time.time()
    pv = build_pool('val', n_val, OOF, GEN_BY_BEAM[beam], lw_set, nbest)
    FCV.fill(pv, UTT_EMS_VAL, LEX, NGRAM, DEVICE, log=lambda *a: None)
    errs, nref = errors_lists(pv, val_refs)
    W, _, _ = stage1_on(pv, errs, nref, TUNE_IDX, _A, _G)
    if exp['top'] > 0 and LLM_CFG['expansion']:
        st = expand_pool(pv, LEX, NGRAM, W, exp['top'], exp['max_nb'], exp['max_new'])
        FCV.fill(pv, UTT_EMS_VAL, LEX, NGRAM, DEVICE, only_new_from=st, log=lambda *a: None)
        errs, nref = errors_lists(pv, val_refs)
        W, _, _ = stage1_on(pv, errs, nref, TUNE_IDX, _A, _G)
    pred = predict_texts(pv, W)[0]
    row = {'beam': int(beam), 'nbest': int(nbest), 'lm_weights': repr(list(lw_set)),
           'expand_top': int(exp['top']), 'expand_max_nb': int(exp['max_nb']),
           'expand_max_new': int(exp['max_new']), 'stage1_grid': s1grid,
           'edge': bool(on_edge(W, _A, None, _G)),
           'pool_mean': round(float(np.mean([len(t) for t in pv.texts])), 1),
           'tune_WER_%': round(100 * wer_on(TUNE_IDX, pred), 3),
           'verify_C_WER_%': round(100 * wer_on(VERIFY_IDX, pred), 3) if VERIFY_IDX else float('nan'),
           'oracle_tune_%': round(100 * sum(errs[j].min() for j in TUNE_IDX) / max(nref[TUNE_IDX].sum(), 1), 3),
           'secs': round(time.time() - t0, 1), 'stage': tag,
           'weights': {k: W[k] for k in ('ng', 'nw')}}
    row['_pred'] = pred
    SW_ROWS.append(row); _SEEN_VARIANTS[key] = row
    print(f"  [{tag}] beam={beam} nbest={nbest:>3} lw={row['lm_weights']:<28} "
          f"exp=({exp['top']},{exp['max_nb']},{exp['max_new']}) s1={s1grid:<4} "
          f"pool={row['pool_mean']:>6.1f}{' EDGE' if row['edge'] else '     '} | "
          f"tune {row['tune_WER_%']:.2f}% | C {row['verify_C_WER_%']:.2f}% | "
          f"oracle {row['oracle_tune_%']:.2f}% | {row['secs']:.0f}s")
    del pv
    gc.collect()
    return row


def budget_left():
    return SWEEP_CFG['time_budget_min'] - (time.time() - SW_T0) / 60


if SWEEP_CFG.get('enable'):
    BEAMS = sorted(GEN_BY_BEAM)
    NBESTS = sorted(set(int(n) for n in SWEEP_CFG['nbest'] if int(n) <= NBEST_MAX))
    LWSETS = [sorted(set(float(w) for w in s) & set(GEN_LW)) for s in SWEEP_CFG['lm_weight_sets']]
    LWSETS = [s for s in LWSETS if s] or [list(GEN_LW)]
    LWSETS = list(dict.fromkeys(tuple(s) for s in LWSETS))
    ETOP = sorted(set(int(v) for v in SWEEP_CFG['expand_top']))
    ENB = sorted(set(int(v) for v in SWEEP_CFG['expand_max_nb']))
    ENEW = sorted(set(int(v) for v in SWEEP_CFG['expand_max_new']))
    S1G = [g for g in SWEEP_CFG['stage1_grid'] if g in STAGE1_GRIDS] or ['base']
    exp0 = {'top': LLM_CFG['expand_top'], 'max_nb': LLM_CFG['expand_max_nb'], 'max_new': LLM_CFG['expand_max_new']}
    print(f'sweep: beams {BEAMS} x nbest {NBESTS} x {len(LWSETS)} weight sets x stage-1 grid {S1G}, '
          f'then expansion {ETOP} x {ENB} x {ENEW} | mode={SWEEP_CFG["mode"]} | '
          f'budget {SWEEP_CFG["time_budget_min"]} min')

    best = None
    if SWEEP_CFG['mode'] == 'grid':
        for beam in BEAMS:
            for lw in LWSETS:
                for nb in NBESTS:
                    for et in ETOP:
                        for en in ENB:
                            for ew in (ENEW if et > 0 else ENEW[:1]):
                                for sg in S1G:
                                    if budget_left() <= 0:
                                        break
                                    eval_variant(beam, list(lw), nb,
                                                 {'top': et, 'max_nb': en, 'max_new': ew}, 'grid', sg)
        best = min(SW_ROWS, key=lambda r: r['tune_WER_%'])
    else:
        cur = {'beam': BEAMS[-1], 'lw': list(LWSETS[-1]), 'nbest': max(NBESTS),
               's1grid': S1G[0], **exp0}
        for p in range(int(SWEEP_CFG.get('passes', 2))):
            for axis in ('lw', 'nbest', 'beam', 'top', 'max_nb', 'max_new', 's1grid'):
                if budget_left() <= 0:
                    print(f'  budget exhausted after pass {p + 1}, axis {axis} - keeping the best so far')
                    break
                grid = {'lw': [list(s) for s in LWSETS], 'nbest': NBESTS, 'beam': BEAMS,
                        'top': ETOP, 'max_nb': ENB, 'max_new': ENEW, 's1grid': S1G}[axis]
                if len(grid) < 2 and p > 0:
                    continue
                if axis in ('max_nb', 'max_new') and cur['top'] == 0:
                    continue
                rows = []
                for v in grid:
                    c = dict(cur); c[axis] = v
                    rows.append((v, eval_variant(c['beam'], c['lw'], c['nbest'],
                                                 {'top': c['top'], 'max_nb': c['max_nb'], 'max_new': c['max_new']},
                                                 f'p{p + 1}:{axis}', c['s1grid'])))
                v, r = min(rows, key=lambda vr: vr[1]['tune_WER_%'])
                if cur[axis] != v:
                    print(f'   -> {axis}: {cur[axis]} -> {v}  (tune {r["tune_WER_%"]:.2f}%)')
                cur[axis] = v
            if budget_left() <= 0:
                break
        best = min(SW_ROWS, key=lambda r: r['tune_WER_%'])

    # ---- results table + noise floor -------------------------------------------
    SW_DF = pd.DataFrame([{k: v for k, v in r.items() if k not in ('_pred', 'weights')} for r in SW_ROWS])
    SW_DF = SW_DF.sort_values('tune_WER_%').reset_index(drop=True)
    save_table(SW_DF, 'decoding_sweep.csv')
    sd = boot_std(TUNE_IDX, best['_pred'], SWEEP_CFG.get('bootstrap', 0))
    print(f'\n{len(SW_ROWS)} variants evaluated in {(time.time() - SW_T0) / 60:.1f} min')
    print(SW_DF.head(10).drop(columns=['stage'], errors='ignore').to_string(index=False))
    print(f'\nbest on TUNE: beam={best["beam"]} nbest={best["nbest"]} lw={best["lm_weights"]} '
          f'expand=({best["expand_top"]},{best["expand_max_nb"]},{best["expand_max_new"]}) '
          f'-> tune {best["tune_WER_%"]:.2f}% (bootstrap sd {sd * 100:.2f} pp), C {best["verify_C_WER_%"]:.2f}%')
    _rival = [r for r in SW_ROWS if r is not best and r['tune_WER_%'] - best['tune_WER_%'] < 100 * sd]
    if _rival:
        print(f'  {len(_rival)} other configs are within one bootstrap sd on tune - the margin is '
              f'inside the noise, so do not read the winner as strictly better.')

    # ---- rebuild the pipeline's pools from the winner ---------------------------
    BEAM_SEL, LW_SEL, NBEST_SEL = best['beam'], eval(best['lm_weights']), best['nbest']
    S1_GRID = best['stage1_grid']
    A1, G1 = stage1_axes(S1_GRID)
    if best['edge']:
        print('NOTE: the winning stage-1 weights sit on the edge of their grid - widen '
              "STAGE1_GRIDS['wide'] and re-run if you want that axis genuinely optimised.")
    LLM_CFG['gen_beam'], LLM_CFG['gen_nbest'] = BEAM_SEL, NBEST_SEL
    LLM_CFG['gen_lm_weights'] = LW_SEL
    LLM_CFG['expand_top'] = best['expand_top']
    LLM_CFG['expand_max_nb'] = best['expand_max_nb']
    LLM_CFG['expand_max_new'] = best['expand_max_new']
    LLM_CFG['expansion'] = LLM_CFG['expansion'] and best['expand_top'] > 0
    SWEEP_BEST = {k: best[k] for k in ('beam', 'nbest', 'lm_weights', 'expand_top', 'expand_max_nb',
                                       'expand_max_new', 'stage1_grid', 'edge', 'tune_WER_%',
                                       'verify_C_WER_%', 'pool_mean')}
    SWEEP_BEST['tune_bootstrap_sd_pp'] = round(100 * sd, 3) if sd == sd else None
    json.dump(SWEEP_BEST, open(os.path.join(tables_dir, 'decoding_sweep_best.json'), 'w'), indent=1, default=float)

    for _b in list(GEN_BY_BEAM):                      # free the beams we did not pick
        if _b != BEAM_SEL:
            del GEN_BY_BEAM[_b]
    gc.collect()
    GEN = GEN_BY_BEAM[BEAM_SEL]
    POOL_VAL = build_pool('val', n_val, OOF, GEN, LW_SEL, NBEST_SEL)
    POOL_TEST = build_pool('test', n_test, [list(range(len(MODELS)))] * n_test, GEN, LW_SEL, NBEST_SEL)
    BASE_VAL = [(GEN[('val', j, OOF[j][0])][_lw0] or [''])[0] if OOF[j] else '' for j in range(n_val)]
    BASE_TEST = [(GEN[('test', j, 0)][_lw0] or [''])[0] for j in range(n_test)]
    SWEEP_TUNE_TARGET = best['tune_WER_%']
    print(f'pools rebuilt from the winner: val mean {np.mean([len(t) for t in POOL_VAL.texts]):.1f} | '
          f'test mean {np.mean([len(t) for t in POOL_TEST.texts]):.1f}')
else:
    SWEEP_TUNE_TARGET = None
    print('sweep disabled -> BLOCK 4.3 uses the fixed LLM_CFG values')
for r in SW_ROWS:
    r.pop('_pred', None)
gc.collect()
print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')


sweep: beams [300, 400] x nbest [25, 50, 75] x 4 weight sets x stage-1 grid ['base', 'wide'], then expansion [0, 3, 5, 8] x [12, 20] x [600, 1200] | mode=staged | budget 75 min
  [p1:lw] beam=400 nbest= 75 lw=[3.0]                        exp=(3,12,600) s1=base pool= 265.9 EDGE | tune 4.95% | C 4.62% | oracle 3.52% | 24s
  [p1:lw] beam=400 nbest= 75 lw=[2.0, 3.0, 4.0]              exp=(3,12,600) s1=base pool= 297.1      | tune 4.95% | C 4.57% | oracle 3.42% | 12s
  [p1:lw] beam=400 nbest= 75 lw=[0.5, 2.0, 3.0, 3.5, 5.0]    exp=(3,12,600) s1=base pool= 353.2 EDGE | tune 4.85% | C 4.67% | oracle 3.24% | 12s
  [p1:lw] beam=400 nbest= 75 lw=[0.5, 1.5, 2.5, 3.0, 4.0, 5.0] exp=(3,12,600) s1=base pool= 359.7      | tune 4.81% | C 4.67% | oracle 3.19% | 9s
  [p1:nbest] beam=400 nbest= 25 lw=[0.5, 1.5, 2.5, 3.0, 4.0, 5.0] exp=(3,12,600) s1=base pool= 252.4      | tune 4.81% | C 4.67% | oracle 3.26% | 4s
  [p1:nbest] beam=400 nbest= 50 lw=[0.5, 1.5, 2.5, 3.0, 4.0, 5.0] exp=(3,12,600) s1=base pool

In [14]:
%%run_if llm --soft --needs-ok
# ============================================================================
# BLOCK 4.3 - EXACT FEATURES (CTC / KenLM / #words), stage-1 tuning, phonetic expansion
# ============================================================================
# LEX / NGRAM / UTT_EMS_* / A1 / G1 / the feature cache are built in BLOCK 4.2b and
# reused here, so the winning configuration is re-scored with exactly the same objects.
t0 = time.time()
FCV.fill(POOL_VAL, UTT_EMS_VAL, LEX, NGRAM, DEVICE)
FCT.fill(POOL_TEST, UTT_EMS_TEST, LEX, NGRAM, DEVICE)
print(f'features: {human_time(time.time() - t0)} '
      f'(val feature cache: {FCV.hits} hits / {FCV.miss} computed)')
ABL = {}          # name -> {utt: text}


def stage1(pool, errs, idx):
    return stage1_on(pool, errs, NREF, idx)


def oracle_wer(errs, idx):
    return sum(errs[j].min() for j in idx) / max(NREF[idx].sum(), 1)


ERRS, NREF = errors_lists(POOL_VAL, val_refs)
ABL['greedy lexicon (1 model)'] = dict(enumerate(GREEDY_TXT))
ABL[f'flashlight 1-best lm={_lw0} (1 model)'] = dict(enumerate(BASE_VAL))
W1, W1raw, ERR1 = stage1(POOL_VAL, ERRS, TUNE_IDX)
ABL['pool rescoring: CTC-ens + KenLM + #words'] = predict_texts(POOL_VAL, W1)[0]
ORACLE = {'pool (no expansion)': (oracle_wer(ERRS, TUNE_IDX), oracle_wer(ERRS, VERIFY_IDX))}
print(f'stage-1 weights (smoothed) {W1} | raw-best {W1raw}')
print(f"  tune {wer_on(TUNE_IDX, ABL['pool rescoring: CTC-ens + KenLM + #words']) * 100:.2f}% | "
      f"verify {wer_on(VERIFY_IDX, ABL['pool rescoring: CTC-ens + KenLM + #words']) * 100:.2f}% | "
      f"oracle tune {ORACLE['pool (no expansion)'][0] * 100:.2f}%")
W_STAGE1 = W1

if LLM_CFG['expansion']:
    t0 = time.time()
    st_v = expand_pool(POOL_VAL, LEX, NGRAM, W1, LLM_CFG['expand_top'], LLM_CFG['expand_max_nb'], LLM_CFG['expand_max_new'])
    fill_features(POOL_VAL, UTT_EMS_VAL, LEX, NGRAM, DEVICE, only_new_from=st_v)
    st_t = expand_pool(POOL_TEST, LEX, NGRAM, W1, LLM_CFG['expand_top'], LLM_CFG['expand_max_nb'], LLM_CFG['expand_max_new'])
    fill_features(POOL_TEST, UTT_EMS_TEST, LEX, NGRAM, DEVICE, only_new_from=st_t)
    ERRS, NREF = errors_lists(POOL_VAL, val_refs)
    W1x, W1xraw, ERR1 = stage1(POOL_VAL, ERRS, TUNE_IDX)
    ABL['+ phonetic-neighbour expansion'] = predict_texts(POOL_VAL, W1x)[0]
    ORACLE['pool + expansion'] = (oracle_wer(ERRS, TUNE_IDX), oracle_wer(ERRS, VERIFY_IDX))
    print(f'expansion: {human_time(time.time() - t0)} | pool mean {np.mean([len(t) for t in POOL_VAL.texts]):.0f} | '
          f'weights {W1x} | tune {wer_on(TUNE_IDX, ABL["+ phonetic-neighbour expansion"]) * 100:.2f}% | '
          f'verify {wer_on(VERIFY_IDX, ABL["+ phonetic-neighbour expansion"]) * 100:.2f}% | '
          f"oracle tune {ORACLE['pool + expansion'][0] * 100:.2f}%")
    W_STAGE1 = W1x

if SWEEP_TUNE_TARGET is not None:
    _got = 100 * wer_on(TUNE_IDX, ABL.get('+ phonetic-neighbour expansion',
                                          ABL['pool rescoring: CTC-ens + KenLM + #words']))
    print(f'sweep reproduction check: BLOCK 4.2b reported {SWEEP_TUNE_TARGET:.2f}% on tune, '
          f'BLOCK 4.3 reproduces {_got:.2f}%'
          + ('  OK' if abs(_got - SWEEP_TUNE_TARGET) < 0.05 else '  <-- MISMATCH, investigate'))
    FCV.release()          # the sweep is over; give the RAM back before the LLM stage

with figure_guard('stage1_surface'):
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(ERR1[:, 0, :] / NREF[TUNE_IDX].sum() * 100, aspect='auto', origin='lower', cmap='viridis',
                   extent=[G1[0], G1[-1], A1[0], A1[-1]])
    ax.scatter([W_STAGE1['nw']], [W_STAGE1['ng']], c='red', marker='*', s=150)
    ax.set_xlabel('word bonus (per word)'); ax.set_ylabel('KenLM weight (ln units)')
    ax.set_title('Stage-1 tune-set WER (%)'); fig.colorbar(im, ax=ax)
    save_fig(fig, 'stage1_weight_surface')
print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')

  features 500/1450  3s
  features 1000/1450  6s
features: 0h00m (val feature cache: 9187833 hits / 1062266 computed)
stage-1 weights (smoothed) {'ng': 1.6, 'llm': 0.0, 'nw': -0.75, 'err': 431.0} | raw-best {'ng': 1.6, 'llm': 0.0, 'nw': -1.0, 'err': 431.0}
  tune 5.68% | verify 5.42% | oracle tune 4.22%
  features 500/1426  3s
  features 1000/1426  7s
  features 500/1450  5s
  features 1000/1450  10s
expansion: 0h00m | pool mean 252 | weights {'ng': 1.6, 'llm': 0.0, 'nw': -8.0, 'err': 364.0} | tune 4.80% | verify 4.67% | oracle tune 3.26%
sweep reproduction check: BLOCK 4.2b reported 4.80% on tune, BLOCK 4.3 reproduces 4.80%  OK
  figure -> /kaggle/working/figures/stage1_weight_surface.png
session clock: 5h42m


In [15]:
%%run_if llm --soft --needs-ok
# ============================================================================
# BLOCK 4.4 - TASK-ADAPTED LLM: LoRA next-word-prediction on TRAIN sentences, score top-K
# ============================================================================
LLM_OK, LLM_HIST = False, {}
if LLM_CFG['use_llm'] and (N_GPUS > 0 or os.environ.get('B2T_LOCAL_TEST')):
    try:
        for M in MODELS:                      # emissions are cached; free the GPU for the LLM
            M['model'] = None
        gc.collect(); torch.cuda.empty_cache()
        t0 = time.time()
        train_sents = [remove_punctuation(m['sentence']) for m in readers['train'].meta]
        if NGRAM is not None:
            NGRAM.m = None                    # free the KenLM RAM during the LLM stage (score cache is kept)
            gc.collect()
        _ft_ok = True
        try:
            LLM_MODEL, LLM_TOK, LLM_HIST = finetune_llm_nwp(LLM_CFG['llm_name'], train_sents, LLM_CFG, log=print)
        except Exception as e:
            traceback.print_exc()
            print('LoRA fine-tuning failed -> using the base LLM without fine-tuning:', repr(e))
            _ft_ok = False
        if not _ft_ok:                        # outside the except block: the failed model is released first
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            LLM_MODEL, LLM_TOK, LLM_HIST = finetune_llm_nwp(LLM_CFG['llm_name'], train_sents, dict(LLM_CFG, llm_finetune=False), log=print)
        SCORER = LLMScorer(LLM_MODEL, LLM_TOK, 1)  # single-GPU only: copy.deepcopy() to a 2nd GPU
                                            # clones the model on its *current* device first,
                                            # which briefly needs 2x its footprint on GPU0
        print(f'LLM ready in {human_time(time.time() - t0)}: {LLM_HIST}')
        # score the LARGEST k the sweep may ask for; any smaller k is a prefix of it,
        # so BLOCK 4.5 can sweep llm_topk without a single extra LLM forward pass
        TOPK_MAX = max([LLM_CFG['llm_topk']] + [int(k) for k in SWEEP_CFG['llm_topk']]) \
            if SWEEP_CFG.get('enable') else LLM_CFG['llm_topk']
        print(f'  scoring top-{TOPK_MAX} per utterance (llm_topk sweep grid: '
              f'{sorted(set(SWEEP_CFG["llm_topk"])) if SWEEP_CFG.get("enable") else [LLM_CFG["llm_topk"]]})')
        MASK_VAL = topk_masks(POOL_VAL, W_STAGE1, TOPK_MAX)
        MASK_TEST = topk_masks(POOL_TEST, W_STAGE1, TOPK_MAX)
        for pool, masks in ((POOL_VAL, MASK_VAL), (POOL_TEST, MASK_TEST)):
            texts = sorted(set(pool.texts[u][i] for u in range(len(pool)) for i in np.where(masks[u] > 0)[0]))
            t1 = time.time()
            lp, nt = SCORER.score(texts, bs=LLM_CFG['llm_batch'])
            lut = dict(zip(texts, zip(lp, nt)))
            for u in range(len(pool)):
                for i in np.where(masks[u] > 0)[0]:
                    pool.llm[u][i], pool.ntok[u][i] = lut[pool.texts[u][i]]
            print(f'  scored {len(texts)} unique hypotheses in {human_time(time.time() - t1)}')
        LLM_OK = True
        json.dump({k: (float(v) if isinstance(v, (int, float, np.floating)) else v) for k, v in LLM_HIST.items()},
                  open(os.path.join(tables_dir, 'llm_finetune_history.json'), 'w'), indent=1)
    except Exception as e:
        traceback.print_exc()
        print('LLM stage failed -> continuing with acoustic + n-gram rescoring only:', repr(e))
else:
    print('LLM disabled (use_llm=False or no GPU).')
print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

  [llm-ft] meta-llama/Llama-3.1-8B: 6942 train / 365 dev sentences, dev NLL/token before = 8.199 (QLoRA 4-bit, batch=6 x accum=6)


/kaggle/working/b2t_code/b2t_decode.py:578: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(opt); scaler.update(); sched.step()


  [llm-ft] epoch 1/2: dev NLL/token 3.092 (1040s)
  [llm-ft] epoch 2/2: dev NLL/token 3.147 (1039s)
LLM ready in 0h37m: {'dev_nll_before': 8.199132789779043, 'dev_nll_ep1': 3.091699439160823, 'dev_nll_ep2': 3.1469309810593193, 'finetuned': True, 'best_dev_nll': 3.091699439160823}
  scoring top-40 per utterance (llm_topk sweep grid: [12, 24, 40])
  scored 56492 unique hypotheses in 0h22m
  scored 58268 unique hypotheses in 0h23m
session clock: 7h06m


In [16]:
%%run_if llm --soft --needs-ok
# ============================================================================
# BLOCK 4.5 - STAGE-2 SWEEP (weight-grid range x tol_rel x smoothing x llm_topk x
#             fluency gate), VERIFY on C, ABLATION, ERROR ANALYSIS
#
#   The static run tuned one grid: ng in [0,2.5], llm in [0,2.0], nw in [-4,6], and
#   the winner came out at ng=2.5, llm=2.0, nw=-4.0 - every single weight pinned to a
#   boundary. That is not an optimum, that is the edge of the search. So the grid
#   RANGE is swept here, and every row is flagged when its weights land on an edge.
#
#   Cost note: the error grid depends on (llm_topk, grid preset, gate percentile).
#   tol_rel and smooth_grid only change how a cell is PICKED out of an already
#   computed grid, so they are swept by re-picking - no extra grid evaluations.
# ============================================================================
S2_PRESETS = [g for g in SWEEP_CFG['stage2_grid'] if g in STAGE2_GRIDS] or ['base'] \
    if SWEEP_CFG.get('enable') else ['base']
TOL_GRID = sorted(set(float(t) for t in SWEEP_CFG['tol_rel'])) if SWEEP_CFG.get('enable') else [0.0]
SMOOTH_GRID = list(dict.fromkeys(SWEEP_CFG['smooth_grid'])) if SWEEP_CFG.get('enable') \
    else [LLM_CFG['smooth_grid']]
A2, B2, G2 = stage2_axes(S2_PRESETS[0])          # replaced by the winner below


def gate_grids(pool, errs, idx, flu, percentiles, A, B, G):
    """error grids per fluency percentile; picking is done separately so tol_rel /
    smoothing can be swept without recomputing anything."""
    tu = Tuner(pool, errs, NREF, idx)
    f = flu[idx]
    out = []
    for p in percentiles:
        tau = -np.inf if p == 0 else float(np.percentile(f, p))
        lo, hi = np.where(f < tau)[0], np.where(f >= tau)[0]
        out.append({'percentile': p, 'tau': tau, 'n_lo': len(lo),
                    'hi_grid': tu.grid(A, B, G, rows=hi),
                    'lo_grid': tu.grid(A, B, G, rows=lo) if len(lo) else None})
    return out


def pick_gate(g, A, B, G, smooth, tol, nref_sum):
    W_hi, _ = pick_from_grid(g['hi_grid'], A, B, G, smooth=smooth, tol_rel=tol)
    W_lo, e_lo = None, 0.0
    if g['lo_grid'] is not None:
        W_lo, _ = pick_from_grid(g['lo_grid'], A, B, G, smooth=smooth, tol_rel=tol)
        e_lo = W_lo['err']
    return {'percentile': g['percentile'], 'tau': g['tau'], 'hi': W_hi, 'lo': W_lo,
            'err': W_hi['err'] + e_lo, 'wer': (W_hi['err'] + e_lo) / max(nref_sum, 1),
            'n_lo': g['n_lo'], 'err_hi_grid': g['hi_grid']}


def gate_search(pool, errs, idx, flu, percentiles, A=None, B=None, G=None,
                smooth=None, tol=None):
    """kept for the refit step: compute + pick in one call, with the winning settings."""
    A = A2 if A is None else A
    B = B2 if B is None else B
    G = G2 if G is None else G
    smooth = LLM_CFG['smooth_grid'] if smooth is None else smooth
    tol = LLM_CFG.get('tol_rel', 0.0) if tol is None else tol
    ns = NREF[idx].sum()
    return [pick_gate(g, A, B, G, smooth, tol, ns)
            for g in gate_grids(pool, errs, idx, flu, percentiles, A, B, G)]


if LLM_OK:
    K_GRID = sorted(set(int(k) for k in SWEEP_CFG['llm_topk'] if int(k) <= TOPK_MAX)) \
        if SWEEP_CFG.get('enable') else [LLM_CFG['llm_topk']]
    K_GRID = K_GRID or [TOPK_MAX]
    t0, S2_ROWS, BESTV = time.time(), [], None
    _ns = NREF[TUNE_IDX].sum()
    _budget = SWEEP_CFG.get('stage2_budget_min', 25) if SWEEP_CFG.get('enable') else 1e9
    for _k in K_GRID:
        _Mv = MASK_VAL if _k == TOPK_MAX else topk_masks(POOL_VAL, W_STAGE1, _k)
        _sv, _se = subpool(POOL_VAL, _Mv, ERRS)
        _flu = fluency(_sv, W_STAGE1)
        for _pre in S2_PRESETS:
            if (time.time() - t0) / 60 > _budget:
                print('  stage-2 budget exhausted - keeping the best so far'); break
            _A, _B, _G = stage2_axes(_pre)
            _grids = gate_grids(_sv, _se, TUNE_IDX, _flu, LLM_CFG['gate_percentiles'], _A, _B, _G)
            for _sm in SMOOTH_GRID:
                for _tol in TOL_GRID:
                    _cand = [pick_gate(g, _A, _B, _G, _sm, _tol, _ns) for g in _grids]
                    _bg = min(_cand, key=lambda r: (round(r['err']), r['percentile']))
                    _edge = on_edge(_bg['hi'], _A, _B, _G) or (
                        _bg['lo'] is not None and on_edge(_bg['lo'], _A, _B, _G))
                    S2_ROWS.append({'llm_topk': _k, 'grid': _pre, 'smooth': bool(_sm), 'tol_rel': _tol,
                                    'gate_percentile': _bg['percentile'],
                                    'tune_WER_%': round(100 * _bg['wer'], 3),
                                    'llm_w': _bg['hi']['llm'], 'ng_w': _bg['hi']['ng'],
                                    'nw_w': _bg['hi']['nw'], 'edge': bool(_edge)})
                    if BESTV is None or _bg['err'] < BESTV['bg']['err']:
                        BESTV = {'k': _k, 'grid': _pre, 'smooth': bool(_sm), 'tol': _tol,
                                 'mask': _Mv, 'sv': _sv, 'se': _se, 'flu': _flu,
                                 'gs': _cand, 'bg': _bg, 'axes': (_A, _B, _G)}
    S2_DF = pd.DataFrame(S2_ROWS).sort_values('tune_WER_%').reset_index(drop=True)
    save_table(S2_DF, 'stage2_sweep.csv')
    print(f'stage-2 sweep: {len(S2_ROWS)} settings in {time.time() - t0:.0f}s')
    print(S2_DF.head(12).to_string(index=False))

    # ---- adopt the winner --------------------------------------------------------
    LLM_CFG['llm_topk'] = BESTV['k']
    LLM_CFG['smooth_grid'] = BESTV['smooth']
    LLM_CFG['tol_rel'] = BESTV['tol']
    A2, B2, G2 = BESTV['axes']
    MASK_VAL = BESTV['mask']
    MASK_TEST = topk_masks(POOL_TEST, W_STAGE1, BESTV['k'])
    SUB_VAL, SUB_ERRS = BESTV['sv'], BESTV['se']
    SUB_TEST, _ = subpool(POOL_TEST, MASK_TEST)
    FLU_VAL, FLU_TEST = BESTV['flu'], fluency(SUB_TEST, W_STAGE1)
    GS = BESTV['gs']
    print(f"stage-2 selected on tune: llm_topk={BESTV['k']} grid={BESTV['grid']} "
          f"smooth={BESTV['smooth']} tol_rel={BESTV['tol']} gate p={BESTV['bg']['percentile']} "
          f"-> tune {BESTV['bg']['wer'] * 100:.2f}%")
    _wh = BESTV['bg']['hi']
    print(f"   hi weights ng={_wh['ng']} llm={_wh['llm']} nw={_wh['nw']}"
          + ("   <-- STILL ON A GRID EDGE: widen STAGE2_GRIDS and re-run"
             if on_edge(_wh, *BESTV['axes']) else "   (inside the grid)"))
    ORACLE[f"top-{LLM_CFG['llm_topk']} (LLM-scored)"] = (oracle_wer(SUB_ERRS, TUNE_IDX),
                                                         oracle_wer(SUB_ERRS, VERIFY_IDX))
    for r in GS:
        print(f"  gate p={r['percentile']:>2} tau={r['tau']:8.3f} n_lo={r['n_lo']:4d} tune WER {r['wer'] * 100:.2f}% | "
              f"hi {dict((k, r['hi'][k]) for k in ('ng', 'llm', 'nw'))} lo {None if r['lo'] is None else dict((k, r['lo'][k]) for k in ('ng', 'llm', 'nw'))}")
    NOGATE = GS[0]
    BEST_G = BESTV['bg']
    ABL['+ adapted LLM (no gate)'] = predict_texts(SUB_VAL, W=NOGATE['hi'])[0]
    ABL['+ fluency gate'] = predict_texts(SUB_VAL, gate=BEST_G, flu=FLU_VAL)[0]
    FINAL_POOL_VAL, FINAL_ERRS, FINAL_POOL_TEST = SUB_VAL, SUB_ERRS, SUB_TEST
    FINAL = {'gate': BEST_G, 'percentile': BEST_G['percentile']}
    # choose the gate only if it also holds on C; otherwise the simpler no-gate system
    v_gate = wer_on(VERIFY_IDX, ABL['+ fluency gate']) if VERIFY_IDX else 0
    v_nog = wer_on(VERIFY_IDX, ABL['+ adapted LLM (no gate)']) if VERIFY_IDX else 0
    if BEST_G['percentile'] != 0 and VERIFY_IDX and v_gate > v_nog:
        print(f'gate did not transfer to C ({v_gate * 100:.2f}% vs {v_nog * 100:.2f}%) -> using no-gate weights')
        FINAL = {'gate': NOGATE, 'percentile': 0}
    if LLM_CFG['refit_on_all_oof'] and VERIFY_IDX:
        FINAL['gate'] = gate_search(SUB_VAL, SUB_ERRS, ALL_IDX, FLU_VAL, [FINAL['percentile']],
                                    A2, B2, G2, BESTV['smooth'], BESTV['tol'])[0]
    FINAL_FLU_VAL, FINAL_FLU_TEST = FLU_VAL, FLU_TEST
else:
    FINAL_POOL_VAL, FINAL_ERRS, FINAL_POOL_TEST = POOL_VAL, ERRS, POOL_TEST
    Wf = W_STAGE1
    if LLM_CFG['refit_on_all_oof'] and VERIFY_IDX:
        Wf, _, _ = stage1(POOL_VAL, ERRS, ALL_IDX)
    FINAL = {'gate': {'tau': -np.inf, 'hi': Wf, 'lo': None}, 'percentile': 0}
    FINAL_FLU_VAL, FINAL_FLU_TEST = np.zeros(n_val), np.zeros(n_test)

FINAL_PRED_VAL, FINAL_REG_VAL = predict_texts(FINAL_POOL_VAL, gate=FINAL['gate'], flu=FINAL_FLU_VAL, sel=ALL_IDX)
ABL['FINAL (refit on A∪B∪C, in-sample on C)'] = FINAL_PRED_VAL

_fg = FINAL['gate']
best_w = {'percentile': FINAL['percentile'], 'tau': _fg['tau'],
          'hi': {k: _fg['hi'][k] for k in ('ng', 'llm', 'nw')},
          'lo': None if _fg['lo'] is None else {k: _fg['lo'][k] for k in ('ng', 'llm', 'nw')},
          'stage1': W_STAGE1, 'llm_used': LLM_OK, 'llm': LLM_CFG['llm_name'],
          'gen_beam': LLM_CFG['gen_beam'], 'gen_nbest': LLM_CFG['gen_nbest'],
          'gen_lm_weights': LLM_CFG['gen_lm_weights'], 'llm_topk': LLM_CFG['llm_topk'],
          'expand': [LLM_CFG['expand_top'], LLM_CFG['expand_max_nb'], LLM_CFG['expand_max_new']],
          'stage1_grid': globals().get('S1_GRID', 'base'),
          'stage2_grid': BESTV['grid'] if LLM_OK and BESTV else None,
          'smooth_grid': LLM_CFG['smooth_grid'], 'tol_rel': LLM_CFG.get('tol_rel', 0.0),
          'swept': bool(SWEEP_CFG.get('enable')),
          'models': [M['tag'] + ':' + os.path.basename(M['ckpt']) for M in MODELS]}
json.dump(best_w, open(os.path.join(tables_dir, 'best_weights.json'), 'w'), indent=1, default=float)
print('final weights:', best_w)

print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')


  table  -> /kaggle/working/tables/stage2_sweep.csv
stage-2 sweep: 54 settings in 127s
 llm_topk  grid  smooth  tol_rel  gate_percentile  tune_WER_%  llm_w  ng_w  nw_w  edge
       12  wide    True    0.005               10       4.471    2.4  3.00  -2.5  True
       12 wider   False    0.000                5       4.471    2.4  3.00  -3.0  True
       12  wide   False    0.005               10       4.471    2.4  3.00  -3.0  True
       12  wide   False    0.000                5       4.471    2.4  3.00  -3.0  True
       12 wider    True    0.005               10       4.471    2.4  3.00  -3.0  True
       12 wider   False    0.005               10       4.471    2.4  3.00  -3.0  True
       12 wider    True    0.000                5       4.484    2.6  3.25   2.5  True
       12  wide    True    0.000                5       4.484    2.5  3.25  -1.0  True
       40  wide   False    0.005               10       4.484    2.5  3.75   6.5  True
       40  wide   False    0.000           

In [17]:
%%run_if llm --optional
# ============================================================================
# BLOCK 4.5b - ABLATION TABLE + FIGURES (optional: never blocks the submission)
# ============================================================================
# ---- ablation table ----
abl_rows = []
for name, preds in ABL.items():
    abl_rows.append({'system': name, 'tune_A∪B_WER_%': round(wer_on(TUNE_IDX, preds) * 100, 2),
                     'verify_C_WER_%': round(wer_on(VERIFY_IDX, preds) * 100, 2) if VERIFY_IDX else np.nan,
                     'all_OOF_WER_%': round(wer_on(ALL_IDX, preds) * 100, 2)})
for name, (ot, ov) in ORACLE.items():
    abl_rows.append({'system': f'oracle: {name}', 'tune_A∪B_WER_%': round(ot * 100, 2),
                     'verify_C_WER_%': round(ov * 100, 2), 'all_OOF_WER_%': np.nan})
ABL_DF = pd.DataFrame(abl_rows)
print('\n' + ABL_DF.to_string(index=False)); save_table(ABL_DF, 'ablation_decoding.csv')
with figure_guard('ablation'):
    d = ABL_DF[~ABL_DF['system'].str.startswith('oracle')]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    x = np.arange(len(d)); w = 0.38
    ax.bar(x - w / 2, d['tune_A∪B_WER_%'], w, label='tune (A∪B, OOF)', color='#4C72B0')
    ax.bar(x + w / 2, d['verify_C_WER_%'], w, label='verify (C, unseen by both)', color='#DD8452')
    for xi, (a, b) in enumerate(zip(d['tune_A∪B_WER_%'], d['verify_C_WER_%'])):
        ax.text(xi - w / 2, a, f'{a:.2f}', ha='center', va='bottom', fontsize=7)
        ax.text(xi + w / 2, b, f'{b:.2f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x); ax.set_xticklabels(d['system'], rotation=25, ha='right', fontsize=8)
    ax.set_ylabel('WER (%)'); ax.legend(frameon=False); ax.set_title('Decoding ablation (official WER)')
    save_fig(fig, 'ablation_decoding')

with figure_guard('oracle_vs_k'):
    ks = [1, 2, 4, 8, 16, 24, 32, 64, 128, 256, 512]
    ov = []
    for k in ks:
        e = 0.0
        for j in ALL_IDX:
            S = utt_scores(POOL_VAL, j, W_STAGE1)
            e += ERRS[j][np.argsort(-S)[:k]].min()
        ov.append(e / NREF[ALL_IDX].sum() * 100)
    fig, ax = plt.subplots(figsize=(5, 3.4))
    ax.plot(ks, ov, marker='o'); ax.set_xscale('log', base=2)
    ax.axhline(wer_on(ALL_IDX, FINAL_PRED_VAL) * 100, ls='--', c='gray', label='final system')
    ax.set_xlabel('top-k candidates (stage-1 ranking)'); ax.set_ylabel('oracle WER (%)')
    ax.set_title('How much the pool can still give'); ax.legend(frameon=False)
    save_table(pd.DataFrame({'k': ks, 'oracle_wer_%': ov}), 'oracle_vs_k.csv')
    save_fig(fig, 'oracle_vs_k')

if LLM_OK:
    with figure_guard('fluency_gate'):
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
        ok = np.array([FINAL_ERRS[j][int(np.argmax(utt_scores(FINAL_POOL_VAL, j, dict(W_STAGE1, llm=0.0))))] == 0 for j in ALL_IDX])
        f = FINAL_FLU_VAL[ALL_IDX]
        axes[0].hist(f[ok], bins=40, alpha=0.6, label='stage-1 correct'); axes[0].hist(f[~ok], bins=40, alpha=0.6, label='stage-1 wrong')
        if np.isfinite(FINAL['gate']['tau']):
            axes[0].axvline(FINAL['gate']['tau'], c='k', ls='--', label='gate tau')
        axes[0].set_xlabel('adapted-LLM log-prob / token of acoustic-best hypothesis'); axes[0].legend(frameon=False)
        eh = FINAL['gate']['err_hi_grid']; gi = int(np.argmin(np.abs(G2 - FINAL['gate']['hi']['nw'])))
        im = axes[1].imshow(eh[:, :, gi] / NREF[ALL_IDX].sum() * 100, origin='lower', aspect='auto', cmap='viridis',
                            extent=[B2[0], B2[-1], A2[0], A2[-1]])
        axes[1].scatter([FINAL['gate']['hi']['llm']], [FINAL['gate']['hi']['ng']], c='red', marker='*', s=150)
        axes[1].set_xlabel('LLM weight'); axes[1].set_ylabel('KenLM weight'); axes[1].set_title('hi-regime errors (share of all words, %)')
        fig.colorbar(im, ax=axes[1]); save_fig(fig, 'fluency_gate_and_weight_surface')


                                  system  tune_A∪B_WER_%  verify_C_WER_%  all_OOF_WER_%
                greedy lexicon (1 model)           42.62           44.63          43.02
      flashlight 1-best lm=3.0 (1 model)            6.07            6.66           6.18
pool rescoring: CTC-ens + KenLM + #words            5.68            5.42           5.63
          + phonetic-neighbour expansion            4.80            4.67           4.78
                 + adapted LLM (no gate)            4.64            4.14           4.54
                          + fluency gate            4.47            3.87           4.35
  FINAL (refit on A∪B∪C, in-sample on C)            4.56            3.76           4.40
             oracle: pool (no expansion)            4.22            3.01            NaN
                oracle: pool + expansion            3.26            2.20            NaN
             oracle: top-12 (LLM-scored)            3.52            2.36            NaN
  table  -> /kaggle/working/tab

In [18]:
%%run_if llm --optional
# ============================================================================
# BLOCK 4.5c - PER-DAY ANALYSIS, outlier_days_analysis.csv / clean_days_analysis.csv (optional)
# ============================================================================
# ---- per-day analysis + outlier / clean day CSVs ----
day_rows = []
for s in sorted(set(readers['val'].meta[j]['session'] for j in ALL_IDX)):
    idx = [j for j in ALL_IDX if readers['val'].meta[j]['session'] == s]
    per_d = official_per([GREEDY_IDS[j] for j in idx], [list(readers['val'].meta[j]['phonemes']) for j in idx])[0]
    wf, ef, nf = official_wer([val_refs[j] for j in idx], [FINAL_PRED_VAL[j] for j in idx])
    wb = wer_on(idx, BASE_VAL)
    day_rows.append({'session': s, 'n_trials': len(idx), 'n_words': nf, 'mean_T': float(np.mean([readers['val'].meta[j]['n'] for j in idx])),
                     'per_greedy_%': round(per_d * 100, 2), 'wer_flashlight_%': round(wb * 100, 2),
                     'wer_final_%': round(wf * 100, 2), 'delta_pp': round((wf - wb) * 100, 2), 'word_errors': ef,
                     'frac_lo_regime': float(np.mean([FINAL_REG_VAL.get(j) == 'lo' for j in idx]))})
DAY_DF = pd.DataFrame(day_rows)
DAY_DF['share_of_all_errors_%'] = (DAY_DF['word_errors'] / max(DAY_DF['word_errors'].sum(), 1) * 100).round(2)
med = DAY_DF['wer_final_%'].median()
DAY_DF['outlier'] = DAY_DF['wer_final_%'] > max(2 * med, med + 5.0)
save_table(DAY_DF[DAY_DF['outlier']].sort_values('wer_final_%', ascending=False), 'outlier_days_analysis.csv')
save_table(DAY_DF[~DAY_DF['outlier']], 'clean_days_analysis.csv')
shutil.copy(os.path.join(tables_dir, 'outlier_days_analysis.csv'), os.path.join(WORK, 'outlier_days_analysis.csv'))
shutil.copy(os.path.join(tables_dir, 'clean_days_analysis.csv'), os.path.join(WORK, 'clean_days_analysis.csv'))
print(f"outlier days (> max(2x median, median+5pp), median={med:.2f}%):")
print(DAY_DF[DAY_DF['outlier']][['session', 'n_trials', 'per_greedy_%', 'wer_flashlight_%', 'wer_final_%']].to_string(index=False))
with figure_guard('per_day'):
    fig, ax = plt.subplots(figsize=(12, 3.6))
    x = np.arange(len(DAY_DF))
    ax.bar(x - 0.2, DAY_DF['wer_flashlight_%'], 0.4, label='flashlight 1-best', color='#BBBBBB')
    ax.bar(x + 0.2, DAY_DF['wer_final_%'], 0.4, label='final', color='#4C72B0')
    ax.set_xticks(x); ax.set_xticklabels(DAY_DF['session'].str.replace('t15.', ''), rotation=70, fontsize=7)
    ax.set_ylabel('OOF WER (%)'); ax.legend(frameon=False); ax.set_title('Per-session out-of-fold WER')
    save_fig(fig, 'per_day_wer')

  table  -> /kaggle/working/tables/outlier_days_analysis.csv
  table  -> /kaggle/working/tables/clean_days_analysis.csv
outlier days (> max(2x median, median+5pp), median=3.00%):
       session  n_trials  per_greedy_%  wer_flashlight_%  wer_final_%
t15.2023.10.01        44         11.03              8.31         8.63
t15.2023.10.13        44         10.94             16.94        16.12
t15.2023.10.20         9         15.10              8.45         8.45
t15.2023.11.26        44          3.33             10.79        11.51
t15.2024.03.08        24         14.51             16.77        12.42
t15.2025.01.10        23         21.63             21.39        17.34
t15.2025.03.30        30         19.66             22.55        19.61
  figure -> /kaggle/working/figures/per_day_wer.png


In [19]:
%%run_if llm --optional
# ============================================================================
# BLOCK 4.5d - ERROR ANALYSIS: search error vs model error (optional)
# ============================================================================
if 'NGRAM' in globals() and NGRAM is not None and NGRAM.m is None:
    try:
        NGRAM._load()
    except Exception as e:
        print('kenlm reload failed:', repr(e)); NGRAM = None
# ---- search error vs model error ----
ea_rows, ref_items = [], []
for j in ALL_IDX:
    ref = remove_punctuation(val_refs[j])
    hyp = FINAL_PRED_VAL[j]
    if ref == hyp:
        continue
    e, n = utt_word_errors(ref, hyp)
    in_pool = ref in set(FINAL_POOL_VAL.texts[j])
    oov = LEX.targets(ref) is None
    ref_items.append((j, ref, hyp, e, n, in_pool, oov))
ref_texts = [r for (_, r, _, _, _, ip, oov) in ref_items if not ip and not oov]
ref_llm = dict(zip(ref_texts, zip(*SCORER.score(ref_texts)))) if (LLM_OK and ref_texts) else {}
for (j, ref, hyp, e, n, in_pool, oov) in ref_items:
    if oov:
        kind = 'reference contains OOV word'
    elif in_pool:
        kind = 'model error: reference in pool but outscored'
    else:
        W = FINAL['gate']['lo'] if (FINAL['gate']['lo'] is not None and FINAL_FLU_VAL[j] < FINAL['gate']['tau']) else FINAL['gate']['hi']
        am = acoustic_scores([ref], UTT_EMS_VAL[j], LEX, DEVICE).mean(1)[0]
        ng = NGRAM.ln(ref) if NGRAM else 0.0
        ll = ref_llm[ref][0] if ref in ref_llm else 0.0
        s_ref = am + W['ng'] * ng + W.get('llm', 0) * ll + W['nw'] * len(ref.split())
        i = FINAL_POOL_VAL.texts[j].index(hyp)
        s_hyp = utt_scores(FINAL_POOL_VAL, j, W)[i]
        kind = 'search error: reference would win but never entered the pool' if s_ref > s_hyp else 'model error: reference not in pool and would lose anyway'
    ea_rows.append({'session': readers['val'].meta[j]['session'], 'ref': ref, 'hyp': hyp, 'word_errors': e, 'kind': kind})
EA_DF = pd.DataFrame(ea_rows)
if len(EA_DF):
    summ = EA_DF.groupby('kind').agg(utterances=('ref', 'size'), word_errors=('word_errors', 'sum')).reset_index()
    summ['share_of_word_errors_%'] = (summ['word_errors'] / summ['word_errors'].sum() * 100).round(1)
    print('\n' + summ.to_string(index=False))
    save_table(summ, 'error_analysis_search_vs_model.csv'); save_table(EA_DF, 'error_analysis_utterances.csv')
pd.DataFrame({'session': [readers['val'].meta[j]['session'] for j in ALL_IDX], 'label': [VAL_LABELS[j] for j in ALL_IDX],
              'ref': [remove_punctuation(val_refs[j]) for j in ALL_IDX], 'flashlight': [BASE_VAL[j] for j in ALL_IDX],
              'final': [FINAL_PRED_VAL[j] for j in ALL_IDX]}).to_csv(os.path.join(tables_dir, 'oof_predictions.csv'), index=False)
print(f'session clock: {human_time(time.time() - NOTEBOOK_T0)}')


                                                        kind  utterances  word_errors  share_of_word_errors_%
                model error: reference in pool but outscored          33           38                     9.1
    model error: reference not in pool and would lose anyway          92          186                    44.7
                                 reference contains OOV word           6            9                     2.2
search error: reference would win but never entered the pool         114          183                    44.0
  table  -> /kaggle/working/tables/error_analysis_search_vs_model.csv
  table  -> /kaggle/working/tables/error_analysis_utterances.csv
session clock: 7h12m


In [20]:
%%run_if llm --soft --needs-ok
# ============================================================================
# BLOCK 4.6 - KAGGLE SUBMISSION (test: both models, same pool/features/weights/gate)
# ============================================================================
TEST_PRED, TEST_REG = predict_texts(FINAL_POOL_TEST, gate=FINAL['gate'], flu=FINAL_FLU_TEST)
texts = [TEST_PRED[j] for j in range(n_test)]
sub_df = pd.DataFrame({'id': range(n_test), 'text': texts})
sub_df.to_csv(os.path.join(WORK, 'submission.csv'), index=False)
detail = pd.DataFrame({'id': range(n_test), 'session': [m['session'] for m in readers['test'].meta],
                       'block': [m['block'] for m in readers['test'].meta], 'trial': [m['trial'] for m in readers['test'].meta],
                       'text': texts, 'flashlight_1best': BASE_TEST, 'regime': [TEST_REG.get(j, 'hi') for j in range(n_test)],
                       'pool_size': [len(POOL_TEST.texts[j]) for j in range(n_test)]})
save_table(detail, 'test_predictions_detail.csv')
print(f"submission.csv: {len(sub_df)} rows | empty predictions: {int((sub_df['text'].str.strip() == '').sum())} | "
      f"changed vs flashlight 1-best: {np.mean([a != b for a, b in zip(texts, BASE_TEST)]) * 100:.1f}% | "
      f"lo-regime: {np.mean([TEST_REG.get(j) == 'lo' for j in range(n_test)]) * 100:.1f}%")
print(sub_df.head(10).to_string(index=False))
print(f'\nDONE. session clock: {human_time(time.time() - NOTEBOOK_T0)}')

  table  -> /kaggle/working/tables/test_predictions_detail.csv
submission.csv: 1450 rows | empty predictions: 0 | changed vs flashlight 1-best: 30.1% | lo-regime: 12.0%
 id                                        text
  0 i get tired with the song and dance routine
  1                              emergency care
  2                you create a bigger surprise
  3                i think maybe you look at it
  4               so that they do have problems
  5                            i enjoy that too
  6                      i moved to kansas city
  7                     maybe four times a week
  8               when he sees it to his events
  9                  there's just no one around

DONE. session clock: 7h12m


---
## Checklist after each run
1. **Train log** (monitor prints every 10 min): `BUDGET … projected TOTAL … epochs` at epochs 2 and 5 = the real epoch count on this T4. Each line also shows host RAM / GPU memory. Lines with `OOM`, `RESUMED` or `relaunching` mean the safety nets worked — training continues. Eval lines should show PER falling below v2's 8.43 %.
2. `tables/training_summary.csv`, `figures/training_curves.png` — if PER is still falling at the end the run was compute-limited (set `cr_weight=0` to roughly double the epochs, or continue with `init_from`); if it bottomed out early, raise `time_mask_frac` / `cr_weight`.
3. **LLM run**: `tables/ablation_decoding.csv` — every row has a *verify (C)* column that no tuning touched. Keep a component only if it helps on C.
4. `tables/error_analysis_search_vs_model.csv` — *search error* share large → increase `gen_beam`, `expand_top`, `expand_max_nb`; *model error* share large → acoustic model is the bottleneck.
5. `outlier_days_analysis.csv` / `clean_days_analysis.csv` in the output root, `submission.csv` in the output root.
6. After v6 is confirmed on C and on the leaderboard: FINAL-FIT run (`train_val_labels=['A','B','C']` for both folds) and reuse `best_weights.json`.